

# SimpleFSDP: Compiler-Optimized Fully Sharded Data Parallelism with Communication–Computation Overlap

## A Production-Grade Technical Report on FSDP Internals, Collective Decomposition, and TorchInductor-Level Scheduling for Large-Scale Distributed Training

---

## Table of Contents

1. [FSDP: First-Principles Architecture and Execution Model](#1-fsdp-first-principles-architecture-and-execution-model)
2. [Collective Communication Decomposition](#2-collective-communication-decomposition)
3. [SimpleFSDP: Compiler-Native FSDP via DTensor and TorchInductor](#3-simplefsdp-compiler-native-fsdp-via-dtensor-and-torchinductor)
4. [Bucketing: Amortizing Base Communication Latency](#4-bucketing-amortizing-base-communication-latency)
5. [Reordering: Prefetching Communication for Overlap](#5-reordering-prefetching-communication-for-overlap)
6. [Model Wrapping Strategies: Manual vs. Auto](#6-model-wrapping-strategies-manual-vs-auto)
7. [Composability with Multi-Dimensional Parallelism](#7-composability-with-multi-dimensional-parallelism)
8. [Memory Analysis and Constraint Modeling](#8-memory-analysis-and-constraint-modeling)
9. [Production Deployment Considerations](#9-production-deployment-considerations)
10. [Cross-Vendor Portability: NVIDIA and AMD Clusters](#10-cross-vendor-portability-nvidia-and-amd-clusters)

---

## 1. FSDP: First-Principles Architecture and Execution Model

### 1.1 Foundational Concept

Fully Sharded Data Parallelism (FSDP), also known as ZeRO Stage-3, eliminates the memory redundancy inherent in standard Data Parallelism (DP) by **sharding all three categories of training state**—model parameters, gradients, and optimizer states—across all participating ranks. Each rank stores only a $\frac{1}{N}$ slice of the full model state, where $N$ is the data-parallel world size.

The per-rank memory footprint under FSDP for a model with $\Phi$ parameters in mixed-precision training (BF16 parameters, FP32 optimizer) is:

$$
M_{\text{FSDP}}^{\text{rank}} = \frac{1}{N}\bigl(2\Phi + 2\Phi + (4\Phi + 4\Phi + 4\Phi)\bigr) = \frac{16\Phi}{N} \text{ bytes}
$$

where the terms correspond to:
- $2\Phi$: BF16 model parameters (sharded)
- $2\Phi$: BF16 gradients (sharded)
- $4\Phi + 4\Phi + 4\Phi$: FP32 master weights, first moment, and second moment of Adam (all sharded)

In contrast, standard DP replicates the full model and optimizer on every rank:

$$
M_{\text{DP}}^{\text{rank}} = 2\Phi + 2\Phi + 12\Phi = 16\Phi \text{ bytes}
$$

FSDP therefore achieves a **linear memory reduction** of $N\times$, at the cost of introducing all-gather and reduce-scatter collectives on the critical path.

### 1.2 Per-FSDP-Unit Execution Lifecycle

Each FSDP unit wraps $L$ contiguous layers and executes the following pipeline per training step. The lifecycle is strictly sequenced per unit and pipelined across units.

#### 1.2.1 Forward Pass Execution (Per FSDP Unit)

| Step | Operation | Communication | Memory Effect |
|------|-----------|---------------|---------------|
| 1 | **LOAD-MODEL-SHARD** | None (local I/O or HBM read) | Load local shard into working buffer; if CPU-offloaded, initiate H2D transfer |
| 2 | **ALL-GATHER** | All-gather across DP group | Materialize full parameter tensor: $2\Phi_{\text{unit}}$ bytes |
| 3 | **FORWARD (LOCAL)** | None | Compute activations locally using full parameters |
| 4 | **FREE FULL WEIGHTS** | None | Deallocate the all-gathered full-parameter buffer, retaining only the local shard |

> **Key insight:** The full parameter tensor exists in HBM **only for the duration of the local forward computation** of that FSDP unit. This transient materialization is the core mechanism enabling memory savings.

#### 1.2.2 Backward Pass Execution (Per FSDP Unit)

| Step | Operation | Communication | Memory Effect |
|------|-----------|---------------|---------------|
| 1 | **ALL-GATHER** | All-gather across DP group | Re-materialize full parameters for gradient computation |
| 2 | **BACKWARD (LOCAL)** | None | Compute local gradients using full parameters and saved activations |
| 3 | **REDUCE-SCATTER** | Reduce-scatter across DP group | Produce averaged gradient shard; each rank receives its $\frac{1}{N}$ slice |
| 4 | **FREE FULL WEIGHTS** | None | Deallocate the re-gathered full-parameter buffer |

#### 1.2.3 Optimizer Step

| Step | Operation | Communication | Memory Effect |
|------|-----------|---------------|---------------|
| 1 | **UPDATE WEIGHTS (LOCAL)** | None | Apply optimizer (e.g., Adam) on the local gradient shard and local optimizer states |

If CPU offloading is enabled:
- Gradients are offloaded D2H after reduce-scatter.
- The optimizer step executes on CPU using the offloaded gradient shard.
- Updated parameter shards are transferred H2D before the next forward pass.

#### 1.2.4 CPU Offloading Flow

```
Forward Entry:
  IF cpu_offload_enabled:
    H2D_transfer(local_parameter_shard)   // PCIe or NVLink-C2C
  ALL_GATHER(local_shard → full_params)

Backward Exit:
  REDUCE_SCATTER(full_grads → local_grad_shard)
  IF cpu_offload_enabled:
    D2H_transfer(local_grad_shard)
    CPU_optimizer_step(local_grad_shard, optimizer_states)
```

### 1.3 Multi-Unit Pipelining and Synchronization

In a model with $K$ FSDP units, the forward pass sequentially processes units $1, 2, \ldots, K$, and the backward pass processes them in reverse order $K, K-1, \ldots, 1$. The collectives form a **chain of dependencies**:

$$
\text{AG}_i^{\text{fwd}} \rightarrow \text{Compute}_i^{\text{fwd}} \rightarrow \text{AG}_{i+1}^{\text{fwd}} \rightarrow \cdots \rightarrow \text{AG}_K^{\text{bwd}} \rightarrow \text{Compute}_K^{\text{bwd}} \rightarrow \text{RS}_K^{\text{bwd}} \rightarrow \text{AG}_{K-1}^{\text{bwd}} \rightarrow \cdots
$$

The critical observation: **adjacent FSDP units' collectives can be overlapped with the current unit's compute** if scheduled on separate CUDA streams. This is the foundational principle that SimpleFSDP's reordering optimization exploits.

### 1.4 FSDP Execution Pseudocode

```
ALGORITHM: FSDP_Training_Step

INPUT:
  model partitioned into K FSDP units: {Unit_1, ..., Unit_K}
  each Unit_k holds local_shard_k of size Φ_k / N
  input mini-batch x

// ============ FORWARD PASS ============
FOR k = 1 TO K:
    IF cpu_offload:
        H2D_ASYNC(local_shard_k)
        STREAM_SYNC(compute_stream, transfer_stream)
    
    full_params_k ← ALL_GATHER(local_shard_k, group=dp_group)
    // full_params_k now has Φ_k parameters
    
    activations_k ← FORWARD_LOCAL(Unit_k, full_params_k, activations_{k-1})
    
    FREE(full_params_k)   // release N-1/N of gathered memory
    SAVE_FOR_BACKWARD(activations_k)  // or discard if using activation checkpointing

// ============ BACKWARD PASS ============
FOR k = K DOWNTO 1:
    full_params_k ← ALL_GATHER(local_shard_k, group=dp_group)
    
    grad_full_k ← BACKWARD_LOCAL(Unit_k, full_params_k, saved_activations_k, grad_output_k)
    
    grad_shard_k ← REDUCE_SCATTER(grad_full_k, op=AVG, group=dp_group)
    // grad_shard_k has Φ_k / N elements
    
    FREE(full_params_k)
    
    IF cpu_offload:
        D2H_ASYNC(grad_shard_k)

// ============ OPTIMIZER STEP ============
FOR k = 1 TO K:
    IF cpu_offload:
        CPU_ADAM_UPDATE(local_shard_k, grad_shard_k, optimizer_state_k)
    ELSE:
        GPU_ADAM_UPDATE(local_shard_k, grad_shard_k, optimizer_state_k)
```

### 1.5 Peak Memory During Execution

The transient peak memory for a single FSDP unit $k$ during training occurs at the moment when both the full parameters and the full gradients coexist in HBM (during backward):

$$
M_{\text{peak}}^{k} = \underbrace{\frac{\Phi_k}{N} \cdot 2}_{\text{local shard (BF16)}} + \underbrace{\Phi_k \cdot 2}_{\text{all-gathered params (BF16)}} + \underbrace{\Phi_k \cdot 2}_{\text{full gradients (BF16)}} + \underbrace{A_k}_{\text{activations}}
$$

After reduce-scatter and free:

$$
M_{\text{steady}}^{k} = \frac{\Phi_k}{N} \cdot 2 + \frac{\Phi_k}{N} \cdot 2 + \frac{\Phi_k}{N} \cdot 12
$$

This transient spike is the **binding constraint** when fitting large models and determines the maximum FSDP-unit granularity.

---

## 2. Collective Communication Decomposition

### 2.1 The Fundamental Identity

The all-reduce collective, which produces a globally reduced result replicated on every rank, decomposes exactly into two primitive collectives:

$$
\boxed{\textbf{All-Reduce} = \textbf{Reduce-Scatter} + \textbf{All-Gather}}
$$

This decomposition is not merely algebraic—it is the **architectural foundation** of FSDP. Standard DP uses all-reduce for gradient synchronization; FSDP replaces this with the two constituent collectives executed at different points in the computation graph, enabling sharded storage between them.

### 2.2 All-Reduce

**Semantics:** Every rank $r \in \{0, 1, \ldots, N-1\}$ holds input tensor $X_r$. After all-reduce with summation:

$$
\forall r: \quad Y_r = \sum_{i=0}^{N-1} X_i
$$

Every rank receives an identical copy of the fully reduced result.

**Communication volume** (ring algorithm):

$$
V_{\text{all-reduce}} = 2 \cdot \frac{N-1}{N} \cdot S \approx 2S \quad \text{for large } N
$$

where $S$ is the tensor size in bytes.

| Rank | Before | After |
|------|--------|-------|
| GPU 0 | $A$ | $A + B + C + D$ |
| GPU 1 | $B$ | $A + B + C + D$ |
| GPU 2 | $C$ | $A + B + C + D$ |
| GPU 3 | $D$ | $A + B + C + D$ |

### 2.3 Reduce-Scatter

**Semantics:** Each rank $r$ holds input tensor $X_r$ logically partitioned into $N$ chunks: $X_r = [X_r^{(0)}, X_r^{(1)}, \ldots, X_r^{(N-1)}]$. After reduce-scatter:

$$
\text{Rank } r \text{ receives: } Y_r = \sum_{i=0}^{N-1} X_i^{(r)}
$$

Each rank receives only its **own shard** of the reduced result.

**Communication volume** (ring algorithm):

$$
V_{\text{reduce-scatter}} = \frac{N-1}{N} \cdot S \approx S \quad \text{for large } N
$$

| Rank | Before (4 chunks each) | After (1 chunk each) |
|------|----------------------|---------------------|
| GPU 0 | $[A_0, A_1, A_2, A_3]$ | $A_0 + B_0 + C_0 + D_0$ |
| GPU 1 | $[B_0, B_1, B_2, B_3]$ | $A_1 + B_1 + C_1 + D_1$ |
| GPU 2 | $[C_0, C_1, C_2, C_3]$ | $A_2 + B_2 + C_2 + D_2$ |
| GPU 3 | $[D_0, D_1, D_2, D_3]$ | $A_3 + B_3 + C_3 + D_3$ |

### 2.4 All-Gather

**Semantics:** Each rank $r$ holds a shard $X_r$. After all-gather:

$$
\forall r: \quad Y_r = [X_0, X_1, \ldots, X_{N-1}]
$$

Every rank receives the **concatenation** of all shards—a full replica of the data.

**Communication volume** (ring algorithm):

$$
V_{\text{all-gather}} = \frac{N-1}{N} \cdot S \approx S \quad \text{for large } N
$$

| Rank | Before (1 shard each) | After (all shards) |
|------|----------------------|-------------------|
| GPU 0 | $A$ | $[A, B, C, D]$ |
| GPU 1 | $B$ | $[A, B, C, D]$ |
| GPU 2 | $C$ | $[A, B, C, D]$ |
| GPU 3 | $D$ | $[A, B, C, D]$ |

### 2.5 Verification of the Decomposition

Consider the all-reduce of tensors $\{X_0, X_1, X_2, X_3\}$ on 4 GPUs:

1. **Reduce-Scatter phase:** Each GPU $r$ receives $\sum_{i} X_i^{(r)}$—the reduced $r$-th chunk.
2. **All-Gather phase:** The reduced chunks are gathered so every GPU holds the full $[\sum_i X_i^{(0)}, \sum_i X_i^{(1)}, \sum_i X_i^{(2)}, \sum_i X_i^{(3)}] = \sum_i X_i$.

The total volume is:

$$
V_{\text{RS}} + V_{\text{AG}} = \frac{N-1}{N}S + \frac{N-1}{N}S = 2\frac{N-1}{N}S = V_{\text{all-reduce}}
$$

This confirms that the decomposition is **bandwidth-optimal** and introduces no overhead.

### 2.6 FSDP's Use of the Decomposition

| DP Phase | Collective Used | When | Why |
|----------|----------------|------|-----|
| Standard DP gradient sync | All-Reduce | After backward | Every rank needs full gradient for full-copy optimizer |
| FSDP forward | **All-Gather** | Before forward compute | Reconstruct full parameters from shards |
| FSDP backward (params) | **All-Gather** | Before backward compute | Re-materialize full parameters for gradient computation |
| FSDP backward (grads) | **Reduce-Scatter** | After backward compute | Produce averaged gradient shard for local optimizer |

> **Critical insight for FSDP designers:** By splitting the all-reduce into its two halves and inserting computation between them, FSDP achieves sharded memory semantics. The reduce-scatter produces a shard; the all-gather (deferred to the next usage point) reconstructs from shards. The interval between these two operations is exactly where the memory saving materializes.

### 2.7 Latency Cost Model

For any collective over $N$ ranks with message size $S$ bytes, using a ring algorithm on a network with per-message latency $\alpha$ and inverse bandwidth $\beta$ (seconds per byte):

$$
T_{\text{all-gather}} = (N-1)\alpha + \frac{N-1}{N} S \beta
$$

$$
T_{\text{reduce-scatter}} = (N-1)\alpha + \frac{N-1}{N} S \beta
$$

$$
T_{\text{all-reduce}} = 2(N-1)\alpha + 2\frac{N-1}{N} S \beta
$$

For NCCL's tree algorithm (commonly used on NVSwitch topologies):

$$
T_{\text{all-reduce}}^{\text{tree}} = 2 \log_2(N) \cdot \alpha + 2S\beta
$$

The per-message latency $\alpha$ is the **base latency** that SimpleFSDP's bucketing optimization targets.

---

## 3. SimpleFSDP: Compiler-Native FSDP via DTensor and TorchInductor

### 3.1 Architectural Philosophy

SimpleFSDP reimagines FSDP as a **compiler-first** distributed training strategy, in contrast to the runtime-centric approach of PyTorch FSDP1/FSDP2. The key architectural decisions are:

| Design Axis | Traditional FSDP (FSDP1/FSDP2) | SimpleFSDP |
|-------------|-------------------------------|-----------|
| Parameter sharding | Runtime hooks on `nn.Module` | DTensor with `Shard` placement |
| All-gather dispatch | Forward pre-hooks | `parametrization` + DTensor `redistribute` |
| Reduce-scatter dispatch | Backward post-hooks | Automatic via autograd of `redistribute` |
| Overlap scheduling | Manual stream management | TorchInductor IR node reordering |
| Bucketing | Runtime gradient bucket manager | TorchInductor IR node fusion |
| Graph capture | Incompatible or partial | Full graph via `torch.compile(fullgraph=True)` |
| Debuggability | Opaque hook-based execution | Transparent eager-mode semantics |

### 3.2 Core Mechanism: Parametrization + DTensor Redistribute

SimpleFSDP represents each model parameter as a **DTensor** with `Shard(0)` placement on the data-parallel mesh. The `ReplicateComputation` parametrization intercepts parameter access and issues the appropriate redistribute:

```
ALGORITHM: ReplicateComputation_Parametrization

CLASS ReplicateComputation:
    ATTRIBUTES: param_dtype, reduce_dtype

    FUNCTION replicate_compute(self, x):
        // x is a DTensor with Shard(0) placement on DP mesh
        // Shape: [Φ_unit / N, ...]
        
        replicated = x.redistribute(
            placements=(Replicate(),),
            forward_dtype=self.param_dtype,    // e.g., BF16
            backward_dtype=self.reduce_dtype   // e.g., FP32
        )
        // replicated: DTensor with Replicate() placement
        // Shape: [Φ_unit, ...]
        // Forward: issues ALL_GATHER on DP group
        // Backward: autograd generates REDUCE_SCATTER on DP group
        
        local_tensor = replicated.to_local(
            grad_placements=(Partial(reduce_op="avg"),)
        )
        // Extract local tensor for computation
        // grad_placements defines backward reduce semantics
        
        RETURN local_tensor
```

**Why this is elegant:**
- The `redistribute` call from `Shard(0)` to `Replicate()` **is** the all-gather—expressed as a differentiable DTensor operation.
- The backward pass of `redistribute` from `Replicate()` to `Shard(0)` **is** the reduce-scatter—generated automatically by autograd.
- Wrapping this in `selective_activation_checkpointing` means the all-gathered parameters are treated as "activations" that can be freed after forward and recomputed (re-gathered) in backward.

### 3.3 Selective Activation Checkpointing for Weight Freeing

The "free full weights after use" behavior of FSDP is implemented as selective activation checkpointing applied specifically to the parametrization output:

$$
\text{Selective AC on } \texttt{replicate\_compute} \Rightarrow \begin{cases} \text{Forward:} & \text{all-gather} \rightarrow \text{compute} \rightarrow \text{free full params} \\ \text{Backward:} & \text{re-all-gather} \rightarrow \text{backward compute} \rightarrow \text{reduce-scatter} \rightarrow \text{free} \end{cases}
$$

This reuse of the activation checkpointing primitive avoids introducing any FSDP-specific memory management hooks.

### 3.4 Graph Tracing and TorchInductor Lowering

After `simple_fsdp(model)` wrapping and `torch.compile`, the entire forward and backward pass—including all communication operations—is traced into a **single FX graph**, which is then lowered to TorchInductor IR nodes:

```
ALGORITHM: SimpleFSDP_Compilation_Pipeline

INPUT: model, training data loader
OUTPUT: optimized compiled training step

Step 1: WRAP
    wrapped_model ← simple_fsdp(model)
    // Applies DTensor sharding + ReplicateComputation parametrization
    // Applies selective activation checkpointing on parametrization

Step 2: COMPILE
    compiled_model ← torch.compile(wrapped_model, fullgraph=True)
    // Dynamo traces full FX graph including:
    //   - ALL_GATHER nodes (from redistribute)
    //   - COMPUTE nodes (matmul, layernorm, etc.)
    //   - REDUCE_SCATTER nodes (from autograd of redistribute)

Step 3: LOWER TO INDUCTOR IR
    ir_graph ← TorchInductor.lower(fx_graph)
    // Each operation becomes an IR node with:
    //   - Module provenance metadata
    //   - Estimated execution time (profiled or modeled)
    //   - Memory footprint

Step 4: OPTIMIZE IR (SimpleFSDP passes)
    bucketed_graph ← BUCKET(ir_graph)        // Section 4
    reordered_graph ← REORDER(bucketed_graph) // Section 5

Step 5: CODEGEN
    executable ← TorchInductor.codegen(reordered_graph)
    // Generate CUDA/HIP kernels with optimized scheduling
```

The critical advantage: **the compiler has visibility into both communication and computation**, enabling global scheduling decisions that are impossible with hook-based runtimes.

### 3.5 Simplicity and Debuggability

Two fundamental properties distinguish SimpleFSDP:

1. **Simplicity:** Users write exactly three lines of configuration beyond the standard model definition:
   - Set bucketing mode (`auto` or manual module list)
   - Enable reordering
   - Wrap with `simple_fsdp()` and `torch.compile()`

2. **Debuggability:** The eager-mode execution semantics are **identical** to the compiled execution. Users can remove `torch.compile` and run the model with standard PyTorch semantics for debugging—the `redistribute` calls still produce correct all-gather/reduce-scatter behavior, just without the bucketing and reordering optimizations.

---

## 4. Bucketing: Amortizing Base Communication Latency

### 4.1 The Latency Problem

In the unoptimized graph, each parameter produces its own all-gather and reduce-scatter IR node. For a Transformer layer with $P$ distinct parameter tensors (e.g., $Q$, $K$, $V$, $O$ projections, MLP up/down/gate, LayerNorm weights and biases), the total communication overhead includes $P$ instances of the base latency $\alpha$:

$$
T_{\text{comm}}^{\text{unbucketed}} = P \cdot \alpha + P \cdot \beta \cdot s_i = P\alpha + \beta \sum_{i=1}^{P} s_i
$$

where $s_i$ is the size of parameter $i$.

After bucketing $P$ parameters into a single collective:

$$
T_{\text{comm}}^{\text{bucketed}} = \alpha + \beta \cdot \sum_{i=1}^{P} s_i
$$

The savings are $(P-1)\alpha$, which is significant when $\alpha$ is non-trivial (typically 5–15 $\mu$s for NCCL on InfiniBand, 3–8 $\mu$s on NVSwitch).

### 4.2 Bucketing Mechanism

#### 4.2.1 All-Gather Bucketing

```
ALGORITHM: Bucket_AllGather

INPUT:
    AG_1, AG_2, ..., AG_k  // Individual all-gather IR nodes
    Each AG_i reads tensor T_i of size s_i

OUTPUT:
    AG_{1..k}  // Single bucketed all-gather IR node

Step 1: ALLOCATE
    S_total ← Σ s_i
    buffer_local ← ALLOCATE(S_total / N)  // Local shard buffer
    buffer_full ← ALLOCATE(S_total)        // Full gathered buffer

Step 2: FLATTEN AND CONCATENATE
    offset ← 0
    FOR i = 1 TO k:
        COPY(T_i.local_shard → buffer_local[offset : offset + s_i/N])
        offset ← offset + s_i/N

Step 3: ISSUE SINGLE ALL-GATHER
    AG_{1..k} ← ALL_GATHER_ASYNC(buffer_local → buffer_full, group=dp_group)
    Wa_{1..k} ← WAIT(AG_{1..k})  // All-gather-wait node

Step 4: COPY-OUT (after wait)
    offset ← 0
    FOR i = 1 TO k:
        T_i.full_params ← VIEW(buffer_full[offset : offset + s_i])
        offset ← offset + s_i
```

#### 4.2.2 Reduce-Scatter Bucketing

```
ALGORITHM: Bucket_ReduceScatter

INPUT:
    RS_1, RS_2, ..., RS_k  // Individual reduce-scatter IR nodes
    Each RS_i operates on gradient G_i of size s_i

OUTPUT:
    RS_{1..k}  // Single bucketed reduce-scatter IR node

Step 1: CHUNK AND CONCATENATE
    // Each gradient G_i is split into N chunks of size s_i/N
    FOR chunk_rank = 0 TO N-1:
        offset ← 0
        FOR i = 1 TO k:
            COPY(G_i.chunk[chunk_rank] → buffer_chunked[chunk_rank][offset : offset + s_i/N])
            offset ← offset + s_i/N

Step 2: ISSUE SINGLE REDUCE-SCATTER
    RS_{1..k} ← REDUCE_SCATTER_ASYNC(buffer_chunked, op=AVG, group=dp_group)
    Wr_{1..k} ← WAIT(RS_{1..k})  // Reduce-scatter-wait node

Step 3: READ-OUT (after wait)
    offset ← 0
    FOR i = 1 TO k:
        G_i.shard ← VIEW(output_buffer[offset : offset + s_i/N])
        offset ← offset + s_i/N
```

### 4.3 Bucketing Cost-Benefit Analysis

| Factor | Unbucketed ($P$ collectives) | Bucketed (1 collective) |
|--------|-------------------------------|------------------------|
| Base latency | $P \cdot \alpha$ | $\alpha$ |
| Bandwidth term | $\beta \sum s_i$ | $\beta \sum s_i$ |
| Extra buffer memory | 0 | $\sum s_i$ (temporary) |
| Copy-out overhead | 0 | $O(\sum s_i)$ memcpy |
| Overlap granularity | Fine-grained (per-param) | Coarse-grained (per-bucket) |

> **Engineering trade-off:** Aggressive bucketing amortizes latency but reduces overlap granularity. If a single bucket encompasses too many parameters, the all-gather for that bucket cannot be hidden behind the previous computation because its volume exceeds the computation time. This is exactly the tension that the auto-wrapping algorithm resolves.

---

## 5. Reordering: Prefetching Communication for Overlap

### 5.1 The Overlap Principle

NCCL all-gather and reduce-scatter are **asynchronous** operations. An all-gather issued on the communication stream can execute concurrently with compute kernels on the default compute stream. The goal of reordering is to **schedule the next unit's communication to overlap with the current unit's computation**.

The ideal scheduling achieves:

$$
T_{\text{step}} = \max\left(\sum_k T_{\text{compute}}^k, \sum_k T_{\text{comm}}^k\right) + T_{\text{exposed}}
$$

where $T_{\text{exposed}}$ is the communication time that **cannot** be hidden behind computation. Perfect overlap yields $T_{\text{exposed}} = 0$.

### 5.2 Forward Pass Reordering

**Before reordering** (sequential execution):

```
AG12 → Wa12 → C1 → C2 → AG34 → Wa34 → C3 → C4
```

Here $\text{AG}_{34}$ starts only after $C_2$ completes—its latency is fully exposed.

**After reordering** (prefetched execution):

```
AG12 → AG34 → Wa12 → C1 → C2 → Wa34 → C3 → C4
```

Now $\text{AG}_{34}$ is issued **before** $\text{Wa}_{12}$, meaning $\text{AG}_{34}$ executes on the communication stream while $C_1$ and $C_2$ execute on the compute stream. The communication for the **next** FSDP unit is prefetched during the **current** unit's computation.

**Timeline visualization:**

```
Compute stream: |----Wa12-copy-out----|---C1---|---C2---|----Wa34-copy-out----|---C3---|---C4---|
Comm stream:    |--AG12--|---AG34 (overlapped with C1, C2)---|
```

The copy-out after $\text{Wa}_{12}$ (extracting individual tensors from the bucketed buffer) is itself computation that $\text{AG}_{34}$ can overlap with.

### 5.3 Backward Pass Reordering

The backward pass is more complex because both all-gather (for parameters) and reduce-scatter (for gradients) must be scheduled.

**Before reordering:**

```
AG12 → Wa12 → C1 → C2 → RS12 → Wr12 → AG34 → Wa34 → C3 → C4 → RS34 → Wr34
```

**After reordering:**

```
AG12 → Wa12 → AG34 → C1 → C2 → RS12 → Wa34 → C3 → C4 → Wr12 → RS34 → Wr34
```

**Overlap achieved:**

| Communication | Overlapped with |
|--------------|-----------------|
| $\text{AG}_{34}$ | Computation $C_1$, $C_2$ |
| $\text{RS}_{12}$ | Computation $C_3$, $C_4$ (after $\text{Wa}_{34}$ copy-out) |

**Key scheduling rule for backward:** $\text{Wr}_{12}$ (reduce-scatter-wait for the previous bucket) is placed **before** $\text{RS}_{34}$, allowing $\text{RS}_{12}$ to overlap with later computation. The $\text{AG}$ is placed **after** the preceding $\text{Wa}$ because the copy-out compute from the all-gather provides additional overlap opportunity.

### 5.4 Reordering Constraints

The reordering must respect data dependencies:

1. $\text{Wa}_i$ must follow $\text{AG}_i$ (cannot use gathered data before gather completes).
2. $C_i$ must follow $\text{Wa}_i$ (computation requires materialized parameters).
3. $\text{RS}_i$ must follow $C_i$ (reduce-scatter requires computed gradients).
4. $\text{Wr}_i$ must follow $\text{RS}_i$ (cannot read reduced gradient before scatter completes).

Within these constraints, the reordering maximizes the temporal distance between each async launch and its corresponding wait.

### 5.5 Formal Overlap Condition

For communication of bucket $j$ to be fully hidden by computation of bucket $i$:

**Forward pass:**

$$
T_{\text{AG}_j} \leq T_{C_i} + T_{\text{copy-out}_i}
$$

**Backward pass:**

$$
T_{\text{RS}_{i-1}} + T_{\text{AG}_j} \leq T_{C_i} + T_{\text{copy-out}_i}
$$

If these inequalities are violated, the excess communication time appears as **exposed latency** on the critical path.

---

## 6. Model Wrapping Strategies: Manual vs. Auto

### 6.1 Manual Wrapping

#### 6.1.1 Mechanism

Users provide a list of module names (e.g., `["layers.0", "layers.1", ...]`). SimpleFSDP uses the TorchInductor IR node metadata—which preserves the original module provenance—to construct a mapping:

$$
\texttt{module\_name} \rightarrow \{IR_{\text{AG}}, IR_{\text{RS}}, IR_{\text{compute}}\}
$$

All communication IR nodes belonging to the same module are bucketed together, and the buckets are reordered according to the scheme in Section 5.

#### 6.1.2 Properties

- **Granularity:** One FSDP unit per user-specified module boundary.
- **Predictability:** Users control the bucket boundaries explicitly.
- **Compatibility:** Equivalent to FSDP2's `fully_shard()` per-module wrapping.
- **Risk:** Sub-optimal bucket sizes if modules have heterogeneous parameter counts.

#### 6.1.3 Example

For a 2-module model (Module 1 contains layers 1–2, Module 2 contains layers 3–4):

```
Forward (after bucket + reorder):
  Compute stream: |---C1---|---C2---|---C3---|---C4---|
  Comm stream:    |--AG12--|---AG34 (overlapped)---|

Backward (after bucket + reorder):
  Compute stream: |---C1---|---C2---|---C3---|---C4---|
  Comm stream:    |--AG12--|--AG34 (overlapped)--|--RS12 (overlapped)--|--RS34--|
```

### 6.2 Auto Wrapping

#### 6.2.1 Design Philosophy

Auto wrapping provides **optimal bucketing without user intervention**. Since SimpleFSDP shards per-parameter (not per-module), it employs a **greedy algorithm** that iteratively considers whether to merge the next parameter's communication into the current bucket, subject to time and memory constraints.

#### 6.2.2 Profiling Phase

Before wrapping decisions, SimpleFSDP profiles each IR node:

**Computation profiling:**
- Convert `FakeTensor` metadata to real tensors.
- Execute the computation kernel with real inputs.
- Record CUDA event time $T_c$ and peak memory $M_c$.

**Communication profiling:**
- Use the analytical model:

$$
T_m = \alpha + \beta \cdot n
$$

where $n$ is the transmitted word size (bytes), $\alpha$ is the base latency, and $\beta$ is the per-byte transmit time.

| Variable | Definition |
|----------|-----------|
| $T_m^{AG}$ | Current step's bucketed all-gather communication time |
| $T_c$ | Current step's computation time |
| $M_c$ | Next step's peak computation memory |
| $T_m^{RS}$ | Last step's bucketed reduce-scatter communication time |
| $T_{m_i}^{AG}$ | $i$-th all-gather's communication time |
| $T_{c_i}$ | Time to compute with parameters prefetched by $i$-th all-gather |
| $M_{c_i}$ | Peak memory to compute with parameters prefetched by $i$-th all-gather |
| $T_{m_i}^{RS}$ | Time to reduce-scatter the gradient for parameters in $i$-th all-gather |

#### 6.2.3 Auto-Wrapping Decision Algorithm

```
ALGORITHM: Auto_Wrapping_Decision

INPUT:
    T_m^AG    // Current bucketed AG time
    T_m^RS    // Previous bucketed RS time
    T_c       // Current computation time available for overlap
    M_c       // Current computation peak memory
    T_mi^AG   // Candidate i-th AG communication time
    T_ci      // Computation time for candidate's parameters
    M_ci      // Peak memory for candidate's computation
    M_max     // Maximum allowed HBM memory

OUTPUT:
    decision ∈ {BUCKET, NO_BUCKET}

IF phase == FORWARD:
    // Time constraint: can the enlarged bucket AG still be hidden?
    time_ok ← ( T_{m+m_i}^{AG} ≤ T_c )
    
    // Memory constraint: does prefetching the next step's
    // parameters stay within HBM budget?
    mem_ok ← ( M_c + M_{c_i} ≤ M_max )
    
    IF time_ok AND mem_ok:
        RETURN BUCKET
    ELSE:
        RETURN NO_BUCKET

ELSE IF phase == BACKWARD:
    // Time constraint: must account for BOTH the previous RS
    // and the enlarged AG being hidden by current compute
    time_ok ← ( T_m^{RS} + T_{m+m_i}^{AG} ≤ T_c )
    
    // Memory constraint: same as forward
    mem_ok ← ( M_c + M_{c_i} ≤ M_max )
    
    IF time_ok AND mem_ok:
        RETURN BUCKET
        // Also bucket the corresponding RS nodes
    ELSE:
        RETURN NO_BUCKET
```

#### 6.2.4 Why the Backward Constraint is Stricter

In the backward pass, the communication stream must execute **both** the reduce-scatter from the previous computation step and the all-gather for the current step. Both must complete within the window of the current computation:

$$
T_m^{RS} + T_{m+m_i}^{AG} \leq T_c
$$

This dual constraint means backward buckets tend to be **smaller** than forward buckets for the same model, which is the correct behavior: backward compute typically has higher arithmetic intensity (gradient computation + activation recomputation), providing more overlap budget.

#### 6.2.5 Greedy Bucketing Walk

```
ALGORITHM: Greedy_Auto_Bucket

INPUT:
    Ordered list of per-parameter AG/RS/Compute IR nodes
    M_max: memory limit

OUTPUT:
    List of bucketed AG/RS groups with reordering

current_bucket ← {}
current_T_AG ← 0
current_T_RS ← 0
current_T_c ← 0
current_M_c ← 0

FOR each parameter p_i in layer order:
    candidate_T_AG ← COMM_TIME(current_bucket ∪ {p_i})  // T_{m+m_i}^AG
    candidate_M ← current_M_c + M_{c_i}
    
    IF phase == FORWARD:
        can_bucket ← (candidate_T_AG ≤ current_T_c) AND (candidate_M ≤ M_max)
    ELSE:  // BACKWARD
        can_bucket ← (current_T_RS + candidate_T_AG ≤ current_T_c) AND (candidate_M ≤ M_max)
    
    IF can_bucket:
        current_bucket ← current_bucket ∪ {p_i}
        UPDATE current_T_AG, current_M_c
    ELSE:
        EMIT_BUCKET(current_bucket)
        current_bucket ← {p_i}
        current_T_RS ← RS_TIME(previous_bucket)  // for backward
        RESET current_T_AG, current_T_c, current_M_c for new bucket

EMIT_BUCKET(current_bucket)  // final bucket

APPLY_REORDER(all emitted buckets)  // Section 5 reordering
```

#### 6.2.6 Auto vs. Manual Wrapping Comparison

| Property | Manual Wrapping | Auto Wrapping |
|----------|----------------|---------------|
| User input required | Module boundary list | None (fully automatic) |
| Bucket granularity | Per-module | Per-parameter (greedy optimal) |
| Adapts to compute/comm ratio | No (fixed boundaries) | Yes (profile-driven) |
| Adapts to memory budget | No (user responsibility) | Yes ($M_{\max}$ constraint) |
| Heterogeneous layers | Sub-optimal | Optimal bucket sizing per layer |
| Implementation complexity | Lower (module metadata lookup) | Higher (profiling + greedy algorithm) |
| Forward/Backward asymmetry | Same boundaries for both | Different bucket sizes for forward/backward |

### 6.3 Auto Wrapping Example

Consider a 4-parameter model where:
- $T_{c_1} = 10\mu s$, $T_{c_2} = 8\mu s$, $T_{c_3} = 12\mu s$, $T_{c_4} = 6\mu s$
- $T_{m_1}^{AG} = 3\mu s$, $T_{m_2}^{AG} = 3\mu s$, $T_{m_3}^{AG} = 7\mu s$, $T_{m_4}^{AG} = 4\mu s$
- Memory per-param: $M_{c_i} = 200\text{MB}$ each, $M_{\max} = 800\text{MB}$

**Forward auto-bucketing:**
1. Start with $p_1$: bucket = $\{p_1\}$, $T_m^{AG} = 3\mu s$.
2. Consider $p_2$: $T_{m_{1+2}}^{AG} = 5\mu s \leq T_{c_1} = 10\mu s$ ✓, $M = 400\text{MB} \leq 800\text{MB}$ ✓ → bucket.
3. Consider $p_3$: $T_{m_{1+2+3}}^{AG} = 11\mu s > T_c = 10\mu s$ ✗ → emit bucket $\{p_1, p_2\}$, start new.
4. Start with $p_3$: bucket = $\{p_3\}$, $T_m^{AG} = 7\mu s$.
5. Consider $p_4$: $T_{m_{3+4}}^{AG} = 10\mu s \leq T_{c_3} = 12\mu s$ ✓, $M = 400\text{MB} \leq 800\text{MB}$ ✓ → bucket.
6. Emit bucket $\{p_3, p_4\}$.

Result: `AG_{12}`, `AG_{34}` — happens to match the manual example but was derived automatically.

---

## 7. Composability with Multi-Dimensional Parallelism

### 7.1 Composability Architecture

SimpleFSDP's DTensor-native design provides seamless composability with other parallelism dimensions because DTensor inherently supports **multi-dimensional meshes**.

#### 7.1.1 Meta Initialization

During model initialization on `torch.device("meta")`, SimpleFSDP disables the all-gather parametrization:

```
ALGORITHM: Meta_Init_Optimization

IF device == "meta":
    DISABLE parametrization.replicate_compute
    // Parameters exist as metadata-only tensors
    // No all-gather is issued during initialization
    // Reduces init time from O(N * Φ) to O(Φ/N) per rank
    
AFTER weight loading:
    ENABLE parametrization.replicate_compute
    // Subsequent forward passes issue all-gather as normal
```

This avoids the $O(N)$ memory and time cost of gathering parameters during initialization, which can be substantial for models with $\Phi > 100B$ parameters.

#### 7.1.2 Mixed Precision Training

Mixed precision is implemented through the DTensor `redistribute` call's dtype arguments:

| Precision Component | Configuration | Effect |
|--------------------|---------------|--------|
| `param_dtype` | BF16 or FP16 | Parameters cast to this dtype during all-gather in forward |
| `reduce_dtype` | FP32 | Gradients cast to this dtype during reduce-scatter in backward |
| Optimizer states | FP32 (always) | Adam moments and master weights maintained at full precision |

The forward all-gather transmits parameters in `param_dtype` (2 bytes per element), reducing communication volume by $2\times$ compared to FP32. The backward reduce-scatter uses `reduce_dtype` for numerical stability in gradient accumulation.

The memory for mixed precision FSDP per rank:

$$
M_{\text{mixed-FSDP}}^{\text{rank}} = \frac{1}{N}\bigl(\underbrace{2\Phi}_{\text{BF16 params}} + \underbrace{4\Phi}_{\text{FP32 grads (reduce dtype)}} + \underbrace{12\Phi}_{\text{FP32 optimizer}}\bigr) = \frac{18\Phi}{N}
$$

However, in practice, the gradient shard is kept in the `reduce_dtype` only for the reduce-scatter operation; the stored gradient shard can be in BF16 for the optimizer if the optimizer itself handles upcasting:

$$
M_{\text{practical}}^{\text{rank}} = \frac{1}{N}\bigl(2\Phi + 2\Phi + 12\Phi\bigr) = \frac{16\Phi}{N}
$$

#### 7.1.3 Tensor Parallel (TP) Composability

For 2D parallelism (FSDP + TP), each parameter is a **2D DTensor** sharded on both dimensions:

```
ALGORITHM: 2D_Parallelism_Redistribute

// Device mesh: (DP=dp_size, TP=tp_size), total world_size = dp_size × tp_size
// Parameter P initialized as DTensor with placement:
//   Shard(0) on DP dim, Shard(col_dim) on TP dim
// Shape per rank: [Φ / (dp_size × tp_size)]

FUNCTION replicate_compute_2d(self, x):
    // x: DTensor([Shard(0), Shard(tp_dim)])
    
    // Step 1: All-gather on DP sub-mesh only
    x_dp_replicated = x.redistribute(
        placements=(Replicate(), Shard(tp_dim)),
        mesh_dim=0  // DP dimension
    )
    // x_dp_replicated: DTensor([Replicate(), Shard(tp_dim)])
    // Now replicated across DP, still sharded across TP
    
    // Step 2: TP computation proceeds with standard column/row parallel
    local_tp_shard = x_dp_replicated.to_local()
    // Ready for TP matmul operations
    
    RETURN local_tp_shard
```

The DP all-gather gathers across the DP group (typically inter-node over InfiniBand), while the TP communication (all-reduce or reduce-scatter within TP forward/backward) occurs across the TP group (typically intra-node over NVLink/NVSwitch).

**Topology-aware placement:**

$$
\text{TP group:} \quad \text{GPUs within the same node (NVLink: 450–900 GB/s per H100)}
$$

$$
\text{DP group:} \quad \text{GPUs across nodes (InfiniBand: 50–100 GB/s per link)}
$$

#### 7.1.4 Pipeline Parallel (PP) Composability

```
ALGORITHM: PP_SimpleFSDP_Composition

// Model partitioned into S pipeline stages
// Each stage assigned to a subset of ranks

FOR each stage s:
    submodel_s ← extract_submodule(model, stage_boundaries[s])
    
    // SimpleFSDP wraps the local stage submodule
    fsdp_submodel_s ← simple_fsdp(submodel_s)
    
    // Compile the wrapped submodule
    compiled_stage_s ← torch.compile(fsdp_submodel_s)

// Pipeline schedule (1F1B, interleaved, etc.) orchestrates
// micro-batch execution across stages
// SimpleFSDP is transparent to the pipeline scheduler
```

No additional code is required because SimpleFSDP operates at the submodule level—it does not need global model visibility.

#### 7.1.5 Activation Checkpointing Composability

```
ALGORITHM: AC_SimpleFSDP_Composition

// Step 1: Apply user-defined activation checkpointing
apply_activation_checkpointing(
    model,
    check_fn=lambda m: isinstance(m, TransformerLayer),
    policy=SELECTIVE  // or FULL
)

// Step 2: SimpleFSDP wraps the model (adds its own selective AC for weights)
wrapped = simple_fsdp(model)

// Step 3: Compile
compiled = torch.compile(wrapped, fullgraph=True)

// The two levels of activation checkpointing compose:
//   - User's AC: recompute activations (attention outputs, MLP intermediates)
//   - SimpleFSDP's AC: recompute all-gathered parameters (weight freeing)
```

The user's activation checkpointing reduces activation memory; SimpleFSDP's internal selective checkpointing reduces parameter memory. They compose orthogonally.

### 7.2 Multi-Dimensional Parallelism Summary

For a full 4D parallelism configuration on an H100 cluster:

| Dimension | Degree | Communication | Network | SimpleFSDP Role |
|-----------|--------|---------------|---------|-----------------|
| TP | 8 (intra-node) | All-reduce, all-gather within TP group | NVSwitch (900 GB/s bisection) | DTensor 2D sharding |
| PP | 4–16 (inter-node) | Point-to-point send/recv | InfiniBand (400 Gbps) | Wraps per-stage submodule |
| DP/FSDP | $W / (TP \times PP)$ | All-gather, reduce-scatter | InfiniBand or NVLink | Core sharding + bucketing + reordering |
| CP | 2–8 | Ring all-gather of KV | NVLink or InfiniBand | Composes at attention level |

The world-size factorization must satisfy:

$$
W = DP \times TP \times PP \times CP
$$

and SimpleFSDP operates on the DP dimension of the resulting device mesh.

---

## 8. Memory Analysis and Constraint Modeling

### 8.1 Memory Budget Decomposition

For a Transformer model with $L$ layers, hidden dimension $H$, vocabulary size $V$, sequence length $S$, micro-batch size $B$, and FSDP world size $N$:

**Per-rank sharded state:**

$$
M_{\text{sharded}} = \frac{1}{N} \cdot \left( 2\Phi + 2\Phi + 12\Phi \right) = \frac{16\Phi}{N}
$$

**Transient all-gathered parameters** (peak during forward or backward of one FSDP unit wrapping $L_u$ layers):

$$
M_{\text{transient}} = 2 \cdot \Phi_{\text{unit}} = 2 \cdot L_u \cdot (12H^2 + 13H) \approx 24 L_u H^2 \text{ bytes (BF16)}
$$

**Activations** (with selective activation checkpointing, per Transformer layer):

$$
M_{\text{act}}^{\text{layer}} \approx 2BSH + 34BSH \cdot \frac{1}{\text{AC\_ratio}}
$$

where $\text{AC\_ratio}$ is the fraction of layers checkpointed.

**Total per-rank peak:**

$$
M_{\text{peak}} = M_{\text{sharded}} + M_{\text{transient}} + L \cdot M_{\text{act}}^{\text{layer}} + M_{\text{buffers}} + M_{\text{fragmentation}}
$$

### 8.2 Memory Constraint in Auto Wrapping

The auto-wrapping algorithm's $M_{\max}$ parameter represents the **maximum allowable transient memory for prefetched parameters**:

$$
M_{\max} = M_{\text{HBM}} - M_{\text{sharded}} - M_{\text{activations}} - M_{\text{overhead}}
$$

For an H100 with 80 GB HBM3:

$$
M_{\max} = 80\text{GB} - M_{\text{sharded}} - M_{\text{activations}} - 2\text{GB (CUDA context, buffers)}
$$

The auto-wrapping algorithm ensures that at any point during execution, the total prefetched parameter memory does not exceed this budget:

$$
\sum_{i \in \text{bucket}} M_{c_i} \leq M_{\max}
$$

### 8.3 Memory Impact of Bucketing

Bucketing introduces **temporary buffer memory** for the concatenated communication buffer:

$$
M_{\text{bucket\_buffer}} = 2 \times \text{bucket\_size\_bytes}
$$

The factor of 2 accounts for both the local shard buffer (input to all-gather) and the full gathered buffer (output of all-gather). For reduce-scatter, a similar buffer is needed for the chunked gradient concatenation.

In the worst case (single bucket for all parameters):

$$
M_{\text{bucket\_buffer}}^{\max} = 2 \cdot 2\Phi = 4\Phi \text{ bytes}
$$

This is why the memory constraint in auto wrapping is essential—it prevents a single giant bucket from causing OOM.

### 8.4 HBM Budget Table for Representative Configurations

| Model | $\Phi$ | GPU | HBM | FSDP $N$ | $M_{\text{sharded}}$ | $M_{\max}$ (approx.) |
|-------|--------|-----|-----|----------|---------------------|---------------------|
| 7B | 7×10⁹ | H100 80GB | 80 GB | 8 | 14 GB | ~50 GB |
| 70B | 70×10⁹ | H100 80GB | 80 GB | 64 | ~17.5 GB | ~40 GB |
| 405B | 405×10⁹ | H100 80GB | 80 GB | 512 | ~12.7 GB | ~45 GB |
| 70B | 70×10⁹ | B200 192GB | 192 GB | 32 | 35 GB | ~130 GB |

---

## 9. Production Deployment Considerations

### 9.1 Launch and Configuration

```
ALGORITHM: Production_SimpleFSDP_Launch

INPUT:
    model_config: architecture definition
    cluster_topology: {num_nodes, gpus_per_node, interconnect}
    memory_budget: per-GPU HBM limit
    target_throughput: tokens/sec goal

Step 1: PARALLELISM PLANNING
    tp_degree ← min(gpus_per_node, 8)  // saturate NVLink
    pp_degree ← ceil(model_layers / max_layers_per_gpu)
    dp_degree ← total_gpus / (tp_degree × pp_degree)
    
Step 2: FSDP CONFIGURATION
    torch._inductor.config.simplefsdp.bucket_mode ← "auto"
    torch._inductor.config.simplefsdp.enable_reorder ← True
    
Step 3: MODEL WRAPPING
    model ← initialize_on_meta_device(model_config)
    model ← apply_tensor_parallel(model, tp_mesh)
    model ← apply_activation_checkpointing(model, policy=SELECTIVE)
    model ← simple_fsdp(model)
    model ← torch.compile(model, fullgraph=True)
    
Step 4: VALIDATION
    // Verify no OOM: run single step with profiling
    // Verify numerical parity: compare loss curve against eager baseline
    // Verify overlap: inspect Nsight Systems trace for exposed comm
    
Step 5: SCALING
    // Run strong-scaling study: measure MFU at 1, 2, 4, 8 nodes
    // Target: > 40% MFU on H100, > 45% on B200
```

### 9.2 Graph Capture Considerations

| Scenario | `fullgraph` Setting | Rationale |
|----------|-------------------|-----------|
| Standard dense Transformer | `True` | Full graph enables maximum bucketing and reordering |
| MoE with top-k routing (data-dependent) | `False` | Control flow breaks graph; subgraphs optimized independently |
| Dynamic sequence lengths | `False` | Shape-dependent control flow requires graph breaks |
| Static shapes, no conditionals | `True` | Always preferred for maximum optimization |

When `fullgraph=False`, TorchInductor generates multiple subgraphs, each of which SimpleFSDP optimizes independently. The cross-subgraph communication cannot be bucketed, resulting in some exposed latency at graph boundaries.

### 9.3 Profiling and Diagnosis

```
ALGORITHM: SimpleFSDP_Performance_Diagnosis

Step 1: TRACE COLLECTION
    profile ← nsight_systems.trace(training_step, iterations=5)
    
Step 2: IDENTIFY EXPOSED COMMUNICATION
    FOR each comm_node in trace:
        overlap_ratio ← compute_overlap(comm_node, adjacent_compute)
        IF overlap_ratio < 0.9:
            REPORT("Exposed comm: {comm_node}, overlap={overlap_ratio}")
            
Step 3: ANALYZE BUCKET SIZES
    FOR each bucket in simplefsdp_buckets:
        comm_time ← bucket.all_gather_duration
        available_compute ← bucket.overlapping_compute_duration
        IF comm_time > available_compute:
            REPORT("Over-sized bucket: {bucket}, excess={comm_time - available_compute}")
            RECOMMEND("Reduce M_max or split bucket manually")

Step 4: MEMORY ANALYSIS
    peak_memory ← torch.cuda.max_memory_allocated()
    fragmentation ← torch.cuda.memory_stats()['active_bytes.all.peak'] - peak_memory
    IF fragmentation > 0.1 * HBM_CAPACITY:
        REPORT("Fragmentation exceeds 10%: {fragmentation}")
        RECOMMEND("Enable memory pool defragmentation or reduce bucket count")

Step 5: MFU CALCULATION
    mfu ← (6 × Φ × tokens_per_step) / (step_time × gpu_flops_peak)
    REPORT("MFU = {mfu}, target > 0.40")
```

### 9.4 Fault Tolerance and Checkpoint Compatibility

SimpleFSDP checkpoints are **DTensor-based**, meaning the saved state dict contains sharded tensors with placement metadata:

```
ALGORITHM: SimpleFSDP_Checkpoint_Resharding

// Save: each rank saves its local shard with DTensor metadata
state_dict ← model.state_dict()
// state_dict[key] = DTensor(data=local_shard, placements=[Shard(0)], mesh=dp_mesh)

// Resume with different world size N' ≠ N:
FOR each key in state_dict:
    old_placement ← state_dict[key].placements  // Shard(0) on old mesh
    new_placement ← Shard(0) on new_mesh(N')
    
    // DTensor redistribute handles resharding automatically
    state_dict[key] ← state_dict[key].redistribute(new_placement)

// This enables elastic training: survive node failures by
// resharding checkpoints to reduced world size
```

---

## 10. Cross-Vendor Portability: NVIDIA and AMD Clusters

### 10.1 NCCL vs. RCCL Considerations

| Aspect | NVIDIA (NCCL) | AMD (RCCL) |
|--------|--------------|------------|
| Transport | NVLink/NVSwitch, IB, PCIe | xGMI, IB, PCIe |
| Ring/tree selection | Auto-tuned via NCCL_ALGO, NCCL_PROTO | Manual tuning often required |
| Latency $\alpha$ | 3–8 μs (NVSwitch), 5–15 μs (IB) | 5–12 μs (xGMI), 8–20 μs (IB) |
| Bandwidth $\beta$ | ~0.9× link BW achieved | ~0.8× link BW (topology-dependent) |

SimpleFSDP's communication cost model ($T_m = \alpha + \beta n$) must be **re-profiled** when switching between NVIDIA and AMD clusters, as $\alpha$ and $\beta$ differ significantly.

### 10.2 Compiler Backend Portability

| Component | NVIDIA | AMD |
|-----------|--------|-----|
| Compute kernels | CUDA, cuDNN, Triton-CUDA | HIP, MIOpen, Triton-HIP |
| FlashAttention | flash-attn (CUDA native) | flash-attn (CK backend or Triton-HIP) |
| TorchInductor backend | Triton codegen (CUDA) | Triton codegen (HIP/ROCm) |
| Graph capture | CUDA Graphs | HIP Graphs (partial support) |

SimpleFSDP's optimizations (bucketing, reordering) operate at the **TorchInductor IR level**, which is **backend-agnostic**. The same IR transformations produce correct schedules for both CUDA and HIP backends. The per-vendor differences only affect:
1. Profiled $\alpha$, $\beta$ values (affects auto-wrapping bucket decisions)
2. Kernel codegen (Triton generates different PTX vs. AMDGPU assembly)
3. Graph capture capabilities (CUDA Graphs vs. HIP Graphs)

### 10.3 Cluster-Specific Auto-Wrapping Behavior

On MI300X (192 GB HBM3, xGMI interconnect):
- Higher $M_{\max}$ due to larger HBM → larger buckets are feasible.
- Higher $\alpha$ for xGMI → bucketing provides greater benefit.
- Different compute/comm ratio → auto-wrapping produces different bucket boundaries than H100.

On B200 (192 GB HBM3e, NVLink5):
- Higher $M_{\max}$ and higher bandwidth → can sustain very large buckets.
- Lower $\alpha$ with NVLink5 → marginal bucketing benefit per merge, but still worthwhile for large parameter counts.

---

## 11. Summary of Key Engineering Principles

### 11.1 Why SimpleFSDP Outperforms Runtime FSDP

| Principle | Runtime FSDP | SimpleFSDP |
|-----------|-------------|-----------|
| **Scheduling scope** | Per-module hooks (local decisions) | Full-graph IR (global decisions) |
| **Bucketing** | Fixed bucket size (bytes threshold) | Profile-driven, memory-aware, asymmetric forward/backward |
| **Reordering** | Manual prefetch heuristics | Compiler-driven IR node scheduling |
| **Overlap quality** | Best-effort stream concurrency | Provably optimal within profiling accuracy |
| **Code complexity** | Hook registration, state machine | 3 lines of user code |
| **Debug path** | Hooks alter execution semantics | Eager mode = compiled mode semantics |

### 11.2 Performance Impact Chain

$$
\text{Bucketing} \xrightarrow{\text{reduces } P\alpha \text{ to } \alpha} \text{Lower base latency} \xrightarrow{\text{enables}} \text{Fewer comm calls}
$$

$$
\text{Reordering} \xrightarrow{\text{prefetches AG/RS}} \text{Overlap with compute} \xrightarrow{\text{reduces}} T_{\text{exposed}} \rightarrow 0
$$

$$
\text{Auto Wrapping} \xrightarrow{\text{profile-driven}} \text{Optimal bucket sizes} \xrightarrow{\text{satisfies}} \text{Time + Memory constraints}
$$

Combined effect on step time:

$$
T_{\text{step}}^{\text{optimized}} \approx \max\left(\sum_k T_{\text{compute}}^k, \; \sum_b T_{\text{comm}}^b\right)
$$

versus unoptimized:

$$
T_{\text{step}}^{\text{naive}} = \sum_k T_{\text{compute}}^k + \sum_i (T_{\text{AG}_i} + T_{\text{RS}_i})
$$

The speedup ratio:

$$
\text{Speedup} = \frac{T_{\text{step}}^{\text{naive}}}{T_{\text{step}}^{\text{optimized}}} = \frac{\sum T_c + \sum T_{\text{comm}}}{\max(\sum T_c, \sum T_{\text{comm}})}
$$

For compute-bound training (typical for large models on fast interconnects), $\sum T_{\text{comm}} < \sum T_c$, and the speedup approaches:

$$
\text{Speedup} \approx 1 + \frac{\sum T_{\text{comm}}}{\sum T_c}
$$

which is typically 1.15–1.35× for well-configured clusters.

---

## 12. End-to-End Deployment Pseudocode

```
ALGORITHM: Complete_SimpleFSDP_Training

// ==================== ENVIRONMENT ====================
INITIALIZE torch.distributed(backend="nccl")  // or "rccl" for AMD
world_size ← GET_WORLD_SIZE()
rank ← GET_RANK()

// ==================== DEVICE MESH ====================
mesh_2d ← DeviceMesh("cuda", shape=(dp_size, tp_size))
dp_mesh ← mesh_2d[0]  // DP dimension
tp_mesh ← mesh_2d[1]  // TP dimension

// ==================== MODEL INIT ====================
WITH meta_device():
    model ← TransformerModel(config)
    // All parameters on meta device (no memory)

// ==================== PARALLELISM ====================
// Apply TP first (innermost dimension)
apply_column_parallel(model.layers.*.mlp.up_proj, tp_mesh)
apply_row_parallel(model.layers.*.mlp.down_proj, tp_mesh)
apply_column_parallel(model.layers.*.attn.qkv_proj, tp_mesh)
apply_row_parallel(model.layers.*.attn.o_proj, tp_mesh)

// Apply activation checkpointing
apply_selective_ac(model, policy=OP_SELECTIVE)

// Materialize parameters from checkpoint
load_sharded_checkpoint(model, checkpoint_path, mesh_2d)

// ==================== SIMPLEFSDP ====================
// Configure SimpleFSDP optimizations
SET torch._inductor.config.simplefsdp.bucket_mode = "auto"
SET torch._inductor.config.simplefsdp.enable_reorder = True

// Wrap model
model ← simple_fsdp(model)

// Compile with full graph
model ← torch.compile(model, fullgraph=True)

// ==================== OPTIMIZER ====================
optimizer ← AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95))
// Optimizer states are automatically sharded (local shard only)

// ==================== TRAINING LOOP ====================
FOR step = 1 TO max_steps:
    batch ← dataloader.next()
    // Deterministic sampling, resume-safe state
    
    loss ← model(batch.input_ids, batch.labels)
    loss.backward()
    
    // Gradient clipping (on sharded gradients)
    clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    optimizer.zero_grad()
    
    IF step % checkpoint_interval == 0:
        save_sharded_checkpoint(model, optimizer, step)
    
    IF step % log_interval == 0:
        tokens_per_sec ← batch_tokens / step_time
        mfu ← compute_mfu(model_flops, step_time, gpu_peak_flops)
        LOG(step, loss, tokens_per_sec, mfu, peak_memory)
```

---

## Appendix A: Quick Reference — FSDP Communication Pattern

| Training Phase | Collective | Direction | Data | Volume per Rank |
|---------------|-----------|-----------|------|-----------------|
| Forward entry | All-Gather | Shard → Full | Parameters | $\frac{N-1}{N} \cdot 2\Phi_{\text{unit}}$ |
| Forward exit | — (free) | — | Full params | 0 (deallocation) |
| Backward entry | All-Gather | Shard → Full | Parameters | $\frac{N-1}{N} \cdot 2\Phi_{\text{unit}}$ |
| Backward exit | Reduce-Scatter | Full → Shard | Gradients | $\frac{N-1}{N} \cdot 2\Phi_{\text{unit}}$ |
| Optimizer | — (local) | — | Sharded state | 0 (no communication) |

**Total communication volume per step per rank:**

$$
V_{\text{total}} = 3 \cdot \frac{N-1}{N} \cdot 2\Phi \approx 6\Phi \text{ bytes (BF16, large } N\text{)}
$$

---

*This report reflects production-grade understanding of SimpleFSDP's compiler-driven FSDP implementation, suitable for deployment on H100, B200, MI300X, and MI350-class clusters at scales from 8 to 16,384+ GPUs.*



# SOTAFSDP2: Production-Grade Fully Sharded Data Parallelism Engine

## Comprehensive Technical Report on Architecture, Failure Modes, Device Affinity, Checkpoint Safety, and Cross-Vendor Distributed Training

---

## Table of Contents

1. [Architectural Overview and Design Philosophy](#1-architectural-overview-and-design-philosophy)
2. [FSDP2 Memory Sharding: First-Principles Analysis](#2-fsdp2-memory-sharding-first-principles-analysis)
3. [The Device Affinity Problem and Its Resolution](#3-the-device-affinity-problem-and-its-resolution)
4. [Checkpoint Pipeline: Modern DCP vs. Legacy FSDP1 APIs](#4-checkpoint-pipeline-modern-dcp-vs-legacy-fsdp1-apis)
5. [Memory Management: Pressure-Aware Pooling and Fragmentation Control](#5-memory-management-pressure-aware-pooling-and-fragmentation-control)
6. [Stream Management: Communication-Computation Overlap Architecture](#6-stream-management-communication-computation-overlap-architecture)
7. [Gradient Accumulation: FSDP-Safe no_sync Protocol](#7-gradient-accumulation-fsdp-safe-no_sync-protocol)
8. [Mixed Precision: Numerical Robustness Across Precision Policies](#8-mixed-precision-numerical-robustness-across-precision-policies)
9. [Hardware Detection and Cross-Vendor Portability](#9-hardware-detection-and-cross-vendor-portability)
10. [Triton Kernel Integration: Fused Collective Primitives](#10-triton-kernel-integration-fused-collective-primitives)
11. [Metrics Collection: Zero-Stall GPU Instrumentation](#11-metrics-collection-zero-stall-gpu-instrumentation)
12. [Activation Checkpointing: Selective Recomputation Strategies](#12-activation-checkpointing-selective-recomputation-strategies)
13. [Auto-Wrap Policy: Transformer-Aware Layer Discovery](#13-auto-wrap-policy-transformer-aware-layer-discovery)
14. [Failure Mode Taxonomy and Root-Cause Resolution Chain](#14-failure-mode-taxonomy-and-root-cause-resolution-chain)
15. [Production Deployment: End-to-End Integration Contract](#15-production-deployment-end-to-end-integration-contract)

---

## 1. Architectural Overview and Design Philosophy

### 1.1 System Architecture

The SOTAFSDP2 engine implements a **production-hardened FSDP orchestrator** that sits between the user's training loop (SOTATrainer) and PyTorch's `FullyShardedDataParallel` runtime. It is not a reimplementation of FSDP sharding logic; rather, it is an **integration layer** that manages the full lifecycle of sharded training—wrapping, forward/backward scheduling, gradient accumulation, checkpoint serialization, memory pressure response, and cross-vendor hardware adaptation.

The architecture follows a **layered composition** pattern:

```
┌─────────────────────────────────────────────────────────────┐
│                       SOTATrainer                           │
│   (Training loop, data loading, logging, scheduling)        │
├─────────────────────────────────────────────────────────────┤
│                        SOTAFSDP2                            │
│   ┌───────────┐ ┌──────────────┐ ┌────────────────────┐    │
│   │ StreamMgr │ │ MixedPrecCtx │ │ GradientAccumulator │    │
│   └───────────┘ └──────────────┘ └────────────────────┘    │
│   ┌───────────┐ ┌──────────────┐ ┌────────────────────┐    │
│   │ MemoryPool│ │ MetricsCollr │ │ FSDPCheckpointMgr  │    │
│   └───────────┘ └──────────────┘ └────────────────────┘    │
├─────────────────────────────────────────────────────────────┤
│              PyTorch FSDP / Distributed Runtime             │
│   (FullyShardedDataParallel, ProcessGroup, NCCL/RCCL)      │
├─────────────────────────────────────────────────────────────┤
│              Hardware Layer (CUDA / ROCm / HIP)             │
│   (A100, H100, H200, B100, B200, MI300X, MI325X, MI350)    │
└─────────────────────────────────────────────────────────────┘
```

### 1.2 Design Principles

| Principle | Implementation |
|-----------|----------------|
| **Explicit error handling** | `Result[T] = Ok[T] | Err[T]` — no exceptions for control flow |
| **Device affinity as invariant** | Every operation wrapped in `_enforce_device_affinity()` context |
| **Vendor-agnostic primitives** | Hardware detection → adaptive stream priorities, timing, kernels |
| **Memory as first-class variable** | Pressure-aware pool, proactive GC, watermark-triggered eviction |
| **Non-blocking instrumentation** | CUDA event timing, no `synchronize()` in hot path |
| **Graceful API degradation** | Modern DCP (PyTorch ≥ 2.3) with fallback to legacy FSDP1 API |
| **Atomic checkpointing** | tmp-file + rename pattern, single-barrier coordination |

### 1.3 Integration Contract (SOTATrainer ↔ SOTAFSDP2)

The engine exposes a strict API boundary:

| Contract ID | Method | Responsibility |
|------------|--------|----------------|
| INT-001 | `wrap_model()` | FSDP wrapping with AC, precision, offload |
| INT-003 | `backward(loss)` | Gradient accumulation, `no_sync`, loss scaling |
| INT-004 | `step(optimizer)` | Unscale → clip → step → zero_grad → reset |
| INT-006 | `FSDPCheckpointManager.save/load_checkpoint()` | Device-safe, memory-aware checkpoint I/O |
| INT-007 | `forward_context()` | Autocast + non-blocking forward timing |
| INT-009 | `memory_summary()` | Human-readable VRAM usage for logging |
| INT-010 | `backward()` returns `bool` | `True` = sync step (call `step()`); `False` = accumulating |

---

## 2. FSDP2 Memory Sharding: First-Principles Analysis

### 2.1 Sharding Strategy Memory Formulas

For a model with $\Phi$ total parameters, $W$ GPUs in the data-parallel group, and mixed-precision training with BF16 parameters and FP32 optimizer (Adam with master weights, first moment $m$, second moment $v$):

**FULL_SHARD (ZeRO-3):**

$$
M_{\text{FULL\_SHARD}}^{\text{rank}} = \frac{1}{W}\bigl(\underbrace{2\Phi}_{\text{BF16 params}} + \underbrace{2\Phi}_{\text{BF16 grads}} + \underbrace{4\Phi}_{\text{FP32 master}} + \underbrace{4\Phi}_{m} + \underbrace{4\Phi}_{v}\bigr) = \frac{16\Phi}{W}
$$

**SHARD_GRAD_OP (ZeRO-2):**

Parameters remain replicated; only gradients and optimizer states are sharded:

$$
M_{\text{SHARD\_GRAD\_OP}}^{\text{rank}} = \underbrace{2\Phi}_{\text{BF16 params (replicated)}} + \frac{1}{W}\bigl(\underbrace{2\Phi}_{\text{grads}} + \underbrace{12\Phi}_{\text{optimizer}}\bigr) = 2\Phi + \frac{14\Phi}{W}
$$

**NO_SHARD (Standard DDP):**

$$
M_{\text{NO\_SHARD}}^{\text{rank}} = 2\Phi + 2\Phi + 12\Phi = 16\Phi
$$

**HYBRID_SHARD:**

Full sharding within a node ($W_{\text{intra}}$ GPUs), replication across nodes ($W_{\text{inter}}$ node groups):

$$
M_{\text{HYBRID}}^{\text{rank}} = \frac{16\Phi}{W_{\text{intra}}}
$$

Communication cost drops from inter-node all-gather to intra-node all-gather (NVLink bandwidth), at the expense of higher per-rank memory than FULL_SHARD across all nodes.

### 2.2 Memory Per Shard Computation

The implementation computes the per-rank memory footprint after wrapping:

$$
M_{\text{shard}}^{\text{GB}} = \frac{\Phi_{\text{rank}} \cdot b_{\text{elem}}}{2^{30}}
$$

where $\Phi_{\text{rank}} = \Phi / W$ is the per-rank parameter count and $b_{\text{elem}}$ is the bytes per element for the configured `param_dtype`.

```
ALGORITHM: Compute_Per_Shard_Memory

INPUT: wrapped_model, world_size W, param_dtype
OUTPUT: memory_per_shard_gb

param_count ← SUM(p.numel() FOR p IN wrapped_model.parameters())
bytes_per_elem ← ELEMENT_SIZE(param_dtype)
Φ_rank ← param_count / W
memory_per_shard_gb ← (Φ_rank × bytes_per_elem) / 2^30

RETURN memory_per_shard_gb
```

### 2.3 VRAM Pressure Watermark

The system defines a pressure watermark at 90% of total HBM capacity:

$$
\text{VRAM\_PRESSURE\_WATERMARK} = 0.90
$$

When allocated memory exceeds this threshold, proactive GC and cache eviction are triggered:

$$
\text{IF } M_{\text{allocated}} > 0.90 \times M_{\text{HBM\_total}} \text{ THEN trigger\_gc\_and\_eviction()}
$$

This is checked:
- After every optimizer step (in `step()`)
- Before checkpoint saves (in `_pre_save_memory_cleanup()`)
- During memory pool allocation (in `_check_pressure()`)

---

## 3. The Device Affinity Problem and Its Resolution

### 3.1 Root Cause Analysis (FIX-005)

This is the single most critical production failure mode in multi-GPU FSDP training. The failure chain is:

```
FAILURE CHAIN: Device Affinity Violation

Step 1: User calls checkpoint save with offload_to_cpu=True
Step 2: FSDP.state_dict_type() internally creates staging tensors
Step 3: Staging tensors are allocated on cuda:0 (PyTorch default device)
        NOT on cuda:{local_rank} (the rank's actual device)
Step 4: state_dict() moves shard tensors to CPU via D2H transfer
Step 5: dcp.save() performs NCCL collective for metadata coordination
Step 6: NCCL discovers tensor on cuda:0 but process group bound to cuda:3
Step 7: NCCL raises:
        "RuntimeError: Tensor on cuda:0 but expected cuda:3"
Step 8: All ranks enter NCCL timeout wait (default 1800s = 30 minutes)
Step 9: Training appears frozen with no visible error for 30 minutes
Step 10: Eventually NCCL timeout fires, all ranks crash simultaneously
```

**Why cuda:0 specifically:** PyTorch's `torch.cuda.default_device()` returns `cuda:0` unless explicitly overridden. Internal FSDP staging buffers are allocated using the default device, not the rank-local device.

### 3.2 The Device Affinity Guard

The resolution is a **defense-in-depth** context manager that enforces correct device binding:

```
ALGORITHM: Enforce_Device_Affinity

CONTEXT MANAGER: _enforce_device_affinity(self)

// Entry:
    torch.cuda.set_device(self._device)          // Pin default device
    WITH torch.cuda.device(self._device):         // Override context
        YIELD                                      // Execute guarded ops

// Exit:
    IF debug_mode:
        _verify_param_devices()                    // O(P) scan for drift
```

The guard has **two layers**:
1. `torch.cuda.set_device()` — sets the global default device for this thread.
2. `torch.cuda.device()` context — overrides the device context for all tensor allocations within the scope.

### 3.3 Post-Operation Device Verification

After any operation that may cause device drift (checkpoint save/load, state dict manipulation), a verification scan detects and corrects drift:

```
ALGORITHM: Verify_Param_Devices

INPUT: wrapped_model, expected_device
OUTPUT: drift_count (side-effect: migrate drifted params)

drift_count ← 0
FOR (name, param) IN wrapped_model.named_parameters():
    IF param.is_cuda AND param.device ≠ expected_device:
        drift_count ← drift_count + 1
        IF drift_count ≤ 5:
            LOG_WARNING("Device drift: {name} on {param.device}")
        param.data ← param.data.to(expected_device)

IF drift_count > 0:
    LOG_WARNING("Migrated {drift_count} params back to {expected_device}")

RETURN drift_count
```

**Complexity:** $O(P)$ where $P$ is the number of parameters. Runs only during checkpoint operations (not in the training hot path) and only in debug mode during normal training.

### 3.4 Device Re-Pinning Pattern

Throughout the checkpoint code, device re-pinning appears after every API call that may internally allocate tensors:

```
ALGORITHM: Device_Repin_Pattern

// Before any FSDP/DCP API call:
torch.cuda.set_device(self._device)

// After API call that may cause drift:
API_CALL(...)                               // e.g., get_state_dict()
torch.cuda.set_device(self._device)         // IMMEDIATE re-pin

// Pattern repetition is INTENTIONAL:
// Each API call may internally call torch.cuda.set_device(0)
// or allocate on default device. Re-pinning after EVERY call
// ensures no drift accumulates.
```

> **Engineering rationale:** Re-pinning may appear redundant, but each is necessary because any PyTorch internal call (especially FSDP state dict operations) can reset the default device as a side effect. The cost of an unnecessary `set_device()` call is ~100ns; the cost of missing one is a 30-minute hang followed by a crash.

---

## 4. Checkpoint Pipeline: Modern DCP vs. Legacy FSDP1 APIs

### 4.1 API Version Detection

The system detects the available checkpoint API at module import time:

```
ALGORITHM: Detect_Checkpoint_API

TRY:
    IMPORT torch.distributed.checkpoint.state_dict.get_state_dict
    IMPORT torch.distributed.checkpoint.state_dict.set_state_dict
    IMPORT torch.distributed.checkpoint.state_dict.StateDictOptions
    _HAS_MODERN_DCP ← True
    // PyTorch ≥ 2.3: DTensor-based, no ShardedTensor
EXCEPT ImportError:
    _HAS_MODERN_DCP ← False
    // PyTorch < 2.3: Legacy FSDP1 API with ShardedTensor
```

### 4.2 Modern DCP Pipeline (PyTorch ≥ 2.3)

The modern DCP path eliminates all known checkpoint failure modes:

| Property | Modern DCP | Legacy FSDP1 |
|----------|-----------|-------------|
| Internal representation | DTensor | ShardedTensor (deprecated) |
| API calls for model+optimizer | 1 (`get_state_dict` with optimizers) | 2 (separate `state_dict_type` scopes) |
| All-gather collectives per save | 1 | 2 (model + optimizer) |
| Barriers per save | 1 | 4 (2 scopes × 2 barriers each) |
| Device drift risk | Minimal (DTensor tracks device) | High (ShardedTensor uses cuda:0) |
| Deprecation warnings | None | Hundreds per save |
| Approximate time (70B, 78% VRAM) | ~15s | ~50–100s (FIX-007 root cause) |

#### 4.2.1 Modern Save Flow

```
ALGORITHM: Save_Sharded_Modern

INPUT: fsdp, optimizer, checkpoint_dir, epoch, step, extra_state
OUTPUT: Result[None]

// Phase 1: Collect state using DTensor API
options ← StateDictOptions(
    full_state_dict=False,    // Sharded, not gathered
    cpu_offload=True          // Offload to CPU for write
)

// SINGLE API call for BOTH model and optimizer
(model_state, optim_state) ← get_state_dict(
    fsdp.wrapped_model,
    optimizers=[optimizer],
    options=options
)

// [FIX-005] Re-pin device after get_state_dict
torch.cuda.set_device(fsdp._device)

// Phase 2: Combine into single state dict
combined_state ← {
    "model": model_state,
    "optimizer": optim_state
}

// Phase 3: Single DCP save (one coordinated write across all ranks)
dcp.save(
    state_dict=combined_state,
    storage_writer=FileSystemWriter(checkpoint_dir, overwrite=True)
)

// Phase 4: Metadata on rank 0 only (no collective required)
IF rank == 0:
    meta ← {epoch, step, api_version="dcp_modern_v2", config}
    torch.save(meta, checkpoint_dir / "meta.pt")

RETURN Ok(None)
```

> **Why single `get_state_dict` matters:** The legacy path calls `state_dict_type()` twice—once for model, once for optimizer. Each call triggers a full all-gather across all ranks. With 78% VRAM utilization, the second all-gather can cause OOM-retry loops and NCCL retransmission, leading to the 50–100 second stalls documented in FIX-007.

#### 4.2.2 Modern Load Flow

```
ALGORITHM: Load_Sharded_Modern

INPUT: fsdp, optimizer, checkpoint_dir
OUTPUT: Result[Dict metadata]

// Phase 1: Get current state structure (DTensor shapes/placements)
options ← StateDictOptions(full_state_dict=False, cpu_offload=False)
(model_state, optim_state) ← get_state_dict(
    fsdp.wrapped_model,
    optimizers=[optimizer],
    options=options
)

// [FIX-005] Re-pin device
torch.cuda.set_device(fsdp._device)

// Phase 2: Load checkpoint data into state structure
combined_state ← {"model": model_state, "optimizer": optim_state}
dcp.load(
    state_dict=combined_state,
    storage_reader=FileSystemReader(checkpoint_dir)
)

// [FIX-005] Re-pin device
torch.cuda.set_device(fsdp._device)

// Phase 3: Apply loaded state to model and optimizer
set_state_dict(
    fsdp.wrapped_model,
    optimizers=[optimizer],
    model_state_dict=combined_state["model"],
    optim_state_dict=combined_state["optimizer"],
    options=options
)

// Phase 4: Load metadata
meta ← torch.load(checkpoint_dir / "meta.pt", map_location="cpu")

RETURN Ok(meta)
```

### 4.3 Legacy FSDP1 Pipeline (PyTorch < 2.3)

When the modern API is unavailable, the system falls back to the legacy path with comprehensive warning suppression and additional safety measures.

#### 4.3.1 Legacy Save Flow

```
ALGORITHM: Save_Sharded_Legacy

INPUT: fsdp, optimizer, checkpoint_dir, epoch, step, extra_state
OUTPUT: Result[None]

// [FIX-008] Suppress ALL legacy deprecation warnings
WITH warnings.catch_warnings():
    SUPPRESS FutureWarning("ShardedTensor")
    SUPPRESS FutureWarning("FSDP.state_dict_type")
    SUPPRESS UserWarning("_get_pg_default_device")
    SUPPRESS FutureWarning("set_state_dict_type")
    SUPPRESS UserWarning("existing checkpoint")

    // ── Model State (First Scope) ──
    model_cfg ← ShardedStateDictConfig(offload_to_cpu=True)
    WITH FSDP.state_dict_type(model, SHARDED_STATE_DICT, model_cfg):
        model_state ← model.state_dict()
    
    // [FIX-005] Re-pin device after CPU offload
    torch.cuda.set_device(fsdp._device)
    
    dcp.save({"model": model_state}, FileSystemWriter(model_dir))
    
    // [FIX-007] Free model state BEFORE optimizer all-gather
    DELETE model_state
    gc.collect()
    torch.cuda.empty_cache()
    
    // [FIX-005] Re-pin device AGAIN
    torch.cuda.set_device(fsdp._device)
    
    // ── Optimizer State (Second Scope) ──
    optim_cfg ← ShardedOptimStateDictConfig(offload_to_cpu=True)
    WITH FSDP.state_dict_type(model, SHARDED_STATE_DICT, optim_cfg):
        optim_state ← FSDP.optim_state_dict(model, optimizer)
    
    // [FIX-005] Re-pin device after CPU offload
    torch.cuda.set_device(fsdp._device)
    
    dcp.save({"optimizer": optim_state}, FileSystemWriter(optim_dir))
    
    DELETE optim_state
    gc.collect()
    torch.cuda.empty_cache()

// [FIX-005] Final device re-pin
torch.cuda.set_device(fsdp._device)

// Metadata on rank 0
IF rank == 0:
    torch.save(meta, checkpoint_dir / "meta.pt")

RETURN Ok(None)
```

**Critical difference from modern path:** The legacy path requires **two separate `state_dict_type()` scopes** (model + optimizer), each triggering independent all-gather collectives. The explicit `gc.collect() + empty_cache()` between them (FIX-007) creates VRAM headroom for the second all-gather.

### 4.4 Full State Dict Save (Portable)

For cross-world-size portability, the full state dict is gathered to rank 0:

```
ALGORITHM: Save_Full_StateDict

INPUT: fsdp, optimizer, path, epoch, step
OUTPUT: Result[None]

// [FIX-005] Pin device
torch.cuda.set_device(fsdp._device)

IF _HAS_MODERN_DCP:
    options ← StateDictOptions(full_state_dict=True, cpu_offload=True)
    (model_state, optim_state) ← get_state_dict(model, [optimizer], options)
ELSE:
    // Legacy with FullStateDictConfig(rank0_only=True)
    ...

IF rank == 0:
    checkpoint ← {model_state, optim_state, epoch, step, extra}
    
    // [FIX-007] ATOMIC WRITE: tmp + rename
    tmp_path ← path.with_suffix(".tmp")
    torch.save(checkpoint, tmp_path)
    tmp_path.rename(path)            // Atomic on POSIX filesystems

DELETE model_state, optim_state
gc.collect()
torch.cuda.empty_cache()

RETURN Ok(None)
```

> **Atomicity guarantee:** The tmp-file + rename pattern ensures that a crash during `torch.save()` never corrupts the checkpoint. The `rename()` syscall is atomic on POSIX filesystems (ext4, XFS, Lustre). If the process crashes between `torch.save()` and `rename()`, only the `.tmp` file exists—the original checkpoint is untouched.

### 4.5 Unified Checkpoint Save Pipeline

The complete save pipeline orchestrates all phases with device safety and memory management:

```
ALGORITHM: Unified_Save_Pipeline

INPUT: fsdp, optimizer, path, epoch, step, extra_state, sharded
OUTPUT: Result[None]

save_start ← monotonic_time()

TRY:
    // Phase 1: Create VRAM headroom
    _pre_save_memory_cleanup(fsdp)
        // Release pool slabs → gc.collect() → empty_cache()
        // Log: "Pre-save cleanup: X.XX/Y.Y GB (Z.Z% VRAM)"
    
    // Phase 2: Synchronize all ranks
    _barrier_with_timeout(fsdp, label="pre-save", timeout=300s)
        // [FIX-007] 5-minute timeout vs. default 30-minute
    
    // Phase 3: Save (under device affinity guard)
    WITH fsdp._enforce_device_affinity():
        IF sharded AND _HAS_MODERN_DCP:
            result ← _save_sharded_modern(...)
        ELSE IF sharded:
            result ← _save_sharded_legacy(...)
        ELSE:
            result ← _save_full(...)
    
    IF result.is_err():
        RETURN result
    
    // Phase 4: Verify no device drift
    _post_save_device_check(fsdp)
    
    // Phase 5: Cleanup gathered params
    gc.collect()
    torch.cuda.empty_cache()
    
    // Phase 6: Post-save barrier
    _barrier_with_timeout(fsdp, label="post-save", timeout=300s)
    
    elapsed ← monotonic_time() - save_start
    LOG("Checkpoint saved in {elapsed:.1f}s: {path}")
    
    RETURN Ok(None)

EXCEPT Exception AS e:
    LOG_ERROR("Checkpoint save failed on rank {rank}: {e}")
    // CRITICAL: Restore device affinity even on failure
    torch.cuda.set_device(fsdp._device)
    RETURN Err("Checkpoint save failed: {e}", code=1)
```

### 4.6 Warning Suppression Strategy (FIX-008)

The warning suppression follows a **fail-open policy**: only known-harmless deprecation warnings from specific PyTorch modules are suppressed; novel or unexpected warnings pass through.

| Warning | Category | Source Module | Suppression Reason |
|---------|----------|--------------|-------------------|
| `_get_pg_default_device will be deprecated` | UserWarning | `torch.distributed.distributed_c10d` | Internal API change; no user action possible |
| `FSDP.state_dict_type() being deprecated` | FutureWarning | `torch.distributed.fsdp` | We use modern API when available |
| `Please use DTensor, deprecating ShardedTensor` | FutureWarning | `torch.distributed` | We use DTensor when available |
| `Detected existing checkpoint, overwriting` | UserWarning | `torch.distributed.checkpoint` | We pass `overwrite=True` explicitly |

```
ALGORITHM: Warning_Suppression_Policy

// Called ONCE at module import time
FUNCTION _suppress_known_deprecation_warnings():
    FOR each (message_pattern, category, module_pattern) IN KNOWN_WARNINGS:
        warnings.filterwarnings(
            action="ignore",
            message=message_pattern,    // Regex match
            category=category,          // Exact class match
            module=module_pattern       // Regex match on source module
        )
    // NO catch-all suppressions
    // Unknown warnings from unknown modules → visible to user
```

---

## 5. Memory Management: Pressure-Aware Pooling and Fragmentation Control

### 5.1 Memory Pool Architecture

The `MemoryPool` eliminates `cudaMalloc` jitter through **power-of-2 bucketed pre-allocation** with pressure-aware eviction.

#### 5.1.1 Bucket Size Computation

For a requested allocation of $R$ bytes:

$$
B(R) = \begin{cases} 512 & \text{if } R \leq 512 \\ R & \text{if } R \geq 256\text{MB} \\ 2^{\lceil \log_2 R \rceil} & \text{otherwise} \end{cases}
$$

This ensures:
- **Small allocations** (< 512B) use minimum bucket size to avoid waste.
- **Large allocations** (≥ 256MB) are sized exactly (no rounding to avoid 2× memory waste at large scales).
- **Mid-range allocations** use power-of-2 rounding for bucket reuse.

#### 5.1.2 Allocation Flow

```
ALGORITHM: Pool_Allocate

INPUT: shape (tuple), dtype (optional)
OUTPUT: Tensor on GPU

num_elements ← PRODUCT(shape)
size_bytes ← num_elements × ELEMENT_SIZE(dtype)
bucket ← BUCKET_SIZE(size_bytes)

WITH lock:
    allocation_count ← allocation_count + 1
    
    // Phase 1: Check pressure and evict if necessary
    _check_pressure()
    
    // Phase 2: Try to reuse existing slab
    IF bucket IN pools AND pools[bucket] NOT EMPTY:
        slab ← pools[bucket].pop()
        IF slab.numel() ≥ num_elements:
            RETURN slab[:num_elements].view(shape)
        ELSE:
            DELETE slab  // Size mismatch (shouldn't happen with correct bucketing)
    
// Phase 3: Fresh allocation (outside lock for concurrency)
tensor ← torch.empty(num_elements, dtype=dtype, device=self._device)

WITH lock:
    total_allocated ← total_allocated + size_bytes
    peak_allocated ← MAX(peak_allocated, total_allocated)

RETURN tensor.view(shape)
```

#### 5.1.3 Pressure-Aware Eviction

```
ALGORITHM: Check_Pressure

// Called within allocation lock
allocated ← torch.cuda.memory_allocated(device)
threshold ← device_total_bytes × 0.90  // VRAM_PRESSURE_WATERMARK

IF allocated < threshold:
    RETURN  // No pressure

// Evict largest buckets first (greedy approach)
FOR bucket_size IN SORTED(pools.keys(), DESCENDING):
    WHILE pools[bucket_size] NOT EMPTY:
        tensor ← pools[bucket_size].pop()
        freed ← tensor.numel() × tensor.element_size()
        total_allocated ← total_allocated - freed
        DELETE tensor  // Triggers cudaFree
        
        IF torch.cuda.memory_allocated(device) < threshold:
            RETURN  // Pressure relieved
    
    // Remove empty bucket entry
    IF pools[bucket_size] IS EMPTY:
        DELETE pools[bucket_size]
```

**Eviction ordering:** Largest buckets are evicted first because:
1. They free the most memory per eviction.
2. Large slabs have lower reuse probability (fewer matching allocations).
3. Small slabs are more likely to be reused (high churn in gradient buffers).

#### 5.1.4 Release (Return to Pool)

```
ALGORITHM: Pool_Release

INPUT: tensor
// Non-contiguous tensors cannot be returned to pool
// (view semantics break bucket reuse)
IF NOT tensor.is_contiguous():
    RETURN  // Let CUDA GC handle it

size_bytes ← tensor.numel() × tensor.element_size()
bucket ← BUCKET_SIZE(size_bytes)

WITH lock:
    IF bucket NOT IN pools:
        pools[bucket] ← []
    pools[bucket].append(tensor.detach().view(-1))
```

### 5.2 Memory Lifecycle During Training

The total VRAM budget during a training step with FSDP and the memory pool is:

$$
M_{\text{total}} = M_{\text{sharded\_state}} + M_{\text{transient\_AG}} + M_{\text{activations}} + M_{\text{pool\_slabs}} + M_{\text{CUDA\_context}} + M_{\text{fragmentation}}
$$

| Component | Size | Lifetime |
|-----------|------|----------|
| Sharded state (params + grads + optimizer) | $16\Phi / W$ | Permanent |
| Transient all-gathered params | $2\Phi_{\text{unit}}$ | Forward/backward of one FSDP unit |
| Activations | $O(BSH \cdot L / \text{AC\_ratio})$ | Forward → backward of that layer |
| Pool slabs (cached) | Variable | Until pressure eviction |
| CUDA context | ~1–2 GB | Permanent |
| Fragmentation | ~5–15% of allocated | Continuous |

---

## 6. Stream Management: Communication-Computation Overlap Architecture

### 6.1 Stream Topology

The `StreamManager` creates four dedicated CUDA streams per rank:

```
ALGORITHM: Initialize_Streams

INPUT: device, is_amd
OUTPUT: StreamManager instance

high_priority ← 0 IF is_amd ELSE -1
// AMD ROCm/HIP: priority 0 is highest schedulable
// NVIDIA CUDA: priority -1 is higher than default (0)

WITH torch.cuda.device(device):
    compute_stream      ← default_stream(device)    // Priority: default
    allgather_stream    ← Stream(device, priority=high_priority)
    reduce_scatter_stream ← Stream(device, priority=high_priority)
    transfer_stream     ← Stream(device, priority=0)   // H2D/D2H
```

| Stream | Purpose | Priority | Overlap Target |
|--------|---------|----------|----------------|
| `compute` | Forward/backward kernels | Default (0) | — |
| `allgather` | Parameter gathering (NCCL all-gather) | High (-1/0) | Compute of previous FSDP unit |
| `reduce_scatter` | Gradient scattering (NCCL reduce-scatter) | High (-1/0) | Compute of next FSDP unit |
| `transfer` | CPU↔GPU data transfers (offload) | Normal (0) | Everything else |

### 6.2 Stream Synchronization Protocol

Inter-stream synchronization uses **CUDA events** (not `synchronize()`) to avoid blocking:

```
ALGORITHM: Sync_Stream_To

INPUT: src_stream, dst_stream

event ← cuda.Event(enable_timing=False, blocking=False)
event.record(src_stream)           // Record completion marker on src
dst_stream.wait_event(event)       // dst waits for src's marker

// Cost: ~1μs (event record + wait) vs. ~100μs (synchronize)
// No CPU-GPU synchronization. No pipeline bubble.
```

Convenience methods:
- `sync_allgather_to_compute()`: After all-gather completes, compute can use the gathered parameters.
- `sync_compute_to_reduce_scatter()`: After backward compute completes, reduce-scatter can begin.

### 6.3 Overlap Timeline

For a single FSDP unit in backward pass:

```
Time →
Compute stream:    |-----backward_compute_k-----|-----backward_compute_{k-1}-----|
AllGather stream:      |--AG_{k-1} (prefetch)----|
ReduceScatter stream:                             |--RS_k (overlap with compute_{k-1})--|
Transfer stream:                                                                   |--D2H grad (if offload)--|
```

The high-priority assignment ensures that communication kernels are scheduled by the GPU scheduler before compute kernels when both are ready, maximizing overlap effectiveness.

### 6.4 AMD-Specific Stream Adaptations

| Aspect | NVIDIA (CUDA) | AMD (ROCm/HIP) |
|--------|--------------|-----------------|
| Highest priority | -1 | 0 |
| Event timing support | Full | Partial (some kernels) |
| GPU timing in MetricsCollector | Enabled | **Disabled** (`enable_gpu_timing=not is_amd`) |
| Stream concurrency | 128+ concurrent kernels | Varies by GCD count on MI300X |

---

## 7. Gradient Accumulation: FSDP-Safe no_sync Protocol

### 7.1 The Problem with Naive Gradient Accumulation Under FSDP

Standard gradient accumulation calls `loss.backward()` for $G$ micro-steps before an optimizer step. Under FSDP, **each `backward()` triggers a reduce-scatter** across all ranks. Without mitigation:

$$
V_{\text{comm}}^{\text{naive}} = G \times \frac{N-1}{N} \times 2\Phi \text{ bytes per cycle}
$$

With `no_sync`:

$$
V_{\text{comm}}^{\text{optimized}} = 1 \times \frac{N-1}{N} \times 2\Phi \text{ bytes per cycle}
$$

The communication reduction is:

$$
\text{Savings} = \frac{G-1}{G} \times 100\%
$$

For $G = 4$ gradient accumulation steps, this is **75% communication reduction**.

### 7.2 The FSDP no_sync Protocol

```
ALGORITHM: FSDP_Gradient_Accumulation

INPUT: loss, accumulation_steps G, current_step counter
OUTPUT: should_sync (bool)

accumulation_counter ← accumulation_counter + 1
scaled_loss ← loss / G

IF accumulation_counter ≥ G:
    // SYNC STEP: Allow FSDP reduce-scatter to execute
    is_sync_step ← True
    scaled_loss ← scale_loss(scaled_loss)  // FP16 loss scaling if needed
    scaled_loss.backward()
    
    RETURN True  // Caller should call step()

ELSE:
    // NON-SYNC STEP: Suppress reduce-scatter
    WITH model.no_sync():
        // Inside no_sync():
        //   - FSDP registers a NOOP for reduce-scatter
        //   - Gradients accumulate in local param.grad buffers
        //   - All-gather still executes (needed for backward compute)
        //   - But reduce-scatter is DEFERRED to sync step
        scaled_loss ← scale_loss(scaled_loss)
        scaled_loss.backward()
    
    RETURN False  // Caller should NOT call step()
```

### 7.3 Communication Pattern with no_sync

| Micro-step | All-Gather (params) | Backward Compute | Reduce-Scatter (grads) | Communication Volume |
|------------|-------------------|-------------------|----------------------|---------------------|
| 1 (non-sync) | ✅ Executes | ✅ Executes | ❌ Suppressed | $\frac{N-1}{N} \cdot 2\Phi$ (AG only) |
| 2 (non-sync) | ✅ Executes | ✅ Executes | ❌ Suppressed | $\frac{N-1}{N} \cdot 2\Phi$ (AG only) |
| 3 (non-sync) | ✅ Executes | ✅ Executes | ❌ Suppressed | $\frac{N-1}{N} \cdot 2\Phi$ (AG only) |
| 4 (sync) | ✅ Executes | ✅ Executes | ✅ Executes | $2 \cdot \frac{N-1}{N} \cdot 2\Phi$ (AG + RS) |

**Total per cycle:** $5 \times \frac{N-1}{N} \cdot 2\Phi$ vs. $8 \times \frac{N-1}{N} \cdot 2\Phi$ without `no_sync`.

### 7.4 Why Manual GradientAccumulator Is Not Used with FSDP

The code includes a `GradientAccumulator` class with named buffer management and optional Triton-fused accumulation. However, for FSDP-wrapped models, **it is intentionally not instantiated**:

```python
# Use native autograd accumulation across micro-steps with
# no_sync(). Manual param.grad injection is unsafe with sharded
# parameters (e.g., local empty shards under FSDP2).
self._gradient_accumulator = None
```

**Root cause:** FSDP shards parameters across ranks. Some ranks may have empty shards for certain parameters (when parameter count is not evenly divisible by world size). The `GradientAccumulator` indexes by parameter name, but the gradient tensor shapes, devices, and even existence differ across ranks. Injecting gradients via `param.grad = buffer` on a shard that expects a different shape causes silent data corruption or NCCL mismatches.

The `no_sync()` context is the **only safe** gradient accumulation mechanism for FSDP because it operates at the FSDP runtime level, deferring the reduce-scatter without modifying gradient tensors directly.

### 7.5 GradientAccumulator Design (For Non-FSDP Use Cases)

The `GradientAccumulator` remains available for non-FSDP training (DDP, single-GPU) and uses Triton-fused accumulation when available:

```
ALGORITHM: Triton_Fused_Gradient_Accumulate

INPUT: grad_ptr, accum_ptr, num_elements, inv_accum_steps
// inv_accum_steps = 1/G (precomputed at init)

KERNEL _fused_gradient_accumulate_kernel:
    pid ← program_id(0)
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < num_elements
    
    grad ← LOAD(grad_ptr + offsets, mask=mask, other=0.0)
    accum ← LOAD(accum_ptr + offsets, mask=mask, other=0.0)
    
    // Fused multiply-add: accum += grad × (1/G)
    accum ← accum + grad × inv_accum_steps
    
    STORE(accum_ptr + offsets, accum, mask=mask)
```

This fuses the scaling ($\times 1/G$) and accumulation ($+=$) into a single kernel, saving one global memory read-write pass compared to:

```python
buffer.add_(grad, alpha=1.0/G)  # PyTorch: 2 kernels (scale + add)
```

---

## 8. Mixed Precision: Numerical Robustness Across Precision Policies

### 8.1 Precision Policy Matrix

| Policy | `param_dtype` | `reduce_dtype` | Loss Scaling | Use Case |
|--------|-------------|---------------|-------------|----------|
| `FULL_BF16` | BF16 | BF16 | No | Default for A100/H100/B200 |
| `FULL_FP16` | FP16 | FP16 | **Yes** (GradScaler) | Legacy GPUs or specific convergence needs |
| `PARAM_FP32` | FP32 | BF16 | No | Debugging, baseline comparison |
| `PURE_FP32` | FP32 | FP32 | No | Numerical validation, no autocast |

### 8.2 BF16 vs. FP16 Numerical Properties

$$
\text{BF16 range:} \quad \pm 3.39 \times 10^{38}, \quad \text{precision:} \approx 3 \text{ decimal digits} \quad (1 + 7 \text{ mantissa bits})
$$

$$
\text{FP16 range:} \quad \pm 6.55 \times 10^{4}, \quad \text{precision:} \approx 3.3 \text{ decimal digits} \quad (1 + 10 \text{ mantissa bits})
$$

BF16 has the **same exponent range as FP32** (8 bits), making it robust against overflow/underflow without loss scaling. FP16's narrow range ($\pm 65504$) requires dynamic loss scaling to prevent gradient underflow.

### 8.3 Loss Scaling for FP16

```
ALGORITHM: Dynamic_Loss_Scaling

// GradScaler configuration
init_scale ← 2^16 = 65536
growth_factor ← 2.0
backoff_factor ← 0.5
growth_interval ← 2000 steps

// Forward pass:
scaled_loss ← loss × current_scale

// Backward pass:
scaled_loss.backward()
// Gradients are now scaled by current_scale

// Before optimizer step:
unscale_(optimizer)
// Divides gradients by current_scale
// Checks for inf/nan

// Optimizer step (only if no inf/nan):
IF no_overflow:
    optimizer.step()
    consecutive_good ← consecutive_good + 1
    IF consecutive_good ≥ growth_interval:
        current_scale ← current_scale × growth_factor
        consecutive_good ← 0
ELSE:
    // Skip optimizer step (gradients are garbage)
    current_scale ← current_scale × backoff_factor
    gradient_overflow_count ← gradient_overflow_count + 1
```

### 8.4 FSDP Mixed Precision Integration

PyTorch FSDP's `MixedPrecision` policy controls three dtype axes:

| Axis | Setting | Effect on FSDP Communication |
|------|---------|------------------------------|
| `param_dtype` | BF16 | All-gather transmits 2 bytes/param (vs. 4 for FP32) |
| `reduce_dtype` | BF16 | Reduce-scatter transmits 2 bytes/grad (vs. 4 for FP32) |
| `buffer_dtype` | BF16 | Non-parameter buffers (BatchNorm stats, etc.) cast to BF16 |

Communication volume per training step with BF16:

$$
V_{\text{BF16}} = 3 \times \frac{N-1}{N} \times 2\Phi \approx 6\Phi \text{ bytes}
$$

With FP32:

$$
V_{\text{FP32}} = 3 \times \frac{N-1}{N} \times 4\Phi \approx 12\Phi \text{ bytes}
$$

BF16 halves communication volume, directly improving scaling efficiency.

### 8.5 Autocast Context Integration

```
ALGORITHM: Forward_Context

CONTEXT MANAGER: forward_context(self)

// Combines autocast + non-blocking forward timing
WITH metrics.measure_forward():                // CUDA event timing
    WITH mp_context.autocast():                 // torch.amp.autocast
        // Inside autocast:
        //   - Linear layers compute in param_dtype (BF16)
        //   - Softmax, LayerNorm, loss in FP32 (auto-promoted)
        //   - Output dtype matches param_dtype
        YIELD
```

---

## 9. Hardware Detection and Cross-Vendor Portability

### 9.1 Hardware Capability Enumeration

The `detect_hardware()` function probes CUDA device properties and infers vendor-specific capabilities:

```
ALGORITHM: Detect_Hardware

INPUT: device_id
OUTPUT: Result[HardwareInfo]

IF NOT cuda.is_available():
    RETURN Err("CUDA runtime not available")

props ← cuda.get_device_properties(device_id)

// Vendor detection via device name pattern matching
vendor ← MATCH props.name AGAINST:
    "NVIDIA", "A100", "H100", "RTX" → NVIDIA
    "AMD", "MI", "INSTINCT"          → AMD
    "INTEL"                           → INTEL
    DEFAULT                           → UNKNOWN

// Compute capability extraction
cc ← ComputeCapability(props.major, props.minor)
// SM version = major × 10 + minor
// SM 80 = A100, SM 89 = L40S/4090, SM 90 = H100/H200, SM 100 = B100/B200

// NVLink detection
has_nvlink ← ANY of ("A100", "H100", "H200", "V100", "DGX") IN props.name

RETURN Ok(HardwareInfo(
    vendor, props.name, cc, props.total_memory,
    props.multi_processor_count,
    props.max_threads_per_multi_processor,
    l2_cache_size, has_nvlink, pcie_bandwidth
))
```

### 9.2 Compute Capability Feature Matrix

| SM Version | GPU | BF16 | FP8 | TMA | Notes |
|-----------|-----|------|-----|-----|-------|
| SM 70 | V100 | ❌ | ❌ | ❌ | FP16 only |
| SM 80 | A100 | ✅ | ❌ | ❌ | BF16 compute, no FP8 |
| SM 89 | L40S, RTX 4090 | ✅ | ✅ | ❌ | FP8 E4M3/E5M2 |
| SM 90 | H100, H200 | ✅ | ✅ | ✅ | TMA, thread block clusters |
| SM 100 | B100, B200 | ✅ | ✅ | ✅ | MXFP8, MXFP6, MXFP4 |

```
ALGORITHM: Feature_Detection

sm_version ← major × 10 + minor

supports_bf16 ← (sm_version ≥ 80)
supports_fp8  ← (sm_version ≥ 89)
supports_tma  ← (sm_version ≥ 90)

// AMD MI300X: gfx942, supports BF16 and FP8
// AMD MI350:  gfx950, supports BF16, FP8, MXFP formats
// Detected via vendor flag, not SM version
```

### 9.3 Vendor-Adaptive Behavior

| Behavior | NVIDIA | AMD (MI300X/MI350) |
|----------|--------|-------------------|
| Stream priority for comm | -1 (high) | 0 (default=highest on HIP) |
| GPU event timing | Enabled | **Disabled** (accuracy issues on ROCm) |
| Triton kernel availability | Full support | Triton-HIP (partial, check at import) |
| CUDA Graphs | Supported | HIP Graphs (partial) |
| NCCL/RCCL | NCCL 2.18+ | RCCL (ROCm 6.x) |
| PCIe bandwidth estimate | 32 GB/s (Gen5, SM≥80) | 32 GB/s (assumed) |

### 9.4 Datacenter GPU Detection

```
ALGORITHM: Is_Datacenter_GPU

patterns ← ("A100", "H100", "H200", "B100", "B200", "MI300", "MI250")
is_datacenter ← ANY(pattern IN device_name FOR pattern IN patterns)

// Used for:
// - Enabling NVLink-optimized collectives
// - Setting aggressive bucket sizes (larger for datacenter interconnects)
// - Enabling advanced features (CPU offload, CUDA graphs)
```

---

## 10. Triton Kernel Integration: Fused Collective Primitives

### 10.1 Kernel Inventory

The system includes four Triton JIT-compiled kernels, each eliminating redundant memory passes:

#### 10.1.1 Fused All-Gather + Scale

```
ALGORITHM: Fused_AllGather_Scale_Kernel

INPUT: src_ptr (local shard), dst_ptr (gathered buffer), scale, num_elements, rank_offset
// Combines the data copy into the all-gather staging buffer
// with a scaling operation (e.g., for mixed-precision cast)

PARALLEL FOR pid IN range(num_blocks):
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < num_elements
    
    data ← LOAD(src_ptr + offsets, mask)
    data ← data × scale                    // Fused scale
    STORE(dst_ptr + rank_offset + offsets, data, mask)

// Saves: 1 kernel launch + 1 global memory pass vs. separate scale + copy
```

#### 10.1.2 Fused Cast and Scale

```
ALGORITHM: Fused_Cast_Scale_Kernel

INPUT: src_ptr (input dtype), dst_ptr (output dtype), scale, num_elements
// Combines dtype cast with scaling in a single pass
// Used for: BF16→FP32 gradient upcasting with 1/G scaling

PARALLEL FOR pid IN range(num_blocks):
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < num_elements
    
    data ← LOAD(src_ptr + offsets, mask)    // Load in source dtype
    data ← data × scale                     // Scale (in compute dtype)
    STORE(dst_ptr + offsets, data, mask)     // Store in target dtype

// Triton handles dtype promotion automatically via tl.load/tl.store
```

#### 10.1.3 Fused Gradient Accumulation

```
ALGORITHM: Fused_Gradient_Accumulate_Kernel

INPUT: grad_ptr, accum_ptr, num_elements, inv_accum_steps
// accum += grad × (1/G) in a single kernel

PARALLEL FOR pid IN range(num_blocks):
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < num_elements
    
    grad ← LOAD(grad_ptr + offsets, mask, other=0.0)
    accum ← LOAD(accum_ptr + offsets, mask, other=0.0)
    accum ← accum + grad × inv_accum_steps
    STORE(accum_ptr + offsets, accum, mask)

// Saves: 2 kernel launches + 1 memory pass vs. PyTorch buffer.add_(grad, alpha=1/G)
// PyTorch decomposes to: mul kernel + add kernel (2 passes over accum)
```

#### 10.1.4 Fused Parameter Sharding

```
ALGORITHM: Fused_Param_Shard_Kernel

INPUT: full_param_ptr, shard_ptr, full_numel, shard_size, rank_offset
// Extracts rank-local shard from all-gathered full parameter
// Replaces: shard = full_param[rank_offset : rank_offset + shard_size]

PARALLEL FOR pid IN range(num_blocks):
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < shard_size
    src_offsets ← rank_offset + offsets
    full_mask ← mask AND (src_offsets < full_numel)
    
    data ← LOAD(full_param_ptr + src_offsets, mask=full_mask, other=0.0)
    STORE(shard_ptr + offsets, data, mask)

// Autotuned across BLOCK_SIZE ∈ {1024, 2048, 4096, 8192}
// Key: shard_size (autotuner selects best config per shard size)
```

### 10.2 Autotune Configuration

The parameter sharding kernel uses Triton's `@autotune` decorator:

| Config | BLOCK_SIZE | num_warps | Best For |
|--------|-----------|-----------|----------|
| 1 | 1024 | 4 | Small shards (< 1M elements) |
| 2 | 2048 | 8 | Medium shards (1M–10M elements) |
| 3 | 4096 | 8 | Large shards (10M–100M elements) |
| 4 | 8192 | 16 | Very large shards (> 100M elements) |

The autotune key is `shard_size`, ensuring the best configuration is cached per unique shard size encountered during training.

### 10.3 Fallback Path

When Triton is unavailable, all operations fall back to PyTorch-native operators:

```
IF NOT TRITON_AVAILABLE:
    // Gradient accumulation fallback:
    buffer.add_(grad, alpha=inv_accum_steps)
    // ~15% slower than fused Triton (two kernel launches + extra memory pass)
```

---

## 11. Metrics Collection: Zero-Stall GPU Instrumentation

### 11.1 The Synchronization Problem

Naive GPU timing requires `torch.cuda.synchronize()`, which:
- Blocks the CPU until all GPU work completes.
- Creates a pipeline bubble between CPU-side scheduling and GPU-side execution.
- Costs 50–200μs per synchronization on H100 (PCIe latency + drain).

At scale, dozens of timing calls per step create **milliseconds of stall** that directly reduce MFU.

### 11.2 Non-Blocking Event-Based Timing

The `MetricsCollector` uses CUDA events with **deferred resolution**:

```
ALGORITHM: NonBlocking_GPU_Timing

// Recording phase (in hot path — zero stall):
CONTEXT MANAGER measure_gpu(field_name):
    start_event ← cuda.Event(enable_timing=True)
    end_event ← cuda.Event(enable_timing=True)
    
    start_event.record()              // Record on current stream (~100ns)
    YIELD                             // Execute timed operation
    end_event.record()                // Record completion (~100ns)
    
    // DO NOT CALL start.elapsed_time(end) HERE
    // That would require synchronization
    
    WITH lock:
        pending_events.append((start_event, end_event, field_name))

// Resolution phase (outside hot path — called at record_step):
FUNCTION _flush_pending_events():
    remaining ← []
    FOR (start, end, field) IN pending_events:
        IF end.query():               // Non-blocking check: is event done?
            elapsed_ms ← start.elapsed_time(end)  // Safe: event completed
            elapsed_ns ← int(elapsed_ms × 1e6)
            current[field] ← current[field] + elapsed_ns
        ELSE:
            remaining.append((start, end, field))  // Not yet done, retry later
    pending_events ← remaining
```

**Key insight:** `event.query()` returns `True/False` without blocking. `elapsed_time()` is only called after `query()` confirms completion. This **never** blocks the CPU.

### 11.3 Metrics Schema

```
DATACLASS FSDPMetrics:
    // Communication timing (nanoseconds)
    allgather_time_ns: int
    reduce_scatter_time_ns: int
    
    // Computation timing (nanoseconds)
    forward_time_ns: int
    backward_time_ns: int
    optimizer_time_ns: int
    
    // Memory (bytes)
    peak_memory_bytes: int
    allocated_memory_bytes: int
    reserved_memory_bytes: int
    
    // Throughput
    tokens_processed: int
    samples_processed: int
    
    // Numerical health
    gradient_norm: float
    gradient_overflow_count: int
```

### 11.4 Rolling Average Computation

```
ALGORITHM: Compute_Rolling_Average

INPUT: last_n steps (default 100)
OUTPUT: Dict[str, float] averaged metrics

WITH lock:
    _flush_pending_events()           // Resolve any completed events
    history ← self._history[-last_n:]

IF history IS EMPTY:
    RETURN {}

n ← len(history)
RETURN {
    "avg_allgather_ms":       SUM(h.allgather_time_ns) / n / 1e6,
    "avg_reduce_scatter_ms":  SUM(h.reduce_scatter_time_ns) / n / 1e6,
    "avg_forward_ms":         SUM(h.forward_time_ns) / n / 1e6,
    "avg_backward_ms":        SUM(h.backward_time_ns) / n / 1e6,
    "avg_optimizer_ms":       SUM(h.optimizer_time_ns) / n / 1e6,
    "max_peak_memory_gb":     MAX(h.peak_memory_bytes) / 2^30,
}
```

### 11.5 AMD Timing Limitation

On AMD GPUs, CUDA event timing is **disabled** due to accuracy issues in ROCm's HIP event implementation:

```
self._metrics = MetricsCollector(enable_gpu_timing=not self._is_amd)
```

When disabled, all timing context managers become no-ops, and the metrics contain zeros for timing fields. Profiling on AMD must use `rocprof` or `roctracer` externally.

---

## 12. Activation Checkpointing: Selective Recomputation Strategies

### 12.1 Checkpointing Modes

| Mode | Behavior | Memory Savings | Recomputation Cost |
|------|----------|----------------|-------------------|
| `full` | Checkpoint every eligible layer | Maximum (~60% activation reduction) | ~33% compute overhead |
| `selective` | Checkpoint every $k$-th layer (default $k=2$) | Moderate (~30% activation reduction) | ~15% compute overhead |
| `offload` | Checkpoint + offload activations to CPU | Maximum + CPU memory | D2H/H2D transfer overhead |

### 12.2 Selective Checkpointing Decision

```
ALGORITHM: Selective_AC_Check

INPUT: module, layer_counter, ac_frequency k, ac_mode
OUTPUT: should_checkpoint (bool)

// Only checkpoint layer-like modules
name ← module.__class__.__name__.lower()
is_layer ← ANY("layer", "block", "decoder", "encoder") IN name

IF NOT is_layer:
    RETURN False

IF ac_mode == "full":
    RETURN True

IF ac_mode == "selective":
    idx ← layer_counter
    layer_counter ← layer_counter + 1
    RETURN (idx MOD k == 0)  // Every k-th layer

RETURN True  // Default: checkpoint
```

For a 32-layer model with `ac_frequency=2`:
- Layers 0, 2, 4, ..., 30 are checkpointed (16 layers).
- Layers 1, 3, 5, ..., 31 retain activations in memory.
- Activation memory is approximately halved compared to no checkpointing.

### 12.3 Ordering Constraint: AC Before FSDP

```
ALGORITHM: Model_Wrapping_Order

// CRITICAL: Activation checkpointing MUST precede FSDP wrapping

Step 1: apply_activation_checkpointing(model)
    // Wraps selected layers with checkpoint_wrapper
    // Uses NO_REENTRANT implementation (safe with FSDP)

Step 2: FSDP(model, ...)
    // FSDP wraps the already-checkpointed model
    // FSDP's own weight freeing (selective AC on parametrization)
    // composes with user's activation checkpointing
```

**Why this order:** FSDP's `auto_wrap_policy` partitions the model into FSDP units based on module structure. If FSDP wraps first, the checkpoint wrapper cannot see the original module boundaries. If AC wraps first, FSDP sees the `CheckpointWrapper` modules and wraps them as atomic units.

### 12.4 Memory Impact

For a Transformer layer with hidden dimension $H$, sequence length $S$, batch size $B$:

**Without checkpointing (per layer):**

$$
M_{\text{act}}^{\text{layer}} \approx 2BSH \cdot (1 + \frac{4n_{\text{heads}} \cdot S}{H} + 8) \approx 34BSH \text{ bytes (BF16)}
$$

**With full checkpointing (per layer):**

$$
M_{\text{act}}^{\text{ckpt}} \approx 2BSH \text{ bytes (only input activation saved)}
$$

**Memory reduction per layer:**

$$
\Delta M = 34BSH - 2BSH = 32BSH \text{ bytes}
$$

For a 70B model ($H = 8192$, $S = 4096$, $B = 1$, $L = 80$):

$$
\Delta M_{\text{total}} = 80 \times 32 \times 1 \times 4096 \times 8192 \approx 80 \text{ GB}
$$

With selective checkpointing ($k=2$, 40 layers checkpointed):

$$
\Delta M_{\text{selective}} \approx 40 \text{ GB saved}
$$

---

## 13. Auto-Wrap Policy: Transformer-Aware Layer Discovery

### 13.1 Layer Class Discovery

The auto-wrap policy searches for known Transformer layer classes across common model libraries:

```
ALGORITHM: Auto_Wrap_Policy_Resolution

INPUT: auto_wrap_policy (list of class names)
OUTPUT: PyTorch auto_wrap_policy callable

// Phase 1: Resolve class names to Python classes
layer_classes ← SET()
FOR cls_name IN auto_wrap_policy:
    cls ← _try_import_class(cls_name)
    IF cls IS NOT None:
        layer_classes.add(cls)

// Phase 2: Select wrapping strategy
IF layer_classes IS NOT EMPTY:
    // Transformer-specific wrapping: one FSDP unit per layer class
    RETURN partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls=layer_classes
    )
ELSE:
    // Fallback: size-based wrapping (one unit per min_num_params)
    RETURN partial(
        size_based_auto_wrap_policy,
        min_num_params=100_000_000  // 100M params per FSDP unit
    )
```

### 13.2 Supported Layer Classes

| Class Name | Model Family | Import Path |
|------------|-------------|-------------|
| `LlamaDecoderLayer` | Llama 2/3 | `transformers.models.llama.modeling_llama` |
| `MistralDecoderLayer` | Mistral/Mixtral | `transformers.models.mistral.modeling_mistral` |
| `Qwen2DecoderLayer` | Qwen 2 | `transformers.models.qwen2.modeling_qwen2` |
| `GPT2Block` | GPT-2 | `transformers.models.gpt2.modeling_gpt2` |
| `FalconDecoderLayer` | Falcon | `transformers.models.falcon.modeling_falcon` |
| `GemmaDecoderLayer` | Gemma | `transformers.models.gemma.modeling_gemma` |
| `Phi3DecoderLayer` | Phi-3 | `transformers.models.phi3.modeling_phi3` |
| `TransformerEncoderLayer` | Generic | `torch.nn` |
| `TransformerDecoderLayer` | Generic | `torch.nn` |
| `TransformerSentenceEncoderLayer` | Fairseq | `fairseq.modules` |

### 13.3 Why Transformer-Aware Wrapping Matters

**Per-layer wrapping** (one FSDP unit per Transformer layer) is optimal because:

1. **Uniform memory footprint:** Each layer has approximately equal parameter count ($\approx 12H^2$), producing balanced shard sizes.
2. **Natural pipeline boundary:** Forward/backward execution is sequential by layer—wrapping per layer enables maximum overlap between adjacent units' communication and computation.
3. **Activation checkpointing alignment:** Checkpointing is naturally per-layer; matching FSDP boundaries to AC boundaries avoids cross-boundary recomputation.

**Size-based wrapping** (fallback) is suboptimal because it may split a single attention block across two FSDP units, creating unnecessary all-gather/reduce-scatter boundaries within a logically atomic computation.

---

## 14. Failure Mode Taxonomy and Root-Cause Resolution Chain

### 14.1 Complete Fix Registry

| Fix ID | Category | Symptom | Root Cause | Resolution | Version |
|--------|----------|---------|------------|------------|---------|
| FIX-001 | Memory | OOM during training | Unbounded allocation without pooling | `MemoryPool` with pressure-aware eviction | v2.0 |
| FIX-002 | Compute | Low GPU utilization, CPU stalls | `torch.cuda.synchronize()` in metrics | Non-blocking CUDA event timing | v2.0 |
| FIX-003 | Communication | $G\times$ excess reduce-scatter during accumulation | Missing `no_sync()` for non-sync micro-steps | `_no_sync_context()` wrapping non-sync backward | v2.0 |
| FIX-004 | Portability | Crashes on MI300X | CUDA-specific stream priorities, timing | AMD detection → adaptive priorities, disabled event timing | v2.0 |
| FIX-005 | Device Affinity | 30-minute hang → NCCL crash during checkpoint | `offload_to_cpu` creates staging tensors on `cuda:0` | `_enforce_device_affinity()` context, re-pinning after every API call | v2.1 |
| FIX-006 | API Deprecation | Hundreds of `FutureWarning` per save | Legacy `FSDP.state_dict_type()` uses `ShardedTensor` | Modern DCP API (`get_state_dict`/`set_state_dict`) with DTensor | v2.1 |
| FIX-007 | Performance | 50–100s checkpoint stall at 78% VRAM | Two separate all-gather scopes + no memory cleanup between | Single `get_state_dict` call, proactive GC, reduced NCCL timeout | v2.1 |
| FIX-008 | UX | Warning spam in logs | Four categories of known-harmless deprecation warnings | Targeted `warnings.filterwarnings()` at module import, fail-open | v2.1 |
| FIX-009 | Correctness | Overwrite warning noise | `FileSystemWriter` default `overwrite=False` | Explicit `overwrite=True` in constructor | v2.1 |

### 14.2 Failure Chain Visualization

```
USER ACTION: save_checkpoint() at 78% VRAM

    ┌──────────────────────────────────┐
    │ FIX-007: Pre-save memory cleanup │
    │ gc.collect() + empty_cache()     │
    │ Release pool slabs (FIX-001)     │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-005: Device affinity guard   │
    │ torch.cuda.set_device(rank_dev)  │
    │ torch.cuda.device(rank_dev)      │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-006: API selection           │
    │ IF modern DCP → single scope     │
    │ ELSE legacy → dual scope + GC    │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-008: Warning suppression     │
    │ (legacy path only)               │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-009: FileSystemWriter        │
    │ overwrite=True                   │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-005: Post-save verification  │
    │ O(P) device scan + migration     │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │ FIX-007: Post-save barrier       │
    │ 5-minute timeout (vs. 30-min)    │
    └──────────────────────────────────┘
```

### 14.3 Diagnostic Decision Tree

```
ALGORITHM: Diagnose_FSDP_Failure

SYMPTOM: Training hangs with no error message

    Q1: Is the hang during checkpoint save?
    ├── YES → Check FIX-005: device affinity
    │         Run: torch.cuda.current_device() on each rank
    │         If any rank shows cuda:0 AND rank ≠ 0 → DEVICE DRIFT
    │         Fix: _enforce_device_affinity() context
    │
    └── NO → Q2: Is the hang during backward?
            ├── YES → Check FIX-003: no_sync missing
            │         Monitor: NCCL traffic during non-sync micro-steps
            │         If reduce-scatter observed → no_sync not active
            │
            └── NO → Q3: Is the hang at barrier?
                    ├── YES → Check FIX-007: NCCL timeout
                    │         One rank may have failed silently
                    │         Set NCCL_TIMEOUT=300 for faster diagnosis
                    │
                    └── NO → Check NCCL logs for topology mismatch

SYMPTOM: OOM during checkpoint save

    Check FIX-007: Pre-save memory cleanup
    Verify: VRAM utilization before save (target < 85%)
    If > 85%: increase GC aggressiveness or reduce model size
    If using legacy API: ensure GC between model + optimizer scopes

SYMPTOM: Hundreds of FutureWarning in logs

    Check FIX-008: Warning suppression installed?
    Verify: _suppress_known_deprecation_warnings() called at import
    If yes but warnings persist: new PyTorch version with new warnings
    → Update suppression patterns (fail-open means new warnings visible)
```

---

## 15. Production Deployment: End-to-End Integration Contract

### 15.1 Complete Training Step Flow

```
ALGORITHM: Production_Training_Step

// ==================== INITIALIZATION ====================
config ← FSDP2Config(
    sharding_strategy=FULL_SHARD,
    mixed_precision=FULL_BF16,
    activation_checkpointing=True,
    ac_mode="selective",
    ac_frequency=2,
    gradient_accumulation_steps=4,
    gradient_clipping_norm=1.0,
    use_triton_kernels=True,
    use_memory_pool=True,
    bucket_size_mb=25,
    forward_prefetch=True,
    backward_prefetch=BACKWARD_PRE,
    limit_all_gathers=True
)

fsdp ← SOTAFSDP2(config)
model ← create_model(model_config)
wrapped_model ← fsdp.wrap_model(model)
optimizer ← AdamW(wrapped_model.parameters(), lr=lr)

// ==================== TRAINING LOOP ====================
FOR step = 1 TO max_steps:
    FOR micro_step = 1 TO gradient_accumulation_steps:
        batch ← dataloader.next()
        
        // Forward pass [INT-007]
        WITH fsdp.forward_context():
            loss ← wrapped_model(batch.input_ids, batch.labels)
        
        // Backward pass [INT-003, INT-010]
        should_sync ← fsdp.backward(loss)
        // micro_steps 1,2,3: should_sync=False (no_sync active)
        // micro_step 4: should_sync=True (reduce-scatter executes)
    
    // Optimizer step [INT-004] — only after sync step
    IF should_sync:
        fsdp.step(optimizer, scheduler)
        // Internally:
        //   1. unscale_grads(optimizer)     — FP16 only
        //   2. clip_grad_norm_(1.0)         — distributed FSDP clip
        //   3. optimizer.step()             — AdamW update
        //   4. scheduler.step()             — LR schedule
        //   5. optimizer.zero_grad(set_to_none=True)
        //   6. accumulation_counter ← 0
        //   7. metrics.record_step()
        //   8. IF VRAM > 90%: gc + empty_cache
    
    // Checkpoint [INT-006]
    IF step MOD checkpoint_interval == 0:
        result ← FSDPCheckpointManager.save_checkpoint(
            fsdp, optimizer, checkpoint_dir,
            epoch=epoch, step=step, sharded=True
        )
        IF result.is_err():
            LOG_ERROR(result.error)
    
    // Logging [INT-009]
    IF step MOD log_interval == 0:
        LOG(fsdp.memory_summary())
        LOG(fsdp.metrics.get_average(last_n=100))
```

### 15.2 Cluster Configuration Matrix

| Cluster | GPU | HBM | Interconnect | Recommended Config |
|---------|-----|-----|-------------|-------------------|
| 8× A100 80GB | A100 | 80 GB | NVLink 600 GB/s | FULL_SHARD, BF16, AC selective, bucket 25MB |
| 8× H100 80GB | H100 | 80 GB | NVSwitch 900 GB/s | FULL_SHARD, BF16, AC selective, bucket 25MB, forward_prefetch |
| 8× H200 141GB | H200 | 141 GB | NVSwitch 900 GB/s | FULL_SHARD, BF16, bucket 50MB, larger batches |
| 8× B200 192GB | B200 | 192 GB | NVLink5 1.8 TB/s | FULL_SHARD, BF16, bucket 100MB, limit_all_gathers=False |
| 8× MI300X 192GB | MI300X | 192 GB | xGMI 896 GB/s | FULL_SHARD, BF16, Triton off (HIP), event timing off |
| 4× MI350 288GB | MI350 | 288 GB | xGMI | FULL_SHARD, BF16, larger FSDP units |

### 15.3 Per-Rank Memory Budget Verification

Before launching training, verify the model fits within the per-rank HBM budget:

$$
M_{\text{required}}^{\text{rank}} = \underbrace{\frac{16\Phi}{W}}_{\text{sharded state}} + \underbrace{2\Phi_{\text{unit}}}_{\text{transient AG}} + \underbrace{L \cdot M_{\text{act}}^{\text{layer}} / \text{AC\_factor}}_{\text{activations}} + \underbrace{2 \text{ GB}}_{\text{CUDA context + buffers}}
$$

$$
M_{\text{required}}^{\text{rank}} \leq 0.90 \times M_{\text{HBM}} \quad \text{(VRAM\_PRESSURE\_WATERMARK)}
$$

For a 70B model on 64× H100 80GB ($W=64$, $H=8192$, $L=80$, $S=4096$, $B=1$):

$$
M_{\text{sharded}} = \frac{16 \times 70 \times 10^9}{64} = 17.5 \text{ GB}
$$

$$
M_{\text{transient}} = 2 \times \frac{12 \times 8192^2}{64} \approx 25 \text{ MB (per FSDP unit, 1 layer)}
$$

$$
M_{\text{act}} = 80 \times 2 \times 1 \times 4096 \times 8192 / 2 \approx 2.5 \text{ GB (selective AC, k=2)}
$$

$$
M_{\text{total}} \approx 17.5 + 0.025 + 2.5 + 2.0 = 22.0 \text{ GB} \ll 72 \text{ GB (90\% of 80GB)}
$$

**Verdict:** Fits with significant headroom. Can increase batch size or sequence length.

### 15.4 Result Type Usage Pattern

The `Result[T]` type eliminates exception-based control flow for recoverable errors:

```
ALGORITHM: Result_Type_Pattern

// Checkpoint save with graceful degradation
result ← FSDPCheckpointManager.save_checkpoint(fsdp, optimizer, path)

IF result.is_ok():
    // Checkpoint saved successfully
    LOG("Checkpoint saved")
    
ELSE:  // result.is_err()
    // Checkpoint failed, but training continues
    LOG_ERROR(f"Checkpoint failed: {result.error} (code={result.code})")
    // Training is NOT interrupted
    // Next checkpoint will be attempted at the next interval
    // This is INTENTIONAL: checkpoint failure should not crash training

// Map pattern for chaining
result = detect_hardware(device_id).map(lambda hw: hw.memory_gb)
// Ok(HardwareInfo) → Ok(80.0)
// Err("not available") → Err("not available")
```

---

## 16. Summary: Key Engineering Decisions and Their Rationale

| Decision | Rationale | Impact |
|----------|-----------|--------|
| Device affinity guard on every checkpoint op | Prevents 30-minute NCCL hangs from cuda:0 drift | Eliminates the #1 production failure mode |
| Modern DCP API with legacy fallback | Zero deprecation warnings, 2× faster saves, correct device tracking | Forward-compatible, backward-compatible |
| Single `get_state_dict` call for model+optimizer | One all-gather instead of two; prevents OOM at 78% VRAM | 50–100s → ~15s checkpoint time |
| `no_sync()` for non-sync micro-steps | Reduces communication by $(G-1)/G$ during gradient accumulation | 75% communication reduction for $G=4$ |
| Non-blocking CUDA event timing | Eliminates CPU-GPU synchronization in hot path | ~0.5–2% MFU improvement |
| Power-of-2 memory pool with pressure eviction | Eliminates cudaMalloc jitter; prevents OOM via proactive eviction | Stable step times, no OOM surprises |
| AMD-specific stream priority and timing | ROCm HIP has different priority semantics and event accuracy | Training runs correctly on MI300X/MI350 |
| Atomic checkpoint writes (tmp + rename) | Crash during save never corrupts existing checkpoint | Zero checkpoint corruption in production |
| Triton-fused gradient accumulation | Single kernel for scale+accumulate vs. two PyTorch kernels | ~15% gradient accumulation speedup |
| Targeted warning suppression (fail-open) | Clean logs without hiding novel warnings | Developer experience + debuggability |
| Result type for error handling | Checkpoint failures don't crash training; errors are explicit | Production resilience |
| AC before FSDP wrapping order | Ensures FSDP sees checkpoint-wrapped modules as atomic units | Correct composition of memory optimizations |

---

*This report documents the SOTAFSDP2 v2.1 production system for FSDP-based distributed training, covering the full failure-mode taxonomy, checkpoint safety protocol, memory management strategy, and cross-vendor portability layer for deployment on A100, H100, H200, B100, B200, MI300X, MI325X, and MI350-class accelerator clusters at scales from single-node to multi-thousand GPU.*



# Technical Report: Production-Grade FSDP2 Implementation — SOTAFSDP2 v2.1

## Fully Sharded Data Parallelism with Device-Safe Checkpointing, Triton-Accelerated Primitives, and Cross-Vendor Hardware Support

---

## Table of Contents

1. [Executive Architecture Overview](#1-executive-architecture-overview)
2. [FSDP2 Sharding Strategies: Memory-Communication Tradeoff Algebra](#2-fsdp2-sharding-strategies)
3. [Collective Communication Primitives and Decomposition](#3-collective-communication-primitives)
4. [SimpleFSDP Compiler Optimizations: Bucketing and Reordering](#4-simplefsdp-compiler-optimizations)
5. [SOTAFSDP2 Engine: Component Architecture](#5-sotafsdp2-engine-component-architecture)
6. [Device Affinity Enforcement: Root Cause Analysis of cuda:0 Drift](#6-device-affinity-enforcement)
7. [Checkpoint System: Modern DCP vs. Legacy FSDP1 Pathways](#7-checkpoint-system)
8. [Memory Management: Pool Architecture and Pressure-Aware Eviction](#8-memory-management)
9. [Stream Manager: Communication-Computation Overlap Architecture](#9-stream-manager)
10. [Triton-Accelerated Kernel Suite](#10-triton-accelerated-kernel-suite)
11. [Gradient Accumulation with FSDP no_sync Semantics](#11-gradient-accumulation)
12. [Mixed Precision: Numerical Robustness and Loss Scaling](#12-mixed-precision)
13. [Activation Checkpointing: Memory-Compute Tradeoff Engineering](#13-activation-checkpointing)
14. [Hardware Detection and Cross-Vendor Portability](#14-hardware-detection)
15. [Metrics Collection: Non-Blocking GPU Timing](#15-metrics-collection)
16. [Auto-Wrapping Algorithm: Formal Specification](#16-auto-wrapping-algorithm)
17. [Production Bug Taxonomy: FIX-001 through FIX-009](#17-production-bug-taxonomy)
18. [End-to-End Training Integration Contract](#18-training-integration-contract)
19. [Performance Engineering: MFU, Step-Time Decomposition](#19-performance-engineering)
20. [Summary of Engineering Principles](#20-summary)

---

## 1. Executive Architecture Overview

SOTAFSDP2 v2.1 is a production-grade Fully Sharded Data Parallel engine designed for training large language models (LLMs) across heterogeneous GPU clusters spanning NVIDIA (A100, H100, H200, B100, B200) and AMD (MI300X, MI325X) accelerator families. The implementation addresses nine categorized production failure modes (FIX-001 through FIX-009) discovered during multi-node training at scale, and provides a clean integration contract with an external trainer (SOTATrainer) through ten well-defined interface points (INT-001 through INT-010).

### 1.1 System Component Map

| Component | Responsibility | Key Classes |
|---|---|---|
| **Sharding Engine** | Parameter/gradient/optimizer-state partitioning across data-parallel ranks | `SOTAFSDP2`, `ShardingStrategy` |
| **Communication Overlap** | Multi-stream scheduling of all-gather, reduce-scatter, compute | `StreamManager` |
| **Memory Subsystem** | Pool allocation, pressure-aware eviction, fragmentation control | `MemoryPool` |
| **Checkpoint Manager** | Device-safe, atomic, async-capable save/load with API version detection | `FSDPCheckpointManager` |
| **Precision Control** | BF16/FP16/FP32 autocast, loss scaling, gradient dtype management | `MixedPrecisionContext`, `MixedPrecisionPolicy` |
| **Gradient Accumulation** | FSDP-safe micro-step accumulation with `no_sync` | `GradientAccumulator`, `SOTAFSDP2.backward()` |
| **Kernel Acceleration** | Fused Triton kernels for gradient accumulation, parameter sharding, casting | Triton JIT kernels |
| **Metrics** | Non-blocking CUDA event timing, memory tracking | `MetricsCollector`, `FSDPMetrics` |
| **Hardware Abstraction** | Vendor detection, compute capability enumeration, feature gating | `HardwareInfo`, `HardwareVendor`, `ComputeCapability` |

### 1.2 Design Invariants

The following invariants hold across all execution paths:

> **Invariant 1 (Device Affinity):** Every tensor operation on rank $r$ with local rank $l$ executes exclusively on device `cuda:$l$`. No operation may create, move, or reference tensors on `cuda:0` from a non-zero local rank. This is enforced by `_enforce_device_affinity()`.

> **Invariant 2 (Communication Correctness):** Reduce-scatter is issued exactly once per training step during the sync micro-step. All non-sync micro-steps suppress reduce-scatter via `no_sync()`. Gradient accumulation across micro-steps preserves numerical equivalence to single-step training with the effective batch size.

> **Invariant 3 (Checkpoint Atomicity):** Every checkpoint save is atomic—either fully committed or fully absent. Partial writes are prevented by tmp-file-then-rename on full saves and by DCP's built-in transaction semantics on sharded saves.

---

## 2. FSDP2 Sharding Strategies: Memory-Communication Tradeoff Algebra

### 2.1 Strategy Taxonomy

SOTAFSDP2 supports four sharding strategies, each defining a different partitioning of the three training state categories across $W$ data-parallel ranks for a model with $P$ total parameters:

| Strategy | Parameters | Gradients | Optimizer States | Per-Rank Memory | Communication per Step |
|---|---|---|---|---|---|
| `FULL_SHARD` (ZeRO-3) | Sharded | Sharded | Sharded | $O(3P/W)$ | $2 \times$ all-gather + reduce-scatter |
| `SHARD_GRAD_OP` (ZeRO-2) | Replicated | Sharded | Sharded | $O(P + 2P/W)$ | reduce-scatter only |
| `NO_SHARD` (DDP) | Replicated | Replicated | Replicated | $O(3P)$ | all-reduce |
| `HYBRID_SHARD` | Sharded intra-node, replicated inter-node | Mixed | Mixed | Between FULL and NO | Hierarchical collectives |

### 2.2 Per-Rank Memory Formulas

For BF16 parameters and gradients ($d_\theta = d_g = 2$ bytes) with FP32 Adam optimizer states ($d_m = d_v = d_{\text{master}} = 4$ bytes):

**FULL_SHARD:**

$$
M_{\text{full\_shard}} = \frac{P}{W} \times (2 + 2 + 4 + 4 + 4) = \frac{16P}{W} \text{ bytes}
$$

**SHARD_GRAD_OP:**

$$
M_{\text{shard\_grad\_op}} = P \times 2 + \frac{P}{W} \times (2 + 4 + 4 + 4) = 2P + \frac{14P}{W} \text{ bytes}
$$

**NO_SHARD:**

$$
M_{\text{no\_shard}} = P \times (2 + 2 + 4 + 4 + 4) = 16P \text{ bytes}
$$

**HYBRID_SHARD** (with intra-node size $W_{\text{intra}}$, inter-node size $W_{\text{inter}}$, $W = W_{\text{intra}} \times W_{\text{inter}}$):

$$
M_{\text{hybrid}} = \frac{P}{W_{\text{intra}}} \times (2 + 2 + 4 + 4 + 4) = \frac{16P}{W_{\text{intra}}} \text{ bytes}
$$

### 2.3 Communication Volume Analysis

For each strategy, the total communication volume per training step per rank is:

**FULL_SHARD:**

$$
V_{\text{full\_shard}} = \underbrace{2 \times P \times d_\theta \times \frac{W-1}{W}}_{\text{2 all-gathers (fwd + bwd)}} + \underbrace{P \times d_g \times \frac{W-1}{W}}_{\text{reduce-scatter}} \approx 6P \text{ bytes (BF16)}
$$

The factor of 2 for all-gather arises because FSDP performs one all-gather in the forward pass and a second all-gather in the backward pass (since full parameters are freed after forward to save memory).

**SHARD_GRAD_OP:**

$$
V_{\text{shard\_grad\_op}} = \underbrace{P \times d_g \times \frac{W-1}{W}}_{\text{reduce-scatter}} \approx 2P \text{ bytes (BF16)}
$$

No all-gather needed since parameters are replicated.

**NO_SHARD (DDP):**

$$
V_{\text{no\_shard}} = \underbrace{2 \times P \times d_g \times \frac{W-1}{W}}_{\text{all-reduce (= RS + AG)}} \approx 4P \text{ bytes (BF16)}
$$

### 2.4 Strategy Selection Decision Tree

```
PSEUDOCODE: Sharding Strategy Selection
────────────────────────────────────────
function SELECT_STRATEGY(P, W, M_HBM, BW_intra, BW_inter):
    
    M_full = 16 * P / W
    M_shard_grad = 2 * P + 14 * P / W
    M_no_shard = 16 * P
    
    // Phase 1: Eliminate strategies that don't fit in HBM
    candidates ← []
    if M_full + M_activation_estimate < M_HBM:
        candidates.add(FULL_SHARD)
    if M_shard_grad + M_activation_estimate < M_HBM:
        candidates.add(SHARD_GRAD_OP)
    if M_no_shard + M_activation_estimate < M_HBM:
        candidates.add(NO_SHARD)
    
    if candidates is empty:
        ERROR("Model does not fit; increase W or enable CPU offload")
    
    // Phase 2: Among feasible strategies, minimize step time
    // FULL_SHARD: more communication but smaller memory → larger batch
    // SHARD_GRAD_OP: less communication but more memory → smaller batch
    // NO_SHARD: least communication but most memory → smallest batch possible
    
    // For frontier LLMs (P > 10B), FULL_SHARD is typically the only
    // feasible option with reasonable activation memory
    
    // Phase 3: HYBRID_SHARD if cluster has heterogeneous interconnect
    if BW_intra >> BW_inter AND W > W_intra:
        // Shard within NVLink domain, replicate across IB
        candidates.add(HYBRID_SHARD)
    
    return argmin(candidates, key=estimated_step_time)
```

### 2.5 Semantic Methods on ShardingStrategy

The implementation attaches semantic queries to each strategy enum:

| Method | FULL_SHARD | SHARD_GRAD_OP | NO_SHARD | HYBRID_SHARD |
|---|---|---|---|---|
| `requires_all_gather()` | ✅ | ❌ | ❌ | ✅ |
| `requires_reduce_scatter()` | ✅ | ✅ | ❌ | ✅ |

These methods drive conditional logic throughout the engine: if a strategy does not require all-gather, the forward-pass all-gather path and the re-gather in backward are both skipped, eliminating unnecessary stream synchronization and event recording.

---

## 3. Collective Communication Primitives and Decomposition

### 3.1 Fundamental Decomposition

The identity that architecturally enables FSDP is:

$$
\textbf{All-Reduce} = \textbf{Reduce-Scatter} + \textbf{All-Gather}
$$

This decomposition permits FSDP to temporally separate gradient synchronization (reduce-scatter at backward exit) from parameter reconstruction (all-gather at next forward entry), creating scheduling freedom for overlap.

### 3.2 All-Reduce

**Semantics:** Each of $W$ GPUs holds a tensor of identical shape. All-reduce computes an element-wise reduction (sum or average) and places the **complete result** on every GPU.

**Before (4 GPUs):**

| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A$ | $B$ | $C$ | $D$ |

**After:**

| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A{+}B{+}C{+}D$ | $A{+}B{+}C{+}D$ | $A{+}B{+}C{+}D$ | $A{+}B{+}C{+}D$ |

**Cost (ring):**

$$
T_{\text{all-reduce}} = 2(W-1) \cdot \alpha + 2 \cdot \frac{W-1}{W} \cdot n \cdot \beta
$$

### 3.3 Reduce-Scatter

**Semantics:** Each GPU holds a tensor partitioned into $W$ chunks. Reduce-scatter reduces corresponding chunks across GPUs and delivers chunk $i$ to GPU $i$.

**Before (4 GPUs, tensor split into 4 chunks):**

| | Chunk 0 | Chunk 1 | Chunk 2 | Chunk 3 |
|---|---|---|---|---|
| GPU 0 | $A_0$ | $A_1$ | $A_2$ | $A_3$ |
| GPU 1 | $B_0$ | $B_1$ | $B_2$ | $B_3$ |
| GPU 2 | $C_0$ | $C_1$ | $C_2$ | $C_3$ |
| GPU 3 | $D_0$ | $D_1$ | $D_2$ | $D_3$ |

**After:**

| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A_0{+}B_0{+}C_0{+}D_0$ | $A_1{+}B_1{+}C_1{+}D_1$ | $A_2{+}B_2{+}C_2{+}D_2$ | $A_3{+}B_3{+}C_3{+}D_3$ |

**Cost (ring):**

$$
T_{\text{reduce-scatter}} = (W-1) \cdot \alpha + \frac{W-1}{W} \cdot n \cdot \beta
$$

**FSDP role:** Issued after backward computation. Each rank receives the gradient shard corresponding to its parameter shard, enabling direct local optimizer update.

### 3.4 All-Gather

**Semantics:** Each GPU holds a distinct shard. All-gather concatenates all shards and places the complete tensor on every GPU.

**Before (4 GPUs):**

| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A$ | $B$ | $C$ | $D$ |

**After:**

| | Shard 0 | Shard 1 | Shard 2 | Shard 3 |
|---|---|---|---|---|
| GPU 0 | $A$ | $B$ | $C$ | $D$ |
| GPU 1 | $A$ | $B$ | $C$ | $D$ |
| GPU 2 | $A$ | $B$ | $C$ | $D$ |
| GPU 3 | $A$ | $B$ | $C$ | $D$ |

**Cost (ring):**

$$
T_{\text{all-gather}} = (W-1) \cdot \alpha + \frac{W-1}{W} \cdot n \cdot \beta
$$

**FSDP role:** Issued at forward and backward entry to reconstruct full parameters from shards.

### 3.5 FSDP Communication Flow Summary

| Phase | Collective | Purpose | Volume per Rank |
|---|---|---|---|
| Forward entry | ALL_GATHER | Reconstruct full params from shards | $P \cdot d_\theta \cdot (W{-}1)/W$ |
| Forward exit | FREE | Release full params, retain shard | — |
| Backward entry | ALL_GATHER | Re-gather full params for gradient computation | $P \cdot d_\theta \cdot (W{-}1)/W$ |
| Backward exit | REDUCE_SCATTER | Synchronize + shard gradients | $P \cdot d_g \cdot (W{-}1)/W$ |
| Optimizer | LOCAL | Update shard using local grad shard + local optimizer state | — |

---

## 4. SimpleFSDP Compiler Optimizations: Bucketing and Reordering

### 4.1 Compiler-Level FSDP via DTensor

SimpleFSDP represents FSDP semantics declaratively using three PyTorch primitives:

| Primitive | FSDP Semantic |
|---|---|
| **DTensor** with `Shard(0)` placement | Parameter sharding across DP mesh |
| **Parametrization** (`ReplicateComputation`) | All-gather in forward, reduce-scatter in backward (via autograd) |
| **Selective Activation Checkpointing** | Free full params after forward, re-gather in backward |

This enables `torch.compile` to capture the entire communication + computation graph, allowing TorchInductor to perform IR-level optimizations.

### 4.2 Bucketing: Communication Amortization

**Problem:** Per-parameter sharding generates $K$ separate all-gathers and reduce-scatters per layer. Each pays the base latency $\alpha$.

$$
T_{\text{unbucketed}} = K \cdot \alpha + \beta \cdot \sum_{i=1}^{K} n_i
$$

$$
T_{\text{bucketed}} = \alpha + \beta \cdot \sum_{i=1}^{K} n_i
$$

**Savings:**

$$
\Delta T = (K-1) \cdot \alpha
$$

**Bucketing Mechanism (All-Gather):**

```
PSEUDOCODE: All-Gather Bucketing
─────────────────────────────────
function BUCKET_ALL_GATHER(ag_nodes: [AG1, AG2, ..., AGk]):
    total_size ← SUM(shard_size(AGi) for i in 1..k)
    bucket_buffer ← ALLOCATE_CONTIGUOUS(total_size)
    
    // Pack shards into contiguous buffer
    offset ← 0
    for each AGi in ag_nodes:
        COPY(AGi.shard → bucket_buffer[offset : offset + shard_size(AGi)])
        offset += shard_size(AGi)
    
    // Single all-gather on entire bucket
    full_bucket ← ALL_GATHER(bucket_buffer, group=dp_group)    // → AG_{1..k}
    WAIT(full_bucket)                                           // → Wa_{1..k}
    
    // Unpack individual full tensors
    offset ← 0
    for each AGi in ag_nodes:
        AGi.full_tensor ← SLICE(full_bucket, offset, full_size(AGi))
        offset += full_size(AGi)
    
    return ag_nodes
```

**Bucketing Mechanism (Reduce-Scatter):**

```
PSEUDOCODE: Reduce-Scatter Bucketing
─────────────────────────────────────
function BUCKET_REDUCE_SCATTER(rs_nodes: [RS1, RS2, ..., RSk]):
    total_size ← SUM(grad_size(RSi) for i in 1..k)
    bucket_buffer ← ALLOCATE_CONTIGUOUS(total_size)
    
    // Pack gradients into contiguous buffer
    offset ← 0
    for each RSi in rs_nodes:
        COPY(RSi.grad → bucket_buffer[offset : offset + grad_size(RSi)])
        offset += grad_size(RSi)
    
    // Single reduce-scatter on entire bucket
    shard_bucket ← REDUCE_SCATTER(bucket_buffer, group=dp_group, op=AVG)
    WAIT(shard_bucket)
    
    // Extract individual gradient shards
    offset ← 0
    for each RSi in rs_nodes:
        RSi.grad_shard ← SLICE(shard_bucket, offset, shard_size(RSi))
        offset += shard_size(RSi)
    
    return rs_nodes
```

### 4.3 Reordering: Communication-Computation Overlap

**Principle:** All-gather and reduce-scatter are asynchronous—they execute on the NIC/copy engine while SMs perform computation. Reordering prefetches the next step's communication before the current step's computation starts.

**Forward (before reordering, bucketed):**

```
AG12 → Wa12 → C1 → C2 → AG34 → Wa34 → C3 → C4
```

AG34 is fully exposed—no overlap possible.

**Forward (after reordering):**

```
AG12 → AG34 → Wa12 → C1 → C2 → Wa34 → C3 → C4
```

AG34 now overlaps with Wa12's copy-out and computations C1, C2.

**Backward (after reordering):**

```
AG12 → Wa12 → AG34 → C1 → C2 → RS12 → Wa34 → C3 → C4 → Wr12 → RS34 → Wr34
```

- $AG34$ overlaps with $C1$, $C2$
- $RS12$ overlaps with $C3$, $C4$

**GPU execution timeline (backward, after reordering):**

| Stream | Time → |
|---|---|
| Compute | `[Wa12] [C1] [C2] [Wa34] [C3] [C4] [Wr12] [Wr34]` |
| Comm (AG) | `[AG12] [AG34]....................` |
| Comm (RS) | `............[RS12]...........[RS34]` |

### 4.4 Auto-Wrapping: Greedy Bucketing with Dual Constraints

The auto-wrapping algorithm makes per-parameter bucketing decisions subject to:

**Forward pass constraint:**

$$
T_{m+m_i}^{AG} \leq T_c \quad \text{(time)} \quad \wedge \quad M_c + M_{c_i} \leq M_{\max} \quad \text{(memory)}
$$

**Backward pass constraint (includes reduce-scatter from previous step):**

$$
T_m^{RS} + T_{m+m_i}^{AG} \leq T_c \quad \text{(time)} \quad \wedge \quad M_c + M_{c_i} \leq M_{\max} \quad \text{(memory)}
$$

```
PSEUDOCODE: Auto-Wrapping Bucketing Decision
─────────────────────────────────────────────
function SHOULD_BUCKET(phase, T_m_AG, T_m_RS, T_c, M_c,
                       T_m_i_AG, T_c_i, M_c_i, M_max):
    
    T_m_combined ← COMM_COST(n_current_bucket + n_i)
    // Note: T_m_combined = α + β·(n_current + n_i), NOT T_m_AG + T_m_i_AG
    
    if phase == FORWARD:
        timeOK ← (T_m_combined ≤ T_c)
        memOK  ← (M_c + M_c_i ≤ M_max)
    
    else if phase == BACKWARD:
        timeOK ← (T_m_RS + T_m_combined ≤ T_c)
        memOK  ← (M_c + M_c_i ≤ M_max)
    
    return timeOK AND memOK
```

The backward time constraint includes $T_m^{RS}$ because the current computation window must simultaneously overlap both the previous step's reduce-scatter **and** the current step's all-gather prefetch.

---

## 5. SOTAFSDP2 Engine: Component Architecture

### 5.1 Initialization Sequence

The SOTAFSDP2 constructor executes a deterministic initialization pipeline:

```
PSEUDOCODE: SOTAFSDP2 Initialization
─────────────────────────────────────
function __init__(config, device_mesh):
    // Phase 1: Distributed topology discovery
    rank, world_size, local_rank ← QUERY_DISTRIBUTED_STATE()
    device ← torch.device("cuda:{local_rank}")
    torch.cuda.set_device(device)
    
    // Phase 2: Hardware capability probe
    hw_info ← detect_hardware(local_rank)
    is_amd ← hw_info.vendor == AMD
    
    // Phase 3: Infrastructure component instantiation
    if config.use_memory_pool:
        memory_pool ← MemoryPool(device, config.mixed_precision.param_dtype)
    
    stream_manager ← StreamManager(device, is_amd)
    metrics ← MetricsCollector(enable_gpu_timing = NOT is_amd)
    mp_context ← MixedPrecisionContext(config.mixed_precision)
    
    // Phase 4: Feature flag logging (rank 0 only)
    if is_rank_zero:
        LOG("strategy={}, precision={}, triton={}, world_size={}, "
            "ckpt_api={modern|legacy}")
```

**Key design decision: GPU timing disabled on AMD.** CUDA event timing (`enable_timing=True`) has correctness issues on some ROCm versions, producing inflated or zero elapsed times. The implementation gates this based on vendor detection, falling back to wall-clock timing on AMD.

### 5.2 Model Wrapping Pipeline

```
PSEUDOCODE: SOTAFSDP2.wrap_model()
──────────────────────────────────
function wrap_model(model, device_mesh):
    with ENFORCE_DEVICE_AFFINITY():
        
        // Step 1: Activation checkpointing (MUST precede FSDP wrapping)
        if config.activation_checkpointing:
            APPLY_ACTIVATION_CHECKPOINTING(model)
        
        // Step 2: Construct FSDP kwargs from config
        fsdp_kwargs ← {
            sharding_strategy: MAP_TO_TORCH_ENUM(config.sharding_strategy),
            auto_wrap_policy: BUILD_WRAP_POLICY(config.auto_wrap_policy),
            use_orig_params: config.use_orig_params,
            forward_prefetch: config.forward_prefetch,
            mixed_precision: BUILD_MIXED_PRECISION(config.mixed_precision),
            backward_prefetch: MAP_BACKWARD_PREFETCH(config.backward_prefetch),
            cpu_offload: BUILD_CPU_OFFLOAD(config.offload_strategy),
            sync_module_states: config.sync_module_states,
            limit_all_gathers: config.limit_all_gathers,
        }
        
        // Step 3: FSDP wrapping (triggers parameter sharding)
        wrapped_model ← FSDP(model, **fsdp_kwargs)
    
    // Step 4: Compute per-shard memory for pressure monitoring
    param_count ← SUM(p.numel() for p in wrapped_model.parameters())
    memory_per_shard_gb ← (param_count / world_size) * bytes_per_elem / 2^30
    
    // Step 5: Initialize gradient accumulation state
    if config.gradient_accumulation_steps > 1:
        accumulation_counter ← 0
        // Use native FSDP no_sync() — NOT manual GradientAccumulator
        // Manual grad injection is unsafe with sharded parameters
    
    return wrapped_model
```

**Critical ordering constraint:** Activation checkpointing MUST be applied before FSDP wrapping. If applied after, the checkpoint wrapper observes sharded parameters, and the recomputation during backward may not correctly reconstruct the all-gathered full parameters.

### 5.3 Auto-Wrap Policy Construction

The auto-wrap policy uses a two-tier fallback:

```
PSEUDOCODE: Auto-Wrap Policy Construction
──────────────────────────────────────────
function BUILD_WRAP_POLICY(policy_class_names, min_num_params):
    layer_classes ← ∅
    
    for cls_name in policy_class_names:
        cls ← TRY_IMPORT_FROM_KNOWN_PATHS(cls_name)
        if cls ≠ None:
            layer_classes.add(cls)
    
    if layer_classes ≠ ∅:
        // Tier 1: Transformer-aware wrapping (one FSDP unit per layer)
        return transformer_auto_wrap_policy(
            transformer_layer_cls = layer_classes
        )
    else:
        // Tier 2: Size-based wrapping (fallback for unknown architectures)
        return size_based_auto_wrap_policy(
            min_num_params = min_num_params  // default: 100M
        )
```

The default `auto_wrap_policy` list covers:

| Framework | Layer Classes |
|---|---|
| HuggingFace Transformers | `LlamaDecoderLayer`, `Llama3DecoderLayer`, `MistralDecoderLayer`, `Qwen2DecoderLayer`, `GPT2Block`, `GPTNeoXLayer`, `FalconDecoderLayer`, `GemmaDecoderLayer`, `Phi3DecoderLayer` |
| PyTorch | `TransformerEncoderLayer`, `TransformerDecoderLayer` |
| Fairseq | `TransformerSentenceEncoderLayer` |

---

## 6. Device Affinity Enforcement: Root Cause Analysis of cuda:0 Drift

### 6.1 Root Cause: FIX-005

This is the single most critical production bug addressed in v2.1. The failure mode is:

> **Symptom:** Multi-GPU training hangs for exactly 1800 seconds (NCCL default timeout), then crashes with `NCCL error: unhandled cuda error`.
>
> **Root cause chain:**
> 1. `ShardedStateDictConfig(offload_to_cpu=True)` instructs FSDP to move parameter shards to CPU during state dict collection.
> 2. When `dcp.save()` subsequently performs NCCL collectives for metadata coordination, PyTorch's NCCL backend moves CPU tensors back to GPU for the collective.
> 3. **The critical bug:** PyTorch moves these tensors to `cuda:0` (the default CUDA device), not to `cuda:{local_rank}`.
> 4. On rank 3 (local rank 3), the NCCL process group is bound to `cuda:3`, but the tensor is on `cuda:0`.
> 5. NCCL detects the device mismatch and enters an error state.
> 6. All other ranks' NCCL operations hang waiting for rank 3, until the 1800-second timeout expires.

### 6.2 The Fix: `_enforce_device_affinity()`

```
PSEUDOCODE: Device Affinity Enforcement
────────────────────────────────────────
function _enforce_device_affinity():
    // Pre-scope: pin device
    torch.cuda.set_device(self._device)  // cuda:{local_rank}
    
    with torch.cuda.device(self._device):
        // All tensor ops in this scope default to self._device
        // Including internal PyTorch allocations during FSDP state dict ops
        YIELD_TO_CALLER
    
    // Post-scope verification (debug mode only)
    if config.debug_mode:
        for name, param in model.named_parameters():
            if param.is_cuda AND param.device ≠ self._device:
                LOG_ERROR("Device drift: {name} on {param.device}")
                param.data ← param.data.to(self._device)  // force migration
```

### 6.3 Deployment Pattern

The `_enforce_device_affinity()` context manager wraps **every** operation that could trigger internal tensor allocation:

- `wrap_model()` — FSDP wrapping
- `get_state_dict()` — state dict collection (both modern and legacy)
- `load_state_dict()` — state dict loading
- `save_checkpoint()` — full save pipeline
- `load_checkpoint()` — full load pipeline
- `summon_full_params()` — temporary parameter materialization

Additionally, explicit `torch.cuda.set_device(self._device)` calls are inserted after every API call that may internally reset the default device (annotated in the code as `[FIX-005] Re-pin device`).

### 6.4 Post-Operation Device Verification

```
PSEUDOCODE: Post-Save Device Verification
──────────────────────────────────────────
function _post_save_device_check(fsdp):
    torch.cuda.set_device(fsdp._device)
    
    drift_count ← 0
    for name, param in fsdp.model.named_parameters():
        if param.is_cuda AND param.device ≠ fsdp._device:
            drift_count += 1
            if drift_count ≤ 5:
                LOG_WARNING("Post-save drift: {name} on {param.device}")
            param.data ← param.data.to(fsdp._device)
    
    if drift_count > 0:
        LOG_WARNING("Migrated {drift_count} params to {fsdp._device}")
```

This is an $O(P)$ scan (where $P$ is the number of parameters, not elements), acceptable because it runs only once per checkpoint.

---

## 7. Checkpoint System: Modern DCP vs. Legacy FSDP1 Pathways

### 7.1 API Version Detection

SOTAFSDP2 detects the available checkpoint API at import time:

```
PSEUDOCODE: API Version Detection
──────────────────────────────────
try:
    import torch.distributed.checkpoint.state_dict
    → get_state_dict, set_state_dict, StateDictOptions
    _HAS_MODERN_DCP ← True     // PyTorch ≥ 2.3
except ImportError:
    _HAS_MODERN_DCP ← False    // PyTorch < 2.3
```

| API | PyTorch Version | Internal Representation | Device Safety | Performance |
|---|---|---|---|---|
| **Modern DCP** | ≥ 2.3 | DTensor | Correct by design | Single coordinated all-gather |
| **Legacy FSDP1** | < 2.3 | ShardedTensor (deprecated) | Requires manual device pinning | Two separate state_dict_type scopes |

### 7.2 Save Pipeline

The unified save entry point (`FSDPCheckpointManager.save_checkpoint()`) executes a six-phase pipeline:

```
PSEUDOCODE: Checkpoint Save Pipeline
─────────────────────────────────────
function save_checkpoint(fsdp, optimizer, path, epoch, step, extra, sharded):
    t_start ← monotonic_clock()
    
    // ═══ Phase 1: Memory Cleanup ═══
    // [FIX-007] Create VRAM headroom before state dict construction
    fsdp.memory_pool.clear()
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    // Report: "{allocated:.2f}/{total:.1f} GB ({pct:.1f}% VRAM)"
    
    // ═══ Phase 2: Pre-Save Barrier ═══
    // Synchronize all ranks; fail-fast timeout (5 min, not 30 min)
    BARRIER_WITH_TIMEOUT(fsdp, label="pre-save", timeout=300s)
    
    // ═══ Phase 3: Save (under device affinity) ═══
    with fsdp._enforce_device_affinity():
        if sharded:
            if _HAS_MODERN_DCP:
                result ← _save_sharded_modern(...)
            else:
                result ← _save_sharded_legacy(...)
        else:
            result ← _save_full(...)
    
    if result.is_err():
        return result
    
    // ═══ Phase 4: Post-Save Device Verification ═══
    _post_save_device_check(fsdp)
    
    // ═══ Phase 5: Post-Save Cleanup ═══
    gc.collect()
    torch.cuda.empty_cache()
    
    // ═══ Phase 6: Post-Save Barrier ═══
    BARRIER_WITH_TIMEOUT(fsdp, label="post-save", timeout=300s)
    
    elapsed ← monotonic_clock() - t_start
    LOG("Checkpoint saved in {elapsed:.1f}s: {path}")
    return Ok(None)
```

### 7.3 Modern DCP Sharded Save (PyTorch ≥ 2.3)

```
PSEUDOCODE: Modern DCP Save
────────────────────────────
function _save_sharded_modern(fsdp, optimizer, checkpoint_dir, epoch, step, extra):
    // [FIX-006] Single get_state_dict call for BOTH model + optimizer
    // This performs ONE coordinated all-gather instead of TWO
    options ← StateDictOptions(full_state_dict=False, cpu_offload=True)
    
    model_state, optim_state ← get_state_dict(
        fsdp.model, optimizers=[optimizer], options=options
    )
    
    // [FIX-005] Re-pin device after get_state_dict
    torch.cuda.set_device(fsdp._device)
    
    // Combined state for atomic save
    combined ← {"model": model_state, "optimizer": optim_state}
    
    // DCP save — parallel writes across ranks
    dcp.save(
        state_dict=combined,
        storage_writer=FileSystemWriter(checkpoint_dir, overwrite=True)
    )
    
    // Metadata (rank 0 only — no collective required)
    if fsdp.is_rank_zero:
        torch.save(meta_dict, checkpoint_dir / "meta.pt")
    
    return Ok(None)
```

**Advantages over legacy:**

| Aspect | Modern DCP | Legacy FSDP1 |
|---|---|---|
| API calls | 1 (`get_state_dict`) | 2 (`state_dict_type` × 2 scopes) |
| All-gathers | 1 coordinated | 2 separate |
| Internal representation | DTensor | ShardedTensor (deprecated) |
| Warning noise | 0 lines | Hundreds per save |
| Barriers | 1 (in `dcp.save`) | 4 (2 scopes × 2 entries) |
| Device drift risk | Low (DTensor tracks placement) | High (ShardedTensor staging on cuda:0) |

### 7.4 Legacy FSDP1 Sharded Save (PyTorch < 2.3)

```
PSEUDOCODE: Legacy FSDP1 Save
──────────────────────────────
function _save_sharded_legacy(fsdp, optimizer, checkpoint_dir, epoch, step, extra):
    
    with SUPPRESS_DEPRECATION_WARNINGS():
        
        // ── Model state (first all-gather) ──
        model_cfg ← ShardedStateDictConfig(offload_to_cpu=True)
        with FSDP.state_dict_type(model, SHARDED_STATE_DICT, model_cfg):
            model_state ← {"model": model.state_dict()}
        
        // [FIX-005] Re-pin device after CPU offload
        torch.cuda.set_device(fsdp._device)
        
        dcp.save(model_state, FileSystemWriter(checkpoint_dir/"model", overwrite=True))
        
        // [FIX-007] Free model state before optimizer all-gather
        del model_state
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.set_device(fsdp._device)  // [FIX-005] Re-pin
        
        // ── Optimizer state (second all-gather) ──
        optim_cfg ← ShardedOptimStateDictConfig(offload_to_cpu=True)
        with FSDP.state_dict_type(model, SHARDED_STATE_DICT, optim_cfg):
            optim_state ← FSDP.optim_state_dict(model, optimizer)
        
        torch.cuda.set_device(fsdp._device)  // [FIX-005] Re-pin
        
        dcp.save({"optimizer": optim_state},
                 FileSystemWriter(checkpoint_dir/"optimizer", overwrite=True))
        
        del optim_state
        gc.collect()
        torch.cuda.empty_cache()
    
    torch.cuda.set_device(fsdp._device)  // [FIX-005] Final re-pin
    
    // Metadata (rank 0 only)
    if fsdp.is_rank_zero:
        torch.save(meta_dict, checkpoint_dir / "meta.pt")
    
    return Ok(None)
```

**Key FIX-007 observation:** The legacy path requires two separate `state_dict_type()` scopes because the model and optimizer state dicts use different configuration objects (`ShardedStateDictConfig` vs `ShardedOptimStateDictConfig`). Each scope triggers an all-gather. At 78% VRAM utilization, the second all-gather (for optimizer state) can push memory past 95%, causing OOM-retry loops and NCCL retransmission. The fix inserts explicit `del` + `gc.collect()` + `empty_cache()` between the two phases.

### 7.5 Full State Dict Save (Rank-0 Only, Portable)

```
PSEUDOCODE: Full State Dict Save
─────────────────────────────────
function _save_full(fsdp, optimizer, path, epoch, step, extra):
    torch.cuda.set_device(fsdp._device)
    
    if _HAS_MODERN_DCP:
        options ← StateDictOptions(full_state_dict=True, cpu_offload=True)
        model_state, optim_state ← get_state_dict(model, [optimizer], options)
    else:
        // Legacy: FullStateDictConfig with rank0_only=True
        with FSDP.state_dict_type(model, FULL_STATE_DICT, full_cfg):
            model_state ← model.state_dict()
            optim_state ← FSDP.optim_state_dict(model, optimizer)
    
    torch.cuda.set_device(fsdp._device)  // [FIX-005]
    
    if fsdp.is_rank_zero:
        checkpoint ← {"model": model_state, "optimizer": optim_state,
                       "epoch": epoch, "step": step, "extra": extra}
        
        // [FIX-007] Atomic write: tmp + rename
        tmp_path ← path.with_suffix(".tmp")
        torch.save(checkpoint, tmp_path)
        os.rename(tmp_path, path)
    
    del model_state, optim_state
    gc.collect()
    torch.cuda.empty_cache()
    
    return Ok(None)
```

**Atomic save justification:** If the process is killed during `torch.save()`, the checkpoint file is corrupted. By writing to a `.tmp` file and then atomically renaming, the final checkpoint path either contains the complete previous checkpoint or the complete new checkpoint—never a partial write.

### 7.6 Load Pipeline

The load path mirrors the save path with API version detection:

| Save Format | Modern Load | Legacy Load |
|---|---|---|
| Sharded (modern DCP) | `dcp.load()` → `set_state_dict()` | N/A (format incompatible) |
| Sharded (legacy FSDP1) | N/A (format incompatible) | `dcp.load()` → `load_state_dict()` |
| Full | `set_state_dict(full=True)` | `FSDP.state_dict_type(FULL)` → `load_state_dict()` |

**Critical note on cross-format compatibility:** Modern DCP checkpoints use DTensor metadata; legacy checkpoints use ShardedTensor metadata. These formats are **not interchangeable** without explicit conversion tooling.

### 7.7 Warning Suppression Strategy (FIX-008)

```
PSEUDOCODE: Warning Suppression (Module-Level, One-Time)
────────────────────────────────────────────────────────
function _suppress_known_deprecation_warnings():
    // Each filter targets a SPECIFIC warning message + module + category
    // Policy: ONLY suppress KNOWN HARMLESS warnings
    // Novel/unexpected warnings pass through (fail-open)
    
    FILTER("_get_pg_default_device.*deprecated", UserWarning,
           module="torch.distributed.distributed_c10d")
    
    FILTER("FSDP.state_dict_type().*deprecated", FutureWarning,
           module="torch.distributed.fsdp")
    
    FILTER("DTensor.*deprecating ShardedTensor", FutureWarning,
           module="torch.distributed")
    
    FILTER("existing checkpoint.*overwriting", UserWarning,
           module="torch.distributed.checkpoint")
```

This is called once at module import, not per-operation. The regex specificity ensures that genuinely novel warnings (e.g., a new deprecation in PyTorch 2.6) are not accidentally suppressed.

---

## 8. Memory Management: Pool Architecture and Pressure-Aware Eviction

### 8.1 MemoryPool Design

The `MemoryPool` eliminates `cudaMalloc` jitter during training by pre-allocating tensors and caching released allocations in power-of-2 buckets.

**Bucket sizing algorithm:**

$$
\text{bucket\_size}(n) = \begin{cases}
512 & \text{if } n \leq 512 \\
2^{\lceil \log_2 n \rceil} & \text{if } 512 < n < 256\text{MB} \\
n & \text{if } n \geq 256\text{MB}
\end{cases}
$$

This ensures:
- Small allocations (< 512 bytes) share a single bucket → minimal fragmentation.
- Medium allocations use power-of-2 rounding → bounded internal fragmentation (at most 2×).
- Large allocations bypass bucketing → no wasted memory for multi-MB tensors.

### 8.2 Allocation and Release

```
PSEUDOCODE: MemoryPool.allocate()
─────────────────────────────────
function allocate(shape, dtype):
    elem_size ← element_size(dtype)
    num_elements ← product(shape)
    size_bytes ← num_elements × elem_size
    bucket ← bucket_size(size_bytes)
    
    with lock:
        allocation_count += 1
        CHECK_PRESSURE()  // evict if above 90% VRAM watermark
        
        if bucket in pools AND pools[bucket] not empty:
            slab ← pools[bucket].pop()
            if slab.numel() ≥ num_elements:
                return slab[:num_elements].view(shape)
            else:
                del slab  // size mismatch, discard
    
    // Cache miss: allocate from CUDA
    tensor ← torch.empty(num_elements, dtype=dtype, device=device)
    
    with lock:
        total_allocated += size_bytes
        peak_allocated ← max(peak_allocated, total_allocated)
    
    return tensor.view(shape)
```

### 8.3 Pressure-Aware Eviction

```
PSEUDOCODE: MemoryPool._check_pressure()
─────────────────────────────────────────
function CHECK_PRESSURE():
    allocated ← torch.cuda.memory_allocated(device)
    threshold ← device_total_bytes × 0.90  // VRAM_PRESSURE_WATERMARK
    
    if allocated < threshold:
        return  // no pressure
    
    // Evict largest buckets first (maximize freed bytes per eviction)
    for bucket_size in SORTED(pools.keys(), DESCENDING):
        while pools[bucket_size] not empty:
            tensor ← pools[bucket_size].pop()
            freed ← tensor.numel() × tensor.element_size()
            total_allocated -= freed
            del tensor
            
            if torch.cuda.memory_allocated(device) < threshold:
                return
        
        if pools[bucket_size] is empty:
            del pools[bucket_size]
```

The 90% watermark is chosen based on production experience: at >90% VRAM utilization, PyTorch's caching allocator begins to fragment, and subsequent large allocations (e.g., all-gather buffers) trigger expensive defragmentation or OOM.

---

## 9. Stream Manager: Communication-Computation Overlap Architecture

### 9.1 Stream Assignment

SOTAFSDP2 maintains four dedicated CUDA streams:

| Stream | Priority | Purpose |
|---|---|---|
| `compute` | Default (0) | Forward/backward computation (default stream) |
| `allgather` | High (-1 NVIDIA, 0 AMD) | All-gather collectives |
| `reduce_scatter` | High (-1 NVIDIA, 0 AMD) | Reduce-scatter collectives |
| `transfer` | Normal (0) | CPU↔GPU transfers (offload, checkpoint) |

**AMD priority note (FIX-004):** CUDA stream priorities -1 (high) and 0 (normal) map differently on ROCm/HIP. Setting `-1` on AMD can cause stream scheduling anomalies in some ROCm versions. The implementation defaults to priority 0 for all streams on AMD.

### 9.2 Stream Synchronization Primitives

```
PSEUDOCODE: Stream Synchronization
───────────────────────────────────
function sync_stream_to(src, dst):
    event ← torch.cuda.Event(enable_timing=False, blocking=False)
    event.record(src)
    dst.wait_event(event)
    // dst will not execute past this point until src has reached the event

function sync_allgather_to_compute():
    // All-gather completion must be visible to compute stream
    // before compute can use the all-gathered parameters
    sync_stream_to(allgather_stream, compute_stream)

function sync_compute_to_reduce_scatter():
    // Gradient computation must complete before reduce-scatter begins
    sync_stream_to(compute_stream, reduce_scatter_stream)
```

### 9.3 Overlap Pattern

The intended execution pattern per FSDP unit during backward:

| Time → | Compute Stream | All-Gather Stream | Reduce-Scatter Stream |
|---|---|---|---|
| $t_0$ | | AG(layer $i+1$) | |
| $t_1$ | Backward(layer $i$) | | |
| $t_2$ | | | RS(layer $i$) |
| $t_3$ | Backward(layer $i+1$) | AG(layer $i+2$) | |

This achieves double-buffering: while layer $i$ is computing, layer $i+1$'s parameters are being gathered and layer $i-1$'s gradients are being scattered.

---

## 10. Triton-Accelerated Kernel Suite

### 10.1 Kernel Inventory

SOTAFSDP2 includes four Triton JIT kernels for performance-critical operations:

| Kernel | Purpose | Fallback |
|---|---|---|
| `_fused_allgather_scale_kernel` | All-gather + scale in single pass | `torch.mul` after all-gather |
| `_fused_cast_and_scale_kernel` | Precision cast + scale | `tensor.to(dtype) * scale` |
| `_fused_gradient_accumulate_kernel` | In-place gradient accumulation with scaling | `buffer.add_(grad, alpha=inv_steps)` |
| `_fused_param_shard_kernel` | Extract parameter shard from full tensor | `full_param[offset:offset+shard_size]` |

### 10.2 Gradient Accumulation Kernel

This is the most frequently called kernel (once per parameter per micro-step):

```
PSEUDOCODE: Fused Gradient Accumulation Kernel
───────────────────────────────────────────────
kernel _fused_gradient_accumulate(grad_ptr, accum_ptr, N, inv_steps, BLOCK_SIZE):
    pid ← program_id(0)
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < N
    
    grad ← load(grad_ptr + offsets, mask=mask, other=0.0)
    accum ← load(accum_ptr + offsets, mask=mask, other=0.0)
    
    accum ← accum + grad × inv_steps
    
    store(accum_ptr + offsets, accum, mask=mask)
```

**Performance advantage:** The fused kernel eliminates a separate multiplication + addition, reducing HBM reads from 3 (grad read + accum read + accum write) to 2 (grad read + accum read-modify-write). For a 70B model with 4 micro-steps, this saves approximately $70 \times 10^9 \times 2 \times 3 = 420$ GB of HBM traffic per training step.

### 10.3 Parameter Shard Extraction Kernel

```
PSEUDOCODE: Fused Parameter Shard Kernel
────────────────────────────────────────
kernel _fused_param_shard(full_ptr, shard_ptr, full_numel, shard_size,
                          rank_offset, BLOCK_SIZE):
    pid ← program_id(0)
    offsets ← pid × BLOCK_SIZE + arange(0, BLOCK_SIZE)
    mask ← offsets < shard_size
    
    src_offsets ← rank_offset + offsets
    full_mask ← mask AND (src_offsets < full_numel)
    
    data ← load(full_ptr + src_offsets, mask=full_mask, other=0.0)
    store(shard_ptr + offsets, data, mask=mask)
```

This kernel is autotuned across four configurations:

| Config | BLOCK_SIZE | num_warps | Best For |
|---|---|---|---|
| 1 | 1024 | 4 | Small shards (< 1M elements) |
| 2 | 2048 | 8 | Medium shards |
| 3 | 4096 | 8 | Large shards |
| 4 | 8192 | 16 | Very large shards (> 100M elements) |

### 10.4 Graceful Degradation

If Triton is unavailable (`pip install triton>=2.1.0` not satisfied), all operations fall back to equivalent PyTorch-native implementations. The estimated performance penalty is ~15% for gradient accumulation-heavy workloads.

```
PSEUDOCODE: Triton Availability Check
──────────────────────────────────────
try:
    import triton
    TRITON_AVAILABLE ← True
except ImportError:
    TRITON_AVAILABLE ← False
    LOG_WARNING("Triton unavailable. Fallback to PyTorch-native (~15% slower)")
```

---

## 11. Gradient Accumulation with FSDP no_sync Semantics

### 11.1 The Problem: FIX-003

Without `no_sync()`, FSDP performs reduce-scatter after **every** backward pass, including intermediate micro-steps:

$$
V_{\text{no\_nosync}} = G \times P \times d_g \times \frac{W-1}{W}
$$

where $G$ is the number of gradient accumulation steps. With `no_sync()`:

$$
V_{\text{with\_nosync}} = P \times d_g \times \frac{W-1}{W}
$$

**Communication savings factor:** $G \times$ reduction.

### 11.2 Accumulation-Aware Backward

```
PSEUDOCODE: SOTAFSDP2.backward()
─────────────────────────────────
function backward(loss, retain_graph=False):
    has_accum ← (config.gradient_accumulation_steps > 1)
    
    if has_accum:
        accumulation_counter += 1
        is_sync_step ← (accumulation_counter ≥ config.gradient_accumulation_steps)
        scaled_loss ← loss / config.gradient_accumulation_steps
        
        if is_sync_step:
            // ═══ Sync step: ALLOW reduce-scatter ═══
            // This is the last micro-step in the accumulation cycle.
            // FSDP will perform reduce-scatter to synchronize gradients.
            scaled_loss ← mp_context.scale_loss(scaled_loss)
            with metrics.measure_backward():
                scaled_loss.backward(retain_graph)
            metrics.update_memory_stats()
            return True  // caller should call step()
        
        else:
            // ═══ Non-sync step: SUPPRESS reduce-scatter ═══
            // [FIX-003] no_sync() prevents O(G×W) communication
            with model.no_sync():
                scaled_loss ← mp_context.scale_loss(scaled_loss)
                with metrics.measure_backward():
                    scaled_loss.backward(retain_graph)
            metrics.update_memory_stats()
            return False  // caller should NOT call step()
    
    else:
        // No accumulation: standard backward
        scaled_loss ← mp_context.scale_loss(loss)
        with metrics.measure_backward():
            scaled_loss.backward(retain_graph)
        metrics.update_memory_stats()
        return True
```

### 11.3 Native vs. Manual Gradient Accumulation

v2.1 uses **native autograd accumulation** (PyTorch accumulates gradients into `.grad` across multiple backward calls) instead of the manual `GradientAccumulator`:

> "Use native autograd accumulation across micro-steps with `no_sync()`. Manual `param.grad` injection is unsafe with sharded parameters (e.g., local empty shards under FSDP2)."

The `GradientAccumulator` class remains in the codebase for non-FSDP use cases but is **not used** when FSDP wrapping is active. The reason: FSDP may produce empty or differently-shaped gradient tensors on some ranks due to uneven parameter sharding. Injecting externally-maintained gradient buffers into these parameters can cause shape mismatches or silent gradient corruption.

---

## 12. Mixed Precision: Numerical Robustness and Loss Scaling

### 12.1 Precision Policy Matrix

| Policy | `param_dtype` | `reduce_dtype` | `buffer_dtype` | Loss Scaling |
|---|---|---|---|---|
| `FULL_BF16` | BF16 | BF16 | BF16 | ❌ |
| `FULL_FP16` | FP16 | FP16 | FP16 | ✅ (GradScaler) |
| `PARAM_FP32` | FP32 | BF16 | FP32 | ❌ |
| `PURE_FP32` | FP32 | FP32 | FP32 | ❌ |

### 12.2 FP16 Loss Scaling

FP16 has a narrow dynamic range ($\pm 65504$, minimum subnormal $\approx 5.96 \times 10^{-8}$), requiring dynamic loss scaling to prevent gradient underflow:

$$
\nabla\theta_{\text{scaled}} = s \cdot \nabla\theta
$$

where $s$ is the loss scale. The `GradScaler` adjusts $s$ dynamically:

$$
s^{(t+1)} = \begin{cases}
s^{(t)} \times \text{growth\_factor} & \text{if no overflow for growth\_interval steps} \\
s^{(t)} \times \text{backoff\_factor} & \text{if overflow detected}
\end{cases}
$$

**Configuration:**

| Parameter | Value | Justification |
|---|---|---|
| `init_scale` | $2^{16} = 65536$ | Midpoint of FP16 range |
| `growth_factor` | 2.0 | Aggressive scale-up for stability |
| `backoff_factor` | 0.5 | Conservative scale-down on overflow |
| `growth_interval` | 2000 | Allow warm-up before increasing scale |

### 12.3 BF16 Advantages

BF16 has the same exponent range as FP32 (8 bits → $\pm 3.39 \times 10^{38}$), eliminating the need for loss scaling:

$$
\text{BF16 range} = [2^{-126}, (2 - 2^{-7}) \times 2^{127}] \approx [1.18 \times 10^{-38}, 3.39 \times 10^{38}]
$$

This is why `FULL_BF16` is the default and recommended policy for all modern datacenter GPUs (A100+, MI300X+).

### 12.4 Optimizer Step with Precision Control

```
PSEUDOCODE: Precision-Aware Optimizer Step
──────────────────────────────────────────
function step(optimizer, scheduler):
    // Phase 1: Unscale gradients (FP16 only)
    mp_context.unscale_grads(optimizer)
    // ↑ Divides all .grad by current loss scale
    // ↑ Checks for inf/nan — if found, optimizer.step() is skipped
    
    // Phase 2: Gradient clipping
    if config.gradient_clipping_norm is not None:
        grad_norm ← model.clip_grad_norm_(config.gradient_clipping_norm)
        metrics.current.gradient_norm ← grad_norm.item()
    
    // Phase 3: Optimizer step (may be skipped if overflow detected)
    with metrics.measure_optimizer():
        mp_context.step_optimizer(optimizer)
        // If FP16 + overflow → scaler.step() skips optimizer.step()
        // If BF16/FP32 → optimizer.step() executes normally
    
    // Phase 4: LR scheduler
    if scheduler is not None:
        scheduler.step()
    
    // Phase 5: Zero gradients
    optimizer.zero_grad(set_to_none=True)
    // set_to_none=True saves memory: sets .grad = None instead of filling zeros
    
    // Phase 6: Reset micro-step counter
    if config.gradient_accumulation_steps > 1:
        accumulation_counter ← 0
    
    // Phase 7: Record metrics
    metrics.record_step()
    
    // Phase 8: Proactive GC under memory pressure
    if allocated > total × 0.90:
        gc.collect()
        torch.cuda.empty_cache()
```

---

## 13. Activation Checkpointing: Memory-Compute Tradeoff Engineering

### 13.1 Checkpointing Modes

| Mode | Behavior | Memory Savings | Recompute Overhead |
|---|---|---|---|
| `full` | Every eligible layer checkpointed | Maximum (~60% activation reduction) | ~33% of forward compute |
| `selective` | Every $k$-th layer checkpointed ($k$ = `ac_frequency`) | Proportional to $1/k$ | ~$33/k$% of forward compute |
| `offload` | Activations offloaded to CPU | Maximum (GPU activations ≈ 0) | PCIe transfer latency |

### 13.2 Selective Checkpointing

```
PSEUDOCODE: Selective Activation Checkpointing
───────────────────────────────────────────────
function APPLY_ACTIVATION_CHECKPOINTING(model, mode, frequency):
    layer_counter ← 0
    
    function check_fn(module):
        name ← module.__class__.__name__.lower()
        is_layer ← ("layer" in name) OR ("block" in name)
                    OR ("decoder" in name) OR ("encoder" in name)
        
        if NOT is_layer:
            return False
        
        if mode == "full":
            return True
        
        if mode == "selective":
            idx ← layer_counter
            layer_counter += 1
            return (idx % frequency == 0)
        
        return True
    
    apply_activation_checkpointing(
        model,
        checkpoint_wrapper_fn = checkpoint_wrapper(NO_REENTRANT),
        check_fn = check_fn
    )
```

**Default configuration:** `selective` mode with `ac_frequency=2` (every other layer checkpointed). This provides approximately 50% activation memory savings at approximately 17% recompute overhead.

### 13.3 Memory Impact

For a Transformer with $L$ layers, hidden dimension $h$, sequence length $s$, micro-batch size $b$:

**Without checkpointing:**

$$
M_{\text{act}} \approx L \times 2 \times b \times s \times h \times d_{\text{act}} \text{ bytes}
$$

**With selective checkpointing (frequency $k$):**

$$
M_{\text{act\_ckpt}} \approx \frac{L}{k} \times 2 \times b \times s \times h \times d_{\text{act}} + (k-1) \times 2 \times b \times s \times h \times d_{\text{act}}
$$

The first term is the stored activations for checkpointed layers; the second term is the activations for the $(k-1)$ non-checkpointed layers within each recomputation segment.

For $k=2$:

$$
M_{\text{act\_ckpt}} \approx \frac{L}{2} \times 2 \times b \times s \times h \times d_{\text{act}} + 1 \times 2 \times b \times s \times h \times d_{\text{act}}
$$

This approximately halves the activation memory.

---

## 14. Hardware Detection and Cross-Vendor Portability

### 14.1 Capability Enumeration

```
PSEUDOCODE: Hardware Detection
──────────────────────────────
function detect_hardware(device_id):
    props ← torch.cuda.get_device_properties(device_id)
    
    // Vendor classification by device name pattern matching
    if "NVIDIA" or "A100" or "H100" or "V100" or "RTX" in props.name:
        vendor ← NVIDIA
    else if "AMD" or "MI" or "INSTINCT" in props.name:
        vendor ← AMD
    else if "INTEL" in props.name:
        vendor ← INTEL
    else:
        vendor ← UNKNOWN
    
    cc ← ComputeCapability(props.major, props.minor)
    
    return HardwareInfo(
        vendor, props.name, cc,
        total_memory = props.total_memory,
        num_sms = props.multi_processor_count,
        supports_nvlink = ("A100" or "H100" or "H200" in props.name),
        ...
    )
```

### 14.2 Compute Capability Feature Gating

| SM Version | Feature | Relevant GPUs |
|---|---|---|
| $\geq$ 80 | BF16 tensor cores | A100, H100, H200, B100, B200 |
| $\geq$ 89 | FP8 tensor cores | H100, H200, B100, B200 |
| $\geq$ 90 | TMA (Tensor Memory Accelerator) | H100, H200 |

```
PSEUDOCODE: Feature Queries
────────────────────────────
function supports_bf16():  return sm_version ≥ 80
function supports_fp8():   return sm_version ≥ 89
function supports_tma():   return sm_version ≥ 90
```

### 14.3 AMD-Specific Adaptations (FIX-004)

| Aspect | NVIDIA | AMD |
|---|---|---|
| Stream priorities | -1 (high), 0 (normal) | 0 for all (avoid ROCm scheduling anomalies) |
| GPU timing | CUDA events with `enable_timing=True` | Disabled (correctness issues on some ROCm versions) |
| Collective library | NCCL | RCCL |
| Interconnect | NVLink/NVSwitch | xGMI/Infinity Fabric |
| Triton backend | CUDA | HIP (via ROCm Triton) |

---

## 15. Metrics Collection: Non-Blocking GPU Timing

### 15.1 Design Principle: Zero GPU Stall

> **[FIX-002]** No `torch.cuda.synchronize()` is called in the training hot path. All timing uses asynchronous CUDA events that are queried lazily.

```
PSEUDOCODE: Non-Blocking GPU Timing
────────────────────────────────────
class MetricsCollector:
    pending_events: List[(start_event, end_event, field_name)]
    
    function measure(field_name):
        start ← CUDAEvent(enable_timing=True)
        end ← CUDAEvent(enable_timing=True)
        start.record()
        
        YIELD_TO_CALLER  // execute the timed operation
        
        end.record()
        pending_events.append((start, end, field_name))
        // NOTE: No synchronize() here!
    
    function flush_pending():
        remaining ← []
        for (start, end, field) in pending_events:
            if end.query():  // non-blocking: has GPU reached this event?
                elapsed_ms ← start.elapsed_time(end)  // CPU-side only
                current[field] += elapsed_ms × 1e6  // → nanoseconds
            else:
                remaining.append((start, end, field))
        pending_events ← remaining
    
    function record_step():
        flush_pending()
        history.append(current)
        current ← FSDPMetrics()
```

### 15.2 Metrics Schema

| Metric | Unit | Source |
|---|---|---|
| `allgather_time_ns` | nanoseconds | CUDA event pair around all-gather |
| `reduce_scatter_time_ns` | nanoseconds | CUDA event pair around reduce-scatter |
| `forward_time_ns` | nanoseconds | CUDA event pair around forward pass |
| `backward_time_ns` | nanoseconds | CUDA event pair around backward pass |
| `optimizer_time_ns` | nanoseconds | CUDA event pair around optimizer step |
| `peak_memory_bytes` | bytes | `torch.cuda.max_memory_allocated()` |
| `allocated_memory_bytes` | bytes | `torch.cuda.memory_allocated()` |
| `reserved_memory_bytes` | bytes | `torch.cuda.memory_reserved()` |
| `gradient_norm` | float | `model.clip_grad_norm_()` return value |
| `gradient_overflow_count` | int | Cumulative overflow detections (FP16) |

---

## 16. Auto-Wrapping Algorithm: Formal Specification

Building upon the SimpleFSDP auto-wrapping from Section 4.4, SOTAFSDP2 provides a practical implementation through PyTorch's wrapping policies:

### 16.1 Two-Tier Wrapping Policy

**Tier 1: Transformer-Aware Wrapping**

Each Transformer layer class (e.g., `LlamaDecoderLayer`) becomes one FSDP unit. This aligns bucket boundaries with the natural parameter grouping of Transformer architectures:

$$
\text{FSDP Unit}_i = \{\theta_{\text{QKV}}^{(i)}, \theta_{\text{O}}^{(i)}, \theta_{\text{gate}}^{(i)}, \theta_{\text{up}}^{(i)}, \theta_{\text{down}}^{(i)}, \theta_{\text{norm1}}^{(i)}, \theta_{\text{norm2}}^{(i)}\}
$$

**Tier 2: Size-Based Wrapping (Fallback)**

For unknown architectures, FSDP units are formed by grouping parameters until the cumulative count reaches `min_num_params` (default: 100M):

$$
\text{FSDP Unit}_k = \{\theta_j : j \in [j_{\text{start}}, j_{\text{end}}]\} \quad \text{s.t.} \quad \sum_{j=j_{\text{start}}}^{j_{\text{end}}} |\theta_j| \geq 10^8
$$

### 16.2 Bucket Size Configuration

SOTAFSDP2 uses a configurable `bucket_size_mb` (default: 25 MB) that controls the NCCL communication granularity. This is related to but distinct from SimpleFSDP's IR-level bucketing:

| `bucket_size_mb` | NCCL Behavior | Best For |
|---|---|---|
| 1 | Fine-grained, high latency overhead | Small models, debugging |
| 25 (default) | Balanced latency/overlap | Most training workloads |
| 100 | Coarse-grained, maximum bandwidth utilization | Large models on fast interconnect |

The NCCL bucket size determines when accumulated gradient data is flushed to a collective:

$$
\text{flush when} \quad \sum_{i} |\nabla\theta_i| \times d_g \geq \text{bucket\_size\_bytes}
$$

---

## 17. Production Bug Taxonomy: FIX-001 through FIX-009

### 17.1 Complete Fix Registry

| Fix ID | Category | Root Cause | Impact | Resolution |
|---|---|---|---|---|
| **FIX-001** | VRAM | Unbounded allocation during gradient accumulation | OOM at >70B params | `MemoryPool` with pressure-aware eviction |
| **FIX-002** | Throughput | `torch.cuda.synchronize()` in metrics collection | 5-15% MFU loss | Non-blocking CUDA event timing |
| **FIX-003** | Communication | Reduce-scatter on every micro-step | $G\times$ excess bandwidth | `no_sync()` context for non-sync steps |
| **FIX-004** | Compatibility | AMD ROCm stream priority and timing bugs | Training crash on MI300X | Vendor-specific feature gating |
| **FIX-005** | Device Affinity | `offload_to_cpu` moves tensors to cuda:0 | 30-min hang → NCCL timeout | `_enforce_device_affinity()` |
| **FIX-006** | API | ShardedTensor deprecation warnings + extra allocs | Warning spam, perf overhead | Modern DCP API (`get_state_dict`) |
| **FIX-007** | Checkpoint | Sync checkpoint blocks training 50s+ | 100% training stall | Pre-save cleanup, single DCP call, fail-fast timeout |
| **FIX-008** | UX | Hundreds of deprecation warning lines | Log pollution | Targeted `warnings.filterwarnings` at import |
| **FIX-009** | UX | `FileSystemWriter` overwrite default change | Warning noise | Explicit `overwrite=True` |

### 17.2 Fix Dependency Graph

```
FIX-005 (device affinity)
  ├── required by: FIX-006 (modern DCP save/load)
  ├── required by: FIX-007 (checkpoint pipeline)
  └── required by: all checkpoint operations

FIX-006 (modern DCP API)
  ├── enables: single get_state_dict call (FIX-007 performance)
  └── eliminates: ShardedTensor warnings (FIX-008)

FIX-007 (checkpoint performance)
  ├── depends on: FIX-005 (device safety)
  ├── depends on: FIX-006 (single API call)
  └── depends on: pre-save memory cleanup (MemoryPool.clear)

FIX-003 (no_sync)
  └── independent: pure communication optimization

FIX-004 (AMD)
  └── independent: vendor-specific feature gating
```

---

## 18. End-to-End Training Integration Contract

### 18.1 Trainer ↔ SOTAFSDP2 API Contract

The integration between SOTATrainer and SOTAFSDP2 is defined through ten interface points:

| Interface | Method | Caller | Contract |
|---|---|---|---|
| **INT-001** | `wrap_model(model)` | Trainer (once) | Returns FSDP-wrapped model. Must call before any forward. |
| **INT-002** | `forward_context()` | Trainer (per forward) | Context manager providing autocast + timing. |
| **INT-003** | `backward(loss)` | Trainer (per backward) | Returns `True` when sync step reached. Trainer must NOT call `loss.backward()` directly. |
| **INT-004** | `step(optimizer, scheduler)` | Trainer (when backward returns True) | Unscale → clip → step → zero_grad → reset. Trainer must NOT call `optimizer.step()` directly. |
| **INT-005** | `clip_grad_norm_(max_norm)` | FSDP2 (internal to step) | Distributed gradient clipping via FSDP sharded computation. |
| **INT-006** | `FSDPCheckpointManager.save_checkpoint()` | Trainer (periodic) | Unified save entry point. |
| **INT-007** | `forward_context()` | Trainer (per forward) | Same as INT-002. |
| **INT-008** | `metrics.get_average()` | Trainer (periodic logging) | Non-blocking metric aggregation. |
| **INT-009** | `memory_summary()` | Trainer (per log line) | Human-readable VRAM string. |
| **INT-010** | `backward(loss)` return value | Trainer (per backward) | `True` → call step; `False` → continue accumulation. |

### 18.2 Canonical Training Loop

```
PSEUDOCODE: SOTATrainer Integration
────────────────────────────────────
function train(model, optimizer, scheduler, dataloader, config):
    fsdp ← create_fsdp2(config)
    model ← fsdp.wrap_model(model)                         // INT-001
    
    for epoch in range(num_epochs):
        for micro_step, batch in enumerate(dataloader):
            
            // Forward
            with fsdp.forward_context():                    // INT-002
                output ← model(batch)
                loss ← compute_loss(output, batch.targets)
            
            // Backward (returns True on sync step)
            should_step ← fsdp.backward(loss)               // INT-003, INT-010
            
            if should_step:
                fsdp.step(optimizer, scheduler)              // INT-004
                
                // Logging
                if global_step % log_interval == 0:
                    avg ← fsdp.metrics.get_average(100)     // INT-008
                    mem ← fsdp.memory_summary()             // INT-009
                    LOG(f"step={global_step} {avg} {mem}")
                
                // Checkpointing
                if global_step % save_interval == 0:
                    FSDPCheckpointManager.save_checkpoint(   // INT-006
                        fsdp, optimizer, checkpoint_dir,
                        epoch, global_step
                    )
                
                global_step += 1
```

---

## 19. Performance Engineering: MFU, Step-Time Decomposition

### 19.1 Model FLOPS Utilization

$$
\text{MFU} = \frac{F_{\text{model}}}{T_{\text{step}} \times N_{\text{GPUs}} \times F_{\text{peak}}}
$$

For a dense Transformer:

$$
F_{\text{model}} \approx 6 \times P \times B_{\text{global}} \times s
$$

where $P$ is parameter count, $B_{\text{global}}$ is global batch size, and $s$ is sequence length.

### 19.2 Step-Time Decomposition

$$
T_{\text{step}} = T_{\text{forward}} + T_{\text{backward}} + T_{\text{optimizer}} + T_{\text{comm\_exposed}} + T_{\text{dataloader}} + T_{\text{misc}}
$$

SOTAFSDP2's metrics collector provides direct measurement of the first three terms via CUDA events. The fourth term ($T_{\text{comm\_exposed}}$) is the portion of all-gather/reduce-scatter communication that is NOT overlapped with computation—this is what bucketing and reordering minimize.

### 19.3 Communication Overlap Efficiency

$$
\eta_{\text{overlap}} = 1 - \frac{T_{\text{comm\_exposed}}}{T_{\text{comm\_total}}}
$$

Ideal: $\eta_{\text{overlap}} = 1.0$ (all communication hidden behind compute). Typical achieved values:

| Configuration | $\eta_{\text{overlap}}$ |
|---|---|
| No overlap (sequential) | 0.0 |
| PyTorch FSDP1 (backward_prefetch) | 0.5–0.7 |
| SimpleFSDP (bucketing + reordering) | 0.8–0.95 |
| SOTAFSDP2 (multi-stream + FSDP prefetch) | 0.6–0.85 |

### 19.4 Memory Efficiency Tracking

SOTAFSDP2 provides continuous memory monitoring via `memory_summary()`:

```
VRAM: 62.34/80.0 GB alloc, 71.50 GB rsv, 67.82 GB peak
```

This is logged per training step (or per logging interval) to detect memory leaks, fragmentation growth, and unexpected allocation patterns.

---

## 20. Summary of Engineering Principles

### 20.1 Core Technical Contributions

| Contribution | Mechanism | Impact |
|---|---|---|
| **Device-safe checkpointing** | `_enforce_device_affinity()` + post-save verification | Eliminates 30-min NCCL timeout hangs |
| **Modern DCP integration** | DTensor-based `get_state_dict()` with legacy fallback | 2× faster saves, zero deprecation warnings |
| **Atomic checkpoint saves** | tmp-file-then-rename (full) + DCP transactions (sharded) | Corruption-proof checkpoint pipeline |
| **Triton-fused primitives** | Gradient accumulation, parameter sharding, cast+scale | ~15% throughput improvement |
| **Multi-stream overlap** | Dedicated streams for all-gather, reduce-scatter, transfer | Near-maximum communication hiding |
| **Pressure-aware memory pool** | Power-of-2 bucketed caching with 90% watermark eviction | Eliminates cudaMalloc jitter |
| **Cross-vendor portability** | Vendor detection + conditional feature gating | Runs on NVIDIA A100/H100 and AMD MI300X |
| **Non-blocking metrics** | Async CUDA event timing + lazy flush | Zero GPU stall in hot path |

### 20.2 Fundamental Equations

| Equation | Description |
|---|---|
| $\textbf{All-Reduce} = \textbf{Reduce-Scatter} + \textbf{All-Gather}$ | Communication decomposition enabling FSDP |
| $M_{\text{full\_shard}} = 16P/W$ | Per-rank steady-state memory (BF16 + Adam) |
| $T_m = \alpha + \beta n$ | Collective communication cost model |
| $T_m^{RS} + T_{m+m_i}^{AG} \leq T_c$ | Backward overlap feasibility constraint |
| $M_c + M_{c_i} \leq M_{\max}$ | Memory feasibility constraint for bucketing |
| $V_{\text{full\_shard}} \approx 6P$ bytes/step | Total communication volume (FULL_SHARD, BF16) |
| $\text{MFU} = F_{\text{model}} / (T_{\text{step}} \times N \times F_{\text{peak}})$ | Training efficiency metric |

### 20.3 Result Type Pattern

SOTAFSDP2 uses an algebraic `Result[T] = Ok[T] | Err[T]` type for all fallible operations, replacing exception-based control flow with explicit error handling:

```
PSEUDOCODE: Result Type Usage
─────────────────────────────
result ← save_checkpoint(fsdp, optimizer, path)

if result.is_ok():
    LOG("Checkpoint saved successfully")
else:
    LOG_ERROR(f"Save failed: {result.error} (code={result.code})")
    INITIATE_RETRY_POLICY()
```

This pattern eliminates silent failures (every error is explicitly handled by the caller) and enables composition:

$$
\text{Ok}(v).\text{map}(f) = \text{Ok}(f(v)) \qquad \text{Err}(e).\text{map}(f) = \text{Err}(e)
$$

### 20.4 Decision Framework

| If you need... | Use... |
|---|---|
| Maximum memory efficiency at scale | `FULL_SHARD` + `activation_checkpointing=True` |
| Maximum throughput (model fits per-GPU) | `SHARD_GRAD_OP` or `NO_SHARD` |
| Heterogeneous interconnect | `HYBRID_SHARD` (NVLink intra, IB inter) |
| BF16 training (A100/H100/MI300X) | `FULL_BF16` (no loss scaling needed) |
| FP16 training (legacy or mixed) | `FULL_FP16` (automatic GradScaler) |
| Fast checkpointing (PyTorch ≥ 2.3) | Modern DCP (single `get_state_dict` call) |
| Checkpoint portability across world sizes | Full state dict save (`sharded=False`) |
| AMD MI300X deployment | Enable vendor detection; accept disabled GPU timing |
| Gradient accumulation with FSDP | Native `no_sync()` (NOT manual GradientAccumulator) |

---

> **Production maxim (SOTAFSDP2 v2.1):** The three most dangerous operations in distributed training are checkpoint save, checkpoint load, and world-size change—because all three involve NCCL collectives with tensors that may have been silently moved to the wrong device. Every such operation must be wrapped in device affinity enforcement, bounded by a fail-fast timeout, and followed by device verification. The cost of this paranoia is $O(P)$ per checkpoint (parameter scan); the cost of its absence is a 30-minute training stall per incident.

---

*This report covers the complete SOTAFSDP2 v2.1 architecture including sharding strategies, collective communication primitives, SimpleFSDP compiler optimizations, device affinity enforcement (FIX-005), modern DCP checkpoint migration (FIX-006), checkpoint performance optimization (FIX-007), Triton kernel suite, multi-stream communication overlap, pressure-aware memory management, gradient accumulation with no_sync semantics, mixed precision with loss scaling, activation checkpointing, cross-vendor hardware detection, non-blocking metrics collection, and the complete production bug taxonomy (FIX-001 through FIX-009). Every algorithm is specified in pseudocode, every constraint is formalized mathematically, and every design decision is justified from first principles of distributed systems engineering.*



# SimpleFSDP: Compiler-Integrated Fully Sharded Data Parallelism with Communication-Computation Overlap Optimization

## A Comprehensive Technical Report on FSDP Workflow, Collective Decomposition, and TorchInductor-Level Bucketing and Reordering for Production-Scale Distributed Training

---

## Table of Contents

1. [FSDP / ZeRO-3 Sharding: Foundational Workflow](#1-fsdp--zero-3-sharding-foundational-workflow)
2. [Collective Communication Decomposition](#2-collective-communication-decomposition)
3. [SimpleFSDP Architecture and Design Principles](#3-simplefsdp-architecture-and-design-principles)
4. [TorchInductor-Level Optimizations: Bucketing and Reordering](#4-torchinductor-level-optimizations-bucketing-and-reordering)
5. [Model Wrapping Strategies: Manual and Automatic](#5-model-wrapping-strategies-manual-and-automatic)
6. [Auto-Wrapping Algorithm: Formal Specification](#6-auto-wrapping-algorithm-formal-specification)
7. [Composability with Multi-Dimensional Parallelism](#7-composability-with-multi-dimensional-parallelism)
8. [Memory Analysis and Communication Cost Models](#8-memory-analysis-and-communication-cost-models)
9. [Production Deployment Considerations](#9-production-deployment-considerations)
10. [Summary of Engineering Impact](#10-summary-of-engineering-impact)

---

## 1. FSDP / ZeRO-3 Sharding: Foundational Workflow

### 1.1 Core Sharding Semantics

Fully Sharded Data Parallelism (FSDP), equivalent in sharding semantics to ZeRO Stage-3, partitions **all three categories** of GPU-resident state across the data-parallel world:

| State Category | Per-Parameter Memory (FP32 Adam) | Sharded Across $N$ Ranks |
|---|---|---|
| Model Parameters ($\Phi$) | $4\Phi$ bytes (FP32) or $2\Phi$ bytes (BF16) | $\frac{\|\Phi\|}{N}$ per rank |
| Gradients ($\nabla\Phi$) | $4\Phi$ bytes (FP32) or $2\Phi$ bytes (BF16) | $\frac{\|\nabla\Phi\|}{N}$ per rank |
| Optimizer States ($\mathcal{O}$) | $8\Phi$ bytes (Adam: $m_t$, $v_t$ in FP32) | $\frac{\|\mathcal{O}\|}{N}$ per rank |

The total per-rank memory for parameter-related state under ZeRO-3/FSDP is:

$$
M_{\text{per-rank}}^{\text{FSDP}} = \frac{2\Phi + 2\Phi + (4\Phi + 4\Phi)}{N} + 2\Phi_{\text{temp-allgather}} = \frac{12\Phi}{N} + 2\Phi_{\text{temp}}
$$

where the $2\Phi_{\text{temp}}$ term reflects the transient full-parameter buffer materialized during all-gather for the currently active FSDP unit. This temporary buffer is freed immediately after computation, which is fundamental to the memory efficiency of the approach.

### 1.2 FSDP Instance Lifecycle: Forward and Backward Pass

Each FSDP unit (wrapping $N$ layers or a logical module group) follows a strict four-phase protocol in the forward pass and a five-phase protocol in the backward pass. The following pseudocode precisely captures this lifecycle:

**Pseudocode 1: FSDP Forward Pass Per-Instance**

```
PROCEDURE FSDP_FORWARD(instance, input):
    ──────────────────────────────────────────────
    Phase 1: LOAD-MODEL-SHARD
        IF cpu_offload_enabled THEN
            Transfer sharded parameters from CPU → GPU (async via pinned memory + D2H stream)
        END IF
        shard ← instance.local_parameter_shard

    Phase 2: ALL-GATHER
        full_params ← ALL_GATHER(shard, process_group)
        // Materializes the complete parameter tensor from all N ranks
        // Communication volume per rank: (N-1)/N × |shard| × dtype_size

    Phase 3: FORWARD (LOCAL)
        output ← instance.module.forward(input, full_params)
        // Standard module forward computation using full (unsharded) parameters
        // Activations are retained for backward (unless activation checkpointing is applied)

    Phase 4: FREE FULL WEIGHTS
        Deallocate full_params buffer
        // Only the local shard persists; transient memory is reclaimed immediately
        // This is the critical memory optimization enabling ZeRO-3 scaling

    RETURN output
    ──────────────────────────────────────────────
```

**Pseudocode 2: FSDP Backward Pass Per-Instance**

```
PROCEDURE FSDP_BACKWARD(instance, grad_output):
    ──────────────────────────────────────────────
    Phase 1: ALL-GATHER (re-materialization)
        full_params ← ALL_GATHER(instance.local_parameter_shard, process_group)
        // Parameters must be re-gathered because they were freed after forward
        // This is a second all-gather per FSDP unit per training step

    Phase 2: BACKWARD (LOCAL)
        grad_input, grad_params ← BACKWARD(instance.module, grad_output, full_params)
        // Compute local gradients w.r.t. both input and full parameters

    Phase 3: REDUCE-SCATTER
        grad_shard ← REDUCE_SCATTER(grad_params, op=AVG, process_group)
        // Each rank receives the reduced gradient corresponding to its parameter shard
        // Communication volume per rank: (N-1)/N × |grad_params| × dtype_size
        // Reduction operator: average (or sum, depending on configuration)

    Phase 4: FREE FULL WEIGHTS
        Deallocate full_params buffer
        // Memory reclaimed; only sharded gradient persists

    Phase 5: OFFLOAD GRADIENTS (conditional)
        IF cpu_offload_enabled THEN
            Asynchronously transfer grad_shard GPU → CPU
            // Enables CPU-based optimizer step, freeing GPU memory for next FSDP unit
        END IF

    RETURN grad_input
    ──────────────────────────────────────────────
```

**Pseudocode 3: FSDP Optimizer Step**

```
PROCEDURE FSDP_OPTIMIZER_STEP(instance):
    ──────────────────────────────────────────────
    // Each rank updates ONLY its local parameter shard
    // using its local gradient shard and local optimizer states

    IF cpu_offload_enabled THEN
        Execute optimizer step on CPU using offloaded gradients and CPU-resident optimizer states
        Transfer updated parameter shard CPU → GPU
    ELSE
        UPDATE_WEIGHTS_LOCAL(instance.local_parameter_shard,
                             instance.local_gradient_shard,
                             instance.local_optimizer_states)
    END IF
    ──────────────────────────────────────────────
```

### 1.3 Data-Flow Topology Across FSDP Instances

The complete training step orchestrates multiple FSDP instances sequentially. The data flow can be summarized as:

- **Shared input data** is distributed identically to both ranks (top and bottom rows in the workflow diagram), representing two data-parallel replicas.
- **Vertical synchronization arrows** between replicas carry:
  - **Upward / Downward: `GATHER WEIGHTS (ALL_GATHER)`** — parameter re-assembly before computation.
  - **Upward / Downward: `SYNC GRADS (REDUCE_SCATTER)`** — gradient reduction after backward.
- Each FSDP instance processes its assigned layers independently, with the all-gather and reduce-scatter collectives being the **only** cross-rank communication points for parameters and gradients, respectively.

> **Key Engineering Insight:** The memory advantage of FSDP/ZeRO-3 comes at the cost of **two all-gathers per FSDP unit per step** (one in forward, one in backward) plus **one reduce-scatter per unit**. The total communication volume per training step for parameters alone is:
>
> $$V_{\text{total}} = 2 \times \frac{(N-1)}{N} \times |\Phi| \times d_{\text{type}} + \frac{(N-1)}{N} \times |\Phi| \times d_{\text{type}} = 3 \times \frac{(N-1)}{N} \times |\Phi| \times d_{\text{type}}$$
>
> where $N$ is the data-parallel world size and $d_{\text{type}}$ is the per-element byte width. This is $1.5\times$ the communication volume of vanilla all-reduce-based DDP. Overlapping these collectives with computation is therefore **critical** to preserving training throughput at scale.

### 1.4 CPU Offload Path

When CPU offload is enabled, the lifecycle extends to include host-device transfers:

| Stage | Direction | Content | Overlap Strategy |
|---|---|---|---|
| Pre-Forward | CPU → GPU | Sharded parameters (pinned memory, async copy) | Overlap with previous FSDP unit's computation |
| Post-Backward | GPU → CPU | Sharded gradients | Overlap with next FSDP unit's backward all-gather |
| Optimizer Step | CPU compute | Adam/AdamW on CPU-resident states | Overlap with next forward step's prefetch |
| Post-Optimizer | CPU → GPU | Updated sharded parameters | Overlap with next step's data loading |

The memory benefit is significant: optimizer states ($8\Phi/N$ bytes per rank) reside on host DRAM, freeing HBM for activations and temporary all-gather buffers. However, the PCIe Gen4/Gen5 bandwidth ($32$–$64$ GB/s bidirectional) becomes a bottleneck if transfers are not meticulously overlapped.

---

## 2. Collective Communication Decomposition

### 2.1 Fundamental Identity

The foundational algebraic identity that underpins FSDP's communication strategy is:

$$
\texttt{All-Reduce} \equiv \texttt{Reduce-Scatter} + \texttt{All-Gather}
$$

This decomposition is not merely theoretical—it is the **operational basis** for why FSDP replaces DDP's all-reduce with separate reduce-scatter and all-gather calls, enabling sharded parameter and gradient storage.

### 2.2 All-Reduce

Given $N$ GPUs, each holding a tensor (e.g., $A$, $B$, $C$, $D$ for $N=4$), all-reduce computes the element-wise reduction and replicates the result to every rank:

**Input state:**
| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A$ | $B$ | $C$ | $D$ |

**Output state (after all-reduce with SUM):**
| GPU 0 | GPU 1 | GPU 2 | GPU 3 |
|---|---|---|---|
| $A+B+C+D$ | $A+B+C+D$ | $A+B+C+D$ | $A+B+C+D$ |

Communication volume per rank for a ring-based implementation:

$$
V_{\text{all-reduce}}^{\text{ring}} = 2 \times \frac{(N-1)}{N} \times S
$$

where $S$ is the tensor size in bytes. The factor of $2$ reflects one reduce-scatter pass and one all-gather pass in the ring algorithm.

### 2.3 Reduce-Scatter

Each GPU's tensor is logically split into $N$ chunks. The reduce-scatter reduces each chunk across all GPUs and scatters the result such that GPU $i$ holds only chunk $i$:

**Input state (4 GPUs, tensor split into 4 chunks):**
| | Chunk 0 | Chunk 1 | Chunk 2 | Chunk 3 |
|---|---|---|---|---|
| GPU 0 | $A_0$ | $A_1$ | $A_2$ | $A_3$ |
| GPU 1 | $B_0$ | $B_1$ | $B_2$ | $B_3$ |
| GPU 2 | $C_0$ | $C_1$ | $C_2$ | $C_3$ |
| GPU 3 | $D_0$ | $D_1$ | $D_2$ | $D_3$ |

**Output state:**
| | Resident Chunk |
|---|---|
| GPU 0 | $A_0 + B_0 + C_0 + D_0$ |
| GPU 1 | $A_1 + B_1 + C_1 + D_1$ |
| GPU 2 | $A_2 + B_2 + C_2 + D_2$ |
| GPU 3 | $A_3 + B_3 + C_3 + D_3$ |

Communication volume per rank:

$$
V_{\text{reduce-scatter}}^{\text{ring}} = \frac{(N-1)}{N} \times S
$$

> **FSDP Usage:** After backward computation produces full-sized gradients, reduce-scatter both **reduces** (averages) gradients across ranks and **shards** the result, so each rank stores only $1/N$ of the gradient tensor. This is the operation labeled `REDUCE-SCATTER` in the FSDP backward lifecycle.

### 2.4 All-Gather

Each GPU holds one shard; after all-gather, every GPU holds the concatenation of all shards:

**Input state:**
| | Resident Shard |
|---|---|
| GPU 0 | $A$ |
| GPU 1 | $B$ |
| GPU 2 | $C$ |
| GPU 3 | $D$ |

**Output state:**
| | Full Tensor |
|---|---|
| GPU 0 | $[A, B, C, D]$ |
| GPU 1 | $[A, B, C, D]$ |
| GPU 2 | $[A, B, C, D]$ |
| GPU 3 | $[A, B, C, D]$ |

Communication volume per rank:

$$
V_{\text{all-gather}}^{\text{ring}} = \frac{(N-1)}{N} \times S
$$

> **FSDP Usage:** Before forward (and again before backward) computation, each FSDP unit calls all-gather to reconstruct the full parameter tensor from the per-rank shards. This is the operation labeled `ALL-GATHER` in the FSDP forward and backward lifecycle.

### 2.5 Communication Cost Model

For any collective on a message of size $S$ bytes across $N$ ranks:

$$
T_{\text{collective}} = \alpha + \beta \cdot n
$$

where:
- $\alpha$ = base latency (connection establishment, kernel launch, synchronization overhead) — typically $5$–$15\;\mu\text{s}$ intra-node (NVLink/NVSwitch), $10$–$50\;\mu\text{s}$ inter-node (InfiniBand/RoCE)
- $\beta$ = inverse bandwidth coefficient (seconds per byte)
- $n$ = transmitted word size (effective bytes moved per rank)

For a ring all-gather of a tensor with $S$ total bytes:

$$
T_{\text{all-gather}}^{\text{ring}} = (N-1) \cdot \alpha_{\text{step}} + \frac{(N-1)}{N} \cdot S \cdot \beta
$$

The critical implication is that **frequent small collectives** incur repeated $\alpha$ overhead, whereas **fewer large collectives** amortize the base latency. This motivates **bucketing** as discussed in Section 4.

---

## 3. SimpleFSDP Architecture and Design Principles

### 3.1 Motivation and Design Philosophy

Conventional FSDP implementations (e.g., PyTorch FSDP1, FSDP2) operate in **eager mode**, issuing all-gather and reduce-scatter collectives at runtime boundaries between FSDP-wrapped modules. This approach has three fundamental limitations:

1. **Communication exposure:** Per-parameter or per-module collectives serialize communication and computation, leaving communication fully exposed on the critical path.
2. **Granularity mismatch:** The wrapping granularity chosen by the user (per-layer, per-block) directly determines communication frequency, creating a tight coupling between model definition and distributed performance.
3. **Compiler opacity:** Eager-mode collectives are invisible to downstream optimizing compilers (e.g., TorchInductor), preventing global scheduling optimizations across communication and computation.

SimpleFSDP addresses all three limitations by implementing FSDP semantics through **three composable PyTorch primitives**:

| Primitive | Role in SimpleFSDP | Mechanism |
|---|---|---|
| **DTensor** | Parameter sharding representation | Parameters initialized as `DTensor` with `Shard` placement on the DP mesh dimension |
| **Parametrization** | All-gather in forward pass | `ReplicateComputation` parametrization takes sharded input, calls `redistribute(Replicate())` to trigger all-gather |
| **Selective Activation Checkpointing** | Memory management (free/re-gather) | All-gathered (replicated) parameters are treated as "activations" that can be checkpointed—released after forward and recomputed (re-gathered) before backward |

### 3.2 ReplicateComputation Parametrization

The core primitive is the `ReplicateComputation` parametrization, which transforms sharded parameters into replicated parameters for computation:

**Pseudocode 4: ReplicateComputation**

```
CLASS ReplicateComputation:
    ATTRIBUTES:
        param_dtype     // Forward computation precision (e.g., BF16)
        reduce_dtype    // Backward gradient precision (e.g., FP32)

    PROCEDURE replicate_compute(self, x):
        ──────────────────────────────────────────────
        // x: sharded DTensor with placement Shard(0) on DP mesh
        //
        // Step 1: Redistribute from Shard → Replicate (triggers all-gather)
        //         Cast forward dtype to param_dtype
        //         Register backward hook: gradient uses reduce_dtype
        //         Backward placement: Partial(avg) → triggers reduce-scatter
        
        replicated ← x.redistribute(
            placements = (Replicate(),),
            forward_dtype = self.param_dtype,
            backward_dtype = self.reduce_dtype
        )
        
        // Step 2: Extract local tensor for module computation
        //         Gradient placement Partial(avg) ensures reduce-scatter on backward
        
        local_tensor ← replicated.to_local(
            grad_placements = (Partial(reduce_op="avg"),)
        )
        
        RETURN local_tensor
        ──────────────────────────────────────────────
```

> **Critical Design Property:** Since DTensor's `redistribute` is differentiable and `Partial` placement on the gradient triggers reduce-scatter automatically in the backward pass, the **entire gradient synchronization protocol is defined declaratively** through the tensor placement specification, not through explicit collective calls. This makes the communication pattern **fully visible** to `torch.compile` and, consequently, to TorchInductor's scheduling passes.

### 3.3 Activation Checkpointing as Memory Semantics for FSDP

The memory optimization in FSDP—freeing full parameters after forward computation and re-gathering them before backward—maps precisely to the semantics of selective activation checkpointing:

$$
\underbrace{\text{All-gather in forward}}_{\text{Compute "activation"}} \xrightarrow{\text{Free after forward}} \underbrace{\text{All-gather in backward}}_{\text{Recompute "activation"}}
$$

By applying selective activation checkpointing **only to the FSDP communication operators** (the `redistribute` call), SimpleFSDP achieves the standard FSDP memory behavior: full parameters are transient during computation and automatically re-materialized during backward.

**Pseudocode 5: Selective Checkpointing Application**

```
PROCEDURE apply_fsdp_checkpointing(model):
    ──────────────────────────────────────────────
    // Define checkpoint policy: only checkpoint redistribute (all-gather) operations
    FUNCTION fsdp_checkpoint_policy(op):
        RETURN op IS redistribute_op

    FOR each module IN model.fsdp_wrapped_modules:
        APPLY selective_activation_checkpoint(
            module,
            policy = fsdp_checkpoint_policy
        )
    END FOR
    ──────────────────────────────────────────────
```

### 3.4 Compilation and Graph Capture

The complete SimpleFSDP workflow produces a **full computation graph** containing both computation and communication nodes:

**Pseudocode 6: SimpleFSDP End-to-End Initialization**

```
PROCEDURE initialize_simplefsdp(model, compile_config):
    ──────────────────────────────────────────────
    // Step 1: Configure TorchInductor optimizations
    SET inductor.config.simplefsdp.bucket_mode ← "auto"  // or "manual"
    SET inductor.config.simplefsdp.enable_reorder ← True

    // Step 2: Wrap model with SimpleFSDP
    //   - Shards all parameters as DTensors on DP mesh
    //   - Registers ReplicateComputation parametrization on each parameter
    //   - Applies selective activation checkpointing for FSDP memory semantics
    wrapped_model ← simple_fsdp(model)

    // Step 3: Compile with torch.compile
    //   - Traces full graph including redistribute (all-gather) and
    //     Partial backward (reduce-scatter) operations
    //   - TorchInductor receives IR nodes for both communication and computation
    //   - Bucketing and reordering passes execute in the compiler backend
    compiled_model ← torch.compile(wrapped_model, fullgraph=True)

    // Note: If model contains untraceable operations (data-dependent control flow),
    //       set fullgraph=False to allow graph-break-based subgraph optimization
    
    RETURN compiled_model
    ──────────────────────────────────────────────
```

> **Engineering Benefit 1 — Simplicity:** The user-facing API is exactly two calls: `simple_fsdp(model)` followed by `torch.compile(model)`. No manual wrapping, no explicit collective insertion, no communication scheduling code.
>
> **Engineering Benefit 2 — Debuggability:** The eager-mode execution of the parametrized model (before compilation) is **semantically identical** to the compiled version. Engineers can debug model logic, loss computation, and gradient flow in eager mode and then compile for production performance.

---

## 4. TorchInductor-Level Optimizations: Bucketing and Reordering

### 4.1 The Exposure Problem

When SimpleFSDP traces the model graph, the initial IR in TorchInductor contains **per-parameter** communication nodes in sequential order:

**Unoptimized Forward IR Sequence (4 parameters, 2 modules):**

$$
\texttt{AG}_1 \to \texttt{Wa}_1 \to \texttt{C}_1 \;\mid\; \texttt{AG}_2 \to \texttt{Wa}_2 \to \texttt{C}_2 \;\mid\; \texttt{AG}_3 \to \texttt{Wa}_3 \to \texttt{C}_3 \;\mid\; \texttt{AG}_4 \to \texttt{Wa}_4 \to \texttt{C}_4
$$

**Unoptimized Backward IR Sequence:**

$$
\texttt{AG}_1 \to \texttt{Wa}_1 \to \texttt{C}_1 \to \texttt{RS}_1 \to \texttt{Wr}_1 \;\mid\; \texttt{AG}_2 \to \texttt{Wa}_2 \to \texttt{C}_2 \to \texttt{RS}_2 \to \texttt{Wr}_2 \;\mid\; \cdots
$$

where:
- $\texttt{AG}_i$ = all-gather for parameter $i$ (asynchronous launch)
- $\texttt{Wa}_i$ = all-gather wait for parameter $i$ (synchronization point)
- $\texttt{C}_i$ = computation using parameter $i$
- $\texttt{RS}_i$ = reduce-scatter for gradient $i$ (asynchronous launch)
- $\texttt{Wr}_i$ = reduce-scatter wait for gradient $i$ (synchronization point)

In this unoptimized schedule, **every communication operation is fully exposed**—the GPU compute stream idles during each all-gather and reduce-scatter because no computation is overlapped.

### 4.2 Optimization 1: Bucketing

#### 4.2.1 Principle

Bucketing merges multiple small communication IR nodes into a single larger collective, **amortizing the base latency $\alpha$** across all bucketed parameters:

$$
T_{\text{unbucketed}} = k \cdot \alpha + \beta \sum_{i=1}^{k} n_i \quad\text{vs.}\quad T_{\text{bucketed}} = \alpha + \beta \sum_{i=1}^{k} n_i
$$

For $k$ individual collectives, bucketing saves $(k-1) \cdot \alpha$ in base latency overhead. With typical $\alpha = 10\;\mu\text{s}$ and $k = 32$ parameters per module, the savings are $310\;\mu\text{s}$ per FSDP unit—significant when step times are on the order of hundreds of milliseconds.

#### 4.2.2 All-Gather Bucketing Mechanism

**Pseudocode 7: All-Gather Bucketing**

```
PROCEDURE BUCKET_ALL_GATHER(ag_nodes = [AG₁, AG₂, ..., AGₖ]):
    ──────────────────────────────────────────────
    // Step 1: Allocate a contiguous buffer for all parameters
    total_size ← SUM(size(AGᵢ.tensor) for i in 1..k)
    bucket_buffer ← ALLOCATE(total_size)
    
    // Step 2: Flatten and concatenate each parameter's shard into the bucket
    offset ← 0
    FOR i IN 1..k:
        COPY AGᵢ.local_shard INTO bucket_buffer[offset : offset + size(AGᵢ.local_shard)]
        offset ← offset + size(AGᵢ.local_shard)
    END FOR
    
    // Step 3: Issue single all-gather on the bucket
    full_bucket ← ALL_GATHER_ASYNC(bucket_buffer, process_group)
    // This replaces k individual all-gathers with one
    
    // Step 4: Create new IR nodes
    AG₁₂...ₖ ← new AllGatherNode(bucket_buffer)
    Wa₁₂...ₖ ← new AllGatherWaitNode(full_bucket)
    
    // Step 5: After wait, copy out each parameter from the gathered buffer
    offset ← 0
    FOR i IN 1..k:
        paramᵢ_full ← SLICE(full_bucket, offset, offset + full_size(AGᵢ))
        offset ← offset + full_size(AGᵢ)
    END FOR
    
    RETURN AG₁₂...ₖ, Wa₁₂...ₖ, [param₁_full, ..., paramₖ_full]
    ──────────────────────────────────────────────
```

#### 4.2.3 Reduce-Scatter Bucketing Mechanism

**Pseudocode 8: Reduce-Scatter Bucketing**

```
PROCEDURE BUCKET_REDUCE_SCATTER(rs_nodes = [RS₁, RS₂, ..., RSₖ]):
    ──────────────────────────────────────────────
    // Step 1: For each gradient tensor, split into N chunks (N = world_size)
    // Step 2: Concatenate corresponding chunks across all rs_nodes
    
    total_grad_size ← SUM(size(RSᵢ.gradient) for i in 1..k)
    bucket_buffer ← ALLOCATE(total_grad_size)
    
    // Flatten all gradients into contiguous buffer
    offset ← 0
    FOR i IN 1..k:
        COPY RSᵢ.gradient INTO bucket_buffer[offset : offset + size(RSᵢ.gradient)]
        offset ← offset + size(RSᵢ.gradient)
    END FOR
    
    // Step 3: Issue single reduce-scatter on the bucket
    reduced_shard ← REDUCE_SCATTER_ASYNC(bucket_buffer, op=AVG, process_group)
    
    // Step 4: Create new IR nodes
    RS₁₂...ₖ ← new ReduceScatterNode(bucket_buffer)
    Wr₁₂...ₖ ← new ReduceScatterWaitNode(reduced_shard)
    
    // Step 5: After wait, read out each gradient shard for local weight update
    offset ← 0
    FOR i IN 1..k:
        grad_shardᵢ ← SLICE(reduced_shard, offset, offset + shard_size(RSᵢ))
        offset ← offset + shard_size(RSᵢ)
    END FOR
    
    RETURN RS₁₂...ₖ, Wr₁₂...ₖ, [grad_shard₁, ..., grad_shardₖ]
    ──────────────────────────────────────────────
```

#### 4.2.4 Bucketed IR Transformation Example (Manual Wrapping)

**Before bucketing (Forward):**

$$
\texttt{AG}_1 \;\texttt{Wa}_1 \;\texttt{C}_1 \;\texttt{AG}_2 \;\texttt{Wa}_2 \;\texttt{C}_2 \;\mid\; \texttt{AG}_3 \;\texttt{Wa}_3 \;\texttt{C}_3 \;\texttt{AG}_4 \;\texttt{Wa}_4 \;\texttt{C}_4
$$

**After bucketing (Forward):**

$$
\texttt{AG}_{12} \;\texttt{Wa}_{12} \;\texttt{C}_1 \;\texttt{C}_2 \;\mid\; \texttt{AG}_{34} \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4
$$

**Before bucketing (Backward):**

$$
\texttt{AG}_1 \;\texttt{Wa}_1 \;\texttt{C}_1 \;\texttt{RS}_1 \;\texttt{Wr}_1 \;\texttt{AG}_2 \;\texttt{Wa}_2 \;\texttt{C}_2 \;\texttt{RS}_2 \;\texttt{Wr}_2 \;\mid\; \cdots
$$

**After bucketing (Backward):**

$$
\texttt{AG}_{12} \;\texttt{Wa}_{12} \;\texttt{C}_1 \;\texttt{C}_2 \;\texttt{RS}_{12} \;\texttt{Wr}_{12} \;\mid\; \texttt{AG}_{34} \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4 \;\texttt{RS}_{34} \;\texttt{Wr}_{34}
$$

### 4.3 Optimization 2: Reordering

#### 4.3.1 Principle

Even after bucketing, communication remains exposed unless collectives for the **next** FSDP unit are **prefetched** during the **current** unit's computation. Reordering moves communication IR nodes earlier in the schedule to achieve overlap on separate CUDA streams.

The key insight is that all-gather and reduce-scatter operations are **asynchronous**: the launch ($\texttt{AG}$ / $\texttt{RS}$) initiates the transfer on the communication stream, and the wait ($\texttt{Wa}$ / $\texttt{Wr}$) synchronizes. Any compute between launch and wait executes concurrently on the compute stream.

#### 4.3.2 Forward Pass Reordering (Manual Wrapping Example)

**After bucketing, before reordering:**

$$
\texttt{AG}_{12} \;\texttt{Wa}_{12} \;\texttt{C}_1 \;\texttt{C}_2 \;\texttt{AG}_{34} \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4
$$

**After reordering:**

$$
\texttt{AG}_{12} \;\texttt{AG}_{34} \;\texttt{Wa}_{12} \;\texttt{C}_1 \;\texttt{C}_2 \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4
$$

The reordering moves $\texttt{AG}_{34}$ **before** $\texttt{Wa}_{12}$. This allows $\texttt{AG}_{34}$'s communication to overlap with $\texttt{C}_1$ and $\texttt{C}_2$ (and the copy-out operations after $\texttt{Wa}_{12}$) on the compute stream.

**GPU Execution Timeline (Forward, after reorder):**

| Stream | Time → |
|---|---|
| **Comm** | `AG₁₂` ████ `AG₃₄` ████ |
| **Compute** | (idle) → `Wa₁₂` `C₁` `C₂` → `Wa₃₄` `C₃` `C₄` |

The communication for the next module's parameters ($\texttt{AG}_{34}$) executes concurrently with the current module's computation ($\texttt{C}_1$, $\texttt{C}_2$), **hiding the communication latency** behind useful work.

#### 4.3.3 Backward Pass Reordering (Manual Wrapping Example)

**After bucketing, before reordering:**

$$
\texttt{AG}_{12} \;\texttt{Wa}_{12} \;\texttt{C}_1 \;\texttt{C}_2 \;\texttt{RS}_{12} \;\texttt{Wr}_{12} \;\texttt{AG}_{34} \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4 \;\texttt{RS}_{34} \;\texttt{Wr}_{34}
$$

**After reordering:**

$$
\texttt{AG}_{12} \;\texttt{Wa}_{12} \;\texttt{AG}_{34} \;\texttt{C}_1 \;\texttt{C}_2 \;\texttt{RS}_{12} \;\texttt{Wa}_{34} \;\texttt{C}_3 \;\texttt{C}_4 \;\texttt{Wr}_{12} \;\texttt{RS}_{34} \;\texttt{Wr}_{34}
$$

The reordering achieves three simultaneous overlaps:

1. $\texttt{AG}_{34}$ is placed after $\texttt{Wa}_{12}$, allowing it to overlap with $\texttt{C}_1$.
2. $\texttt{RS}_{12}$ is launched after $\texttt{C}_2$ completes but its wait ($\texttt{Wr}_{12}$) is deferred, allowing $\texttt{RS}_{12}$'s transfer to overlap with $\texttt{C}_3$ and $\texttt{C}_4$.
3. The copy-out computation after $\texttt{Wa}_{34}$ overlaps with $\texttt{RS}_{12}$'s transfer, and $\texttt{AG}$ can be placed after $\texttt{Wa}$ since the copy-out compute is already overlapped.

**GPU Execution Timeline (Backward, after reorder):**

| Stream | Time → |
|---|---|
| **Comm** | `AG₁₂` ████ `AG₃₄` ████ `RS₁₂` ████ `RS₃₄` ████ |
| **Compute** | `Wa₁₂` copy → `C₁` `C₂` → `Wa₃₄` copy → `C₃` `C₄` → `Wr₁₂` → `Wr₃₄` |

#### 4.3.4 Overlap Efficiency Condition

For perfect overlap (zero communication exposure), the computation time for the current FSDP unit must equal or exceed the communication time for the next unit's prefetched collective:

**Forward:**

$$
T_{\text{compute}}^{(\text{current})} \geq T_{\text{AG}}^{(\text{next})}
$$

**Backward (both AG and RS must be covered):**

$$
T_{\text{compute}}^{(\text{current})} \geq T_{\text{AG}}^{(\text{next})} + T_{\text{RS}}^{(\text{previous})}
$$

When this condition is violated, residual communication exposure appears on the critical path. The auto-wrapping algorithm (Section 6) explicitly checks this condition to determine optimal bucket boundaries.

---

## 5. Model Wrapping Strategies: Manual and Automatic

### 5.1 Manual Wrapping

#### 5.1.1 Mechanism

In TorchInductor, each IR node carries metadata tracing its original PyTorch module name. Manual wrapping exploits this metadata: the user specifies a list of module boundaries, and SimpleFSDP:

1. Constructs a mapping: `module_name → [IR nodes belonging to that module]`
2. Buckets all communication IR nodes (AG, RS) within each module into a single bucketed collective
3. Reorders bucketed collectives to overlap with adjacent module computation

**Pseudocode 9: Manual Wrapping**

```
PROCEDURE MANUAL_WRAP(ir_graph, module_list):
    ──────────────────────────────────────────────
    // module_list: user-specified list of module names defining FSDP boundaries
    // e.g., ["transformer.layer_0_3", "transformer.layer_4_7", ...]
    
    // Step 1: Build module-to-IR mapping
    module_ir_map ← {}
    FOR each ir_node IN ir_graph:
        mod_name ← ir_node.metadata.original_module
        module_ir_map[mod_name].append(ir_node)
    END FOR
    
    // Step 2: For each module boundary, bucket communication nodes
    FOR each module_name IN module_list:
        ag_nodes ← FILTER(module_ir_map[module_name], type=AllGather)
        rs_nodes ← FILTER(module_ir_map[module_name], type=ReduceScatter)
        
        bucketed_ag ← BUCKET_ALL_GATHER(ag_nodes)    // Pseudocode 7
        bucketed_rs ← BUCKET_REDUCE_SCATTER(rs_nodes) // Pseudocode 8
        
        REPLACE ag_nodes WITH bucketed_ag IN ir_graph
        REPLACE rs_nodes WITH bucketed_rs IN ir_graph
    END FOR
    
    // Step 3: Reorder bucketed collectives for overlap
    REORDER_FOR_OVERLAP(ir_graph)  // Move next module's AG before current module's Wa
    
    RETURN optimized_ir_graph
    ──────────────────────────────────────────────
```

#### 5.1.2 Characteristics

| Property | Manual Wrapping |
|---|---|
| Granularity | User-defined (same as FSDP2) |
| Optimality | Depends on user's wrapping choices |
| Flexibility | Users can customize wrapping after model definition |
| Overhead | Requires domain knowledge of model architecture |

### 5.2 Auto Wrapping

#### 5.2.1 Motivation

Manual wrapping requires user expertise to determine optimal module boundaries. Auto wrapping provides a **fully automatic, fine-grained bucketing interface** that requires no user input. Since the model is sharded per-parameter (each parameter has its own AG/RS), SimpleFSDP employs a **greedy algorithm** to atomically bucket communication IR nodes from each parameter, optimizing for **minimal communication exposure** subject to a **memory constraint**.

#### 5.2.2 Auto-Wrapping Forward IR Transformation Example

Consider 4 parameters across 2 modules. Auto wrapping may produce non-module-aligned bucket boundaries:

**After bucketing (auto, forward):**

$$
\texttt{AG}_1 \;\texttt{Wa}_1 \;\texttt{C}_1 \;\texttt{AG}_{23} \;\texttt{Wa}_{23} \;\texttt{C}_2 \;\texttt{C}_3 \;\texttt{AG}_4 \;\texttt{Wa}_4 \;\texttt{C}_4
$$

**After reordering (auto, forward):**

$$
\texttt{AG}_1 \;\texttt{AG}_{23} \;\texttt{Wa}_1 \;\texttt{C}_1 \;\texttt{AG}_4 \;\texttt{Wa}_{23} \;\texttt{C}_2 \;\texttt{C}_3 \;\texttt{Wa}_4 \;\texttt{C}_4
$$

Note that $\texttt{AG}_{23}$ buckets parameters 2 and 3 together (crossing the module 1 / module 2 boundary) because the auto-wrapping algorithm determined that this bucketing satisfies the time and memory constraints for overlap.

**GPU Execution Timeline (Forward, auto, after reorder):**

| Stream | Time → |
|---|---|
| **Comm** | `AG₁` █ `AG₂₃` ████ `AG₄` █ |
| **Compute** | → `Wa₁` `C₁` → `Wa₂₃` `C₂` `C₃` → `Wa₄` `C₄` |

#### 5.2.3 Auto-Wrapping Backward IR Transformation Example

**After bucketing (auto, backward):**

$$
\texttt{AG}_1 \;\texttt{Wa}_1 \;\texttt{C}_1 \;\texttt{RS}_1 \;\texttt{Wr}_1 \;\texttt{AG}_{23} \;\texttt{Wa}_{23} \;\texttt{C}_2 \;\texttt{C}_3 \;\texttt{RS}_{23} \;\texttt{Wr}_{23} \;\texttt{AG}_4 \;\texttt{Wa}_4 \;\texttt{C}_4 \;\texttt{RS}_4 \;\texttt{Wr}_4
$$

**After reordering (auto, backward):**

$$
\texttt{AG}_1 \;\texttt{Wa}_1 \;\texttt{AG}_{23} \;\texttt{C}_1 \;\texttt{RS}_1 \;\texttt{Wa}_{23} \;\texttt{AG}_4 \;\texttt{C}_2 \;\texttt{C}_3 \;\texttt{Wr}_1 \;\texttt{RS}_{23} \;\texttt{Wa}_4 \;\texttt{C}_4 \;\texttt{Wr}_{23} \;\texttt{RS}_4 \;\texttt{Wr}_4
$$

**GPU Execution Timeline (Backward, auto, after reorder):**

| Stream | Time → |
|---|---|
| **Comm** | `AG₁` █ `AG₂₃` ███ `AG₄` █ `RS₁` █ `RS₂₃` ███ `RS₄` █ |
| **Compute** | `Wa₁` → `C₁` → `Wa₂₃` → `C₂` `C₃` → `Wa₄` → `C₄` → `Wr₁` → `Wr₂₃` → `Wr₃₄` |

---

## 6. Auto-Wrapping Algorithm: Formal Specification

### 6.1 Profiling Phase

Before the wrapping algorithm executes, SimpleFSDP profiles each IR node to estimate computation and communication costs.

**Pseudocode 10: IR Node Profiling**

```
PROCEDURE PROFILE_IR_NODES(ir_graph, process_group):
    ──────────────────────────────────────────────
    FOR each ir_node IN ir_graph:
        IF ir_node.type IS ComputationNode THEN
            // Convert FakeTensor metadata to real tensors
            real_inputs ← MATERIALIZE(ir_node.fake_tensor_inputs)
            
            // Execute Python kernel and measure CUDA event time
            START cuda_event_start
            ir_node.python_kernel(real_inputs)
            STOP cuda_event_end
            
            Tc ← cuda_event_end - cuda_event_start  // Computation time
            Mc ← PEAK_MEMORY_DURING(ir_node.python_kernel)  // Peak memory
            
            STORE ir_node.Tc ← Tc
            STORE ir_node.Mc ← Mc
            
        ELSE IF ir_node.type IS CommunicationNode THEN
            n ← ir_node.transmitted_word_size_bytes
            // Use linear cost model: Tm = α + β·n
            // α, β calibrated per-cluster (NVLink vs IB, topology-aware)
            Tm ← α + β × n
            
            STORE ir_node.Tm ← Tm
        END IF
    END FOR
    ──────────────────────────────────────────────
```

### 6.2 Variable Definitions

The auto-wrapping algorithm operates with the following precisely defined variables:

| Variable | Definition |
|---|---|
| $T_m^{AG}$ | Current step's bucketed all-gather communication time |
| $T_c$ | Current step's total computation time (the compute behind which collectives should be hidden) |
| $M_c$ | Next step's peak computation memory |
| $T_m^{RS}$ | Last step's bucketed reduce-scatter communication time |
| $T_{m_i}^{AG}$ | $i$-th individual all-gather's communication time |
| $T_{c_i}$ | Time to compute the parameters pre-fetched by $i$-th all-gather |
| $M_{c_i}$ | Peak memory to compute the parameters pre-fetched by $i$-th all-gather |
| $T_{m_i}^{RS}$ | Time to reduce-scatter the gradient for parameters in $i$-th all-gather |
| $M_{\max}$ | Maximum allowable memory (HBM budget minus reserved headroom) |

### 6.3 Bucketing Decision Algorithm

**Pseudocode 11: Auto-Wrapping Bucketing Decision (Formal)**

```
PROCEDURE AUTO_WRAP_DECISION(
    T_m_AG,        // Current bucketed AG time
    T_m_RS,        // Last bucketed RS time (backward only)
    T_c,           // Current computation time available for overlap
    M_c,           // Current accumulated peak memory
    T_m_AG_i,      // i-th AG's individual communication time
    T_m_RS_i,      // i-th RS's individual communication time (backward only)
    T_c_i,         // Computation time for i-th AG's parameters
    M_c_i,         // Peak memory for i-th AG's parameters
    M_max,         // Memory limit
    is_forward      // Boolean: forward or backward pass
):
    ──────────────────────────────────────────────
    OUTPUT: True → bucket i-th AG with previous; False → start new bucket

    IF is_forward THEN
        // Time Constraint: bucketed AG must fit within current computation window
        //   T_AG_(m+mi) is the communication time of the merged bucket including i-th AG
        timeConstraint ← (T_{(m+m_i)}^{AG} ≤ T_c)
        
        // Memory Constraint: pre-fetched computation memory must not exceed HBM limit
        memConstraint ← (M_c + M_{c_i} ≤ M_max)
        
        IF timeConstraint AND memConstraint THEN
            RETURN True   // Bucket the i-th AG with previous AGs
        END IF
        
    ELSE IF is_backward THEN
        // Time Constraint: BOTH the previous RS AND the expanded AG must fit
        //   within the current computation window
        timeConstraint ← (T_m^{RS} + T_{(m+m_i)}^{AG} ≤ T_c)
        
        // Memory Constraint: same as forward
        memConstraint ← (M_c + M_{c_i} ≤ M_max)
        
        IF timeConstraint AND memConstraint THEN
            // Also bucket the corresponding RS nodes
            RETURN True
        END IF
    END IF
    
    RETURN False  // Do not bucket; start new bucket boundary
    ──────────────────────────────────────────────
```

### 6.4 Full Auto-Wrapping Procedure

**Pseudocode 12: Complete Auto-Wrapping Algorithm**

```
PROCEDURE AUTO_WRAP(ir_graph, M_max, pass_type):
    ──────────────────────────────────────────────
    // ir_graph: TorchInductor IR with per-parameter AG/RS/Compute nodes
    // M_max: memory budget
    // pass_type: "forward" or "backward"
    
    ag_nodes ← EXTRACT_ALL_GATHER_NODES(ir_graph)  // In execution order
    
    // Initialize first bucket with the first AG
    current_bucket ← [ag_nodes[0]]
    T_m_AG ← ag_nodes[0].Tm
    M_c ← ag_nodes[0].associated_compute.Mc
    T_c ← ag_nodes[0].associated_compute.Tc
    T_m_RS ← 0  // No prior RS for first bucket
    
    all_buckets ← []
    
    FOR i IN 1..len(ag_nodes)-1:
        ag_i ← ag_nodes[i]
        T_m_AG_i ← ag_i.Tm
        M_c_i ← ag_i.associated_compute.Mc
        T_c_i ← ag_i.associated_compute.Tc
        
        // Compute merged bucket communication time
        merged_size ← SUM(size(ag.tensor) for ag in current_bucket) + size(ag_i.tensor)
        T_m_merged ← α + β × merged_size
        
        decision ← AUTO_WRAP_DECISION(
            T_m_AG = T_m_AG,
            T_m_RS = T_m_RS,  // From previous bucket (backward pass)
            T_c = T_c,
            M_c = M_c,
            T_m_AG_i = T_m_AG_i,
            T_m_RS_i = ag_i.associated_rs.Tm,
            T_c_i = T_c_i,
            M_c_i = M_c_i,
            M_max = M_max,
            is_forward = (pass_type == "forward")
        )
        
        IF decision == True THEN
            // Extend current bucket
            current_bucket.append(ag_i)
            T_m_AG ← T_m_merged
            M_c ← M_c + M_c_i
            // T_c remains the CURRENT step's compute (the compute that overlaps this bucket)
        ELSE
            // Finalize current bucket and start new one
            all_buckets.append(current_bucket)
            T_m_RS ← SUM(rs.Tm for rs in current_bucket's associated RS nodes)
            current_bucket ← [ag_i]
            T_m_AG ← ag_i.Tm
            M_c ← M_c_i
            T_c ← T_c_i
        END IF
    END FOR
    
    all_buckets.append(current_bucket)  // Finalize last bucket
    
    // Apply bucketing transformations
    FOR each bucket IN all_buckets:
        BUCKET_ALL_GATHER(bucket)       // Pseudocode 7
        IF pass_type == "backward" THEN
            BUCKET_REDUCE_SCATTER(bucket.associated_rs_nodes)  // Pseudocode 8
        END IF
    END FOR
    
    // Apply reordering
    REORDER_FOR_OVERLAP(ir_graph)
    
    RETURN ir_graph
    ──────────────────────────────────────────────
```

### 6.5 Formal Constraints Summary

**Forward pass bucketing condition for adding the $i$-th all-gather to the current bucket:**

$$
\boxed{T_{(m+m_i)}^{AG} \leq T_c \quad \wedge \quad M_c + M_{c_i} \leq M_{\max}}
$$

**Backward pass bucketing condition (additionally requires RS overlap):**

$$
\boxed{T_m^{RS} + T_{(m+m_i)}^{AG} \leq T_c \quad \wedge \quad M_c + M_{c_i} \leq M_{\max}}
$$

The backward condition is stricter because the computation window must absorb **both** the previous bucket's reduce-scatter and the current bucket's expanded all-gather. This naturally produces **smaller buckets** in the backward pass compared to the forward pass, which is desirable because:

1. Backward computation is typically $2\times$ forward computation (gradient computation for both weights and activations), providing more overlap budget.
2. However, each backward bucket carries both AG and RS collectives, doubling the communication to overlap.
3. The net effect is that backward bucket sizes are constrained by the sum $T_m^{RS} + T_{(m+m_i)}^{AG}$.

---

## 7. Composability with Multi-Dimensional Parallelism

### 7.1 Composability Architecture

SimpleFSDP's implementation through DTensor, parametrization, and selective activation checkpointing makes it natively composable with all standard parallelism dimensions:

| Parallelism Dimension | Composability Mechanism | Code Modification Required |
|---|---|---|
| **Meta Initialization** | Disable all-gather parametrization during `meta` device initialization; reduces init time and memory | None (automatic) |
| **Mixed Precision Training** | `param_dtype` and `reduce_dtype` parsed into DTensor `redistribute`; forward uses 16-bit, backward gradient reduction uses 32-bit | Set `param_dtype=torch.bfloat16`, `reduce_dtype=torch.float32` |
| **Tensor Parallelism (TP)** | Parameters initialized as 2D DTensor, sharded on both DP and TP mesh dimensions; all-gather first on DP sub-mesh, then operates as sharded DTensor on TP sub-mesh | Standard TP module replacement (column/row parallel linear) |
| **Pipeline Parallelism (PP)** | Model partitioned into stage submodules; SimpleFSDP wraps each stage's submodule independently | None (SimpleFSDP wraps received submodule) |
| **Activation Checkpointing** | Applied **before** SimpleFSDP wrapping; SimpleFSDP adds its own selective checkpointing for FSDP communication operators on top | Standard `checkpoint_wrapper` application |

### 7.2 2D DTensor Sharding for TP+FSDP

When combining Tensor Parallelism with FSDP, each parameter is a 2D DTensor with placements on a 2D device mesh:

**Pseudocode 13: 2D DTensor TP+FSDP**

```
PROCEDURE INITIALIZE_2D_PARAMETER(param, dp_mesh, tp_mesh):
    ──────────────────────────────────────────────
    // Create 2D device mesh: (dp_size, tp_size)
    mesh_2d ← DeviceMesh("cuda", [[r0,r1,r2,r3],
                                    [r4,r5,r6,r7]])
    // Rows = DP groups, Columns = TP groups
    
    // Initialize parameter as 2D DTensor
    //   Dimension 0 (DP): Shard — each DP rank holds 1/dp_size of the parameter
    //   Dimension 1 (TP): Shard — within each DP group, TP shards the parameter further
    dtensor_param ← distribute_tensor(
        param,
        device_mesh = mesh_2d,
        placements = [Shard(0),    // DP dimension: sharded (FSDP)
                      Shard(tp_dim)] // TP dimension: column or row sharded
    )
    
    // During FSDP's ReplicateComputation:
    //   redistribute on DP sub-mesh: Shard(0) → Replicate (all-gather across DP)
    //   Result: 1D DTensor on TP sub-mesh with Shard(tp_dim)
    //   → Ready for TP computation (column-parallel or row-parallel matmul)
    
    RETURN dtensor_param
    ──────────────────────────────────────────────
```

The memory per rank for a parameter $\Phi$ under 2D sharding:

$$
M_{\text{per-rank}}^{\text{2D}} = \frac{|\Phi|}{N_{DP} \times N_{TP}} \times d_{\text{type}}
$$

This multiplicative reduction is the fundamental reason for combining TP and FSDP at scale: with $N_{DP} = 64$ and $N_{TP} = 8$, the per-rank parameter storage is $\frac{1}{512}$ of the full model.

### 7.3 Pipeline Parallelism Composability

Pipeline parallelism partitions the model into sequential stages:

**Pseudocode 14: PP + SimpleFSDP**

```
PROCEDURE PP_WITH_SIMPLEFSDP(full_model, pp_schedule, dp_mesh):
    ──────────────────────────────────────────────
    // Step 1: Partition model into PP stages
    stages ← PARTITION(full_model, num_stages = pp_degree)
    
    // Step 2: Assign stage to this rank's PP position
    local_stage ← stages[pp_rank]
    
    // Step 3: Wrap local stage with SimpleFSDP
    //   SimpleFSDP treats the stage as a standalone model
    //   All-gather/reduce-scatter operate within the DP process group
    fsdp_stage ← simple_fsdp(local_stage, dp_mesh)
    
    // Step 4: Compile
    compiled_stage ← torch.compile(fsdp_stage, fullgraph=True)
    
    // Step 5: Execute PP schedule (1F1B, interleaved, etc.)
    //   SimpleFSDP's bucketing and reordering operate within each stage
    //   PP schedule manages inter-stage send/recv
    EXECUTE pp_schedule WITH compiled_stage
    ──────────────────────────────────────────────
```

### 7.4 Activation Checkpointing Composability

The composability relies on a layered checkpointing approach:

1. **User-specified activation checkpointing**: Applied first, determining which layers/operations recompute activations during backward.
2. **SimpleFSDP's FSDP-specific checkpointing**: Applied on top, selectively checkpointing the `redistribute` operations to implement the free-and-re-gather memory optimization.

These two checkpointing policies are **orthogonal** and compose correctly because they target different operations:

$$
\text{Total Recomputation} = \underbrace{\text{User ACT-CKPT (attention, MLP)}}_{\text{Reduces activation memory}} + \underbrace{\text{FSDP ACT-CKPT (redistribute)}}_{\text{Reduces parameter memory}}
$$

---

## 8. Memory Analysis and Communication Cost Models

### 8.1 Per-Rank Memory Breakdown Under SimpleFSDP

For a model with $\Phi$ parameters trained with Adam optimizer, BF16 forward/backward, FP32 optimizer states, and FSDP world size $N$:

$$
M_{\text{rank}} = \underbrace{\frac{2\Phi}{N}}_{\text{Sharded params (BF16)}} + \underbrace{\frac{2\Phi}{N}}_{\text{Sharded grads (BF16)}} + \underbrace{\frac{12\Phi}{N}}_{\text{Sharded opt states (FP32: param copy + m + v)}} + \underbrace{2\Phi \cdot f_{\text{FSDP}}}_{\text{Transient AG buffer}} + \underbrace{M_{\text{act}}}_{\text{Activations}}
$$

where $f_{\text{FSDP}}$ is the fraction of the model whose parameters are simultaneously materialized (typically $\frac{1}{\text{num\_fsdp\_units}}$ for sequential execution, or slightly higher with prefetching).

The auto-wrapping algorithm's memory constraint $M_c + M_{c_i} \leq M_{\max}$ directly controls the transient all-gather buffer size. Larger buckets increase $2\Phi \cdot f_{\text{FSDP}}$ but improve overlap efficiency.

### 8.2 Communication Volume Analysis

Per training step, the total communication volume per rank:

| Collective | Count per FSDP Unit | Volume per Call | Total per Step |
|---|---|---|---|
| All-Gather (Forward) | 1 | $\frac{(N-1)}{N} \cdot S_{\text{unit}}$ | $\frac{(N-1)}{N} \cdot |\Phi| \cdot d$ |
| All-Gather (Backward) | 1 | $\frac{(N-1)}{N} \cdot S_{\text{unit}}$ | $\frac{(N-1)}{N} \cdot |\Phi| \cdot d$ |
| Reduce-Scatter (Backward) | 1 | $\frac{(N-1)}{N} \cdot S_{\text{unit}}$ | $\frac{(N-1)}{N} \cdot |\Phi| \cdot d$ |

Total:

$$
V_{\text{FSDP}} = 3 \times \frac{(N-1)}{N} \times |\Phi| \times d_{\text{type}}
$$

For comparison, DDP all-reduce:

$$
V_{\text{DDP}} = 2 \times \frac{(N-1)}{N} \times |\Phi| \times d_{\text{type}}
$$

The $1.5\times$ communication overhead of FSDP relative to DDP is the price paid for $O(1/N)$ memory scaling.

### 8.3 Bucketing Impact on Communication Time

Given $k$ parameters to bucket, let $n_i$ be the $i$-th parameter's transmitted size:

**Without bucketing:**

$$
T_{\text{no-bucket}} = k \cdot \alpha + \beta \sum_{i=1}^{k} n_i
$$

**With bucketing:**

$$
T_{\text{bucketed}} = \alpha + \beta \sum_{i=1}^{k} n_i + T_{\text{copy-out}}
$$

Net savings:

$$
\Delta T = (k-1) \cdot \alpha - T_{\text{copy-out}}
$$

The copy-out overhead ($T_{\text{copy-out}}$) involves slicing the gathered/reduced buffer back into individual parameter tensors. This is a local memory operation (HBM bandwidth-bound) and is typically $1$–$2$ orders of magnitude faster than the collective base latency, making bucketing profitable for $k \geq 2$.

### 8.4 Reordering Impact on Step Time

Let $T_s$ denote the training step time. Without reordering:

$$
T_s^{\text{no-reorder}} = \sum_{j=1}^{B} \left( T_{\text{AG}_j} + T_{\text{compute}_j} + T_{\text{RS}_j} \right)
$$

where $B$ is the number of FSDP buckets. With perfect reordering (all communication hidden):

$$
T_s^{\text{reorder}} = T_{\text{AG}_1} + \sum_{j=1}^{B} T_{\text{compute}_j} + T_{\text{RS}_B}
$$

Only the first all-gather (no prior computation to overlap with) and the last reduce-scatter (no subsequent computation to overlap with) remain exposed. The theoretical speedup from reordering is:

$$
\text{Speedup} = \frac{T_s^{\text{no-reorder}}}{T_s^{\text{reorder}}} = \frac{\sum_{j} (T_{\text{AG}_j} + T_{\text{compute}_j} + T_{\text{RS}_j})}{T_{\text{AG}_1} + \sum_{j} T_{\text{compute}_j} + T_{\text{RS}_B}}
$$

For communication-heavy workloads (small models, large world sizes), this speedup can exceed $1.5\times$–$2\times$.

---

## 9. Production Deployment Considerations

### 9.1 Graph Mode and Traceability

| Configuration | Behavior | When to Use |
|---|---|---|
| `fullgraph=True` | Single full graph; maximum optimization potential; bucketing and reordering apply globally | Standard Transformer architectures without data-dependent control flow |
| `fullgraph=False` | Graph breaks produce subgraphs; SimpleFSDP optimizes each subgraph independently | Models with dynamic control flow, custom routing (MoE), or untraceable operations |

### 9.2 Profiling Calibration

The auto-wrapping algorithm's quality depends on accurate profiling of $\alpha$ and $\beta$ for the target cluster:

**Pseudocode 15: Communication Parameter Calibration**

```
PROCEDURE CALIBRATE_COMM_PARAMS(process_group, dtype):
    ──────────────────────────────────────────────
    // Measure α (base latency) with minimal message size
    warm_up_rounds ← 10
    measure_rounds ← 100
    
    FOR msg_size IN [1B, 1KB, 4KB, 16KB, 64KB, 256KB, 1MB, 4MB, 16MB, 64MB, 256MB]:
        buf ← ALLOCATE(msg_size, dtype=dtype, device="cuda")
        
        // Warm-up
        FOR i IN 1..warm_up_rounds:
            ALL_GATHER(buf, process_group)
            SYNCHRONIZE()
        END FOR
        
        // Measure
        times ← []
        FOR i IN 1..measure_rounds:
            START timer
            ALL_GATHER(buf, process_group)
            SYNCHRONIZE()
            STOP timer
            times.append(elapsed)
        END FOR
        
        RECORD (msg_size, MEDIAN(times))
    END FOR
    
    // Linear regression: T = α + β × msg_size
    α, β ← LINEAR_FIT(recorded_data)
    
    RETURN α, β
    ──────────────────────────────────────────────
```

**Typical values for production clusters:**

| Interconnect | $\alpha$ ($\mu$s) | $\beta$ (ns/byte) | Effective BW (GB/s) |
|---|---|---|---|
| NVLink (intra-node, A100 8-GPU) | $5$–$8$ | $\sim 1.4$ | $\sim 700$ (bidirectional) |
| NVSwitch (intra-node, H100 8-GPU) | $3$–$6$ | $\sim 1.1$ | $\sim 900$ (bidirectional) |
| InfiniBand HDR (inter-node) | $15$–$30$ | $\sim 4.2$ | $\sim 240$ (unidirectional) |
| InfiniBand NDR (inter-node) | $10$–$25$ | $\sim 2.5$ | $\sim 400$ (unidirectional) |
| RoCE v2 (inter-node) | $20$–$50$ | $\sim 3.5$ | $\sim 280$ (unidirectional) |

### 9.3 Memory Budget Determination

The $M_{\max}$ parameter for auto-wrapping must be set conservatively to avoid OOM:

$$
M_{\max} = M_{\text{HBM}} - M_{\text{reserved}} - M_{\text{fragmentation}}
$$

where:
- $M_{\text{HBM}}$ = total HBM capacity ($80$ GB for A100, $80$ GB for H100 SXM, $192$ GB for B200, $192$ GB for MI300X, $288$ GB for MI350)
- $M_{\text{reserved}}$ = CUDA/ROCm context, communication buffers, NCCL/RCCL scratch space (typically $1$–$3$ GB)
- $M_{\text{fragmentation}}$ = safety margin for allocator fragmentation (typically $5$–$10$% of $M_{\text{HBM}}$)

### 9.4 Diagnostic Checklist for Suboptimal Performance

| Symptom | Root Cause | Resolution |
|---|---|---|
| High communication exposure in Nsight trace | Buckets too small; reordering not enabled | Increase bucket size or enable auto-wrapping; verify `enable_reorder=True` |
| OOM during auto-wrapping | $M_{\max}$ set too high; buckets too large | Reduce $M_{\max}$; auto-wrapping will produce smaller buckets |
| Step time regression after enabling SimpleFSDP | Graph breaks producing too many subgraphs | Audit model for untraceable ops; refactor or use `fullgraph=True` |
| Asymmetric step times across ranks | Profiling inaccuracy due to heterogeneous topology | Re-calibrate $\alpha$, $\beta$ per process group; check for straggler nodes |
| Gradient norm divergence | Mixed precision dtype mismatch in `reduce_dtype` | Verify `reduce_dtype=torch.float32` for gradient reduction |

### 9.5 Cross-Vendor Portability

SimpleFSDP's architecture is portable across NVIDIA and AMD accelerator stacks because:

1. **DTensor abstraction** operates at the PyTorch level, independent of the underlying collective library.
2. **TorchInductor backend** generates vendor-specific kernels through Triton (CUDA/HIP) and vendor math libraries.
3. **Collective operations** dispatch to NCCL (NVIDIA) or RCCL (AMD) through PyTorch's `ProcessGroup` abstraction.

However, the profiling calibration ($\alpha$, $\beta$) must be **re-executed** per cluster, and the memory budget ($M_{\max}$) must reflect the target HBM capacity. Auto-wrapping's greedy algorithm adapts automatically to different communication/computation ratios, making it inherently topology-aware through profiling.

---

## 10. Summary of Engineering Impact

### 10.1 Quantitative Improvements

The SimpleFSDP optimizations deliver measurable improvements along three axes:

| Metric | Unoptimized FSDP | SimpleFSDP (Bucketed + Reordered) | Improvement Mechanism |
|---|---|---|---|
| **Communication Exposure** | 100% (all collectives on critical path) | Near 0% (with sufficient compute to overlap) | Prefetch via reordering |
| **Collective Overhead** | $k \cdot \alpha$ per FSDP unit | $\alpha$ per FSDP unit (bucketed) | Base-latency amortization |
| **Engineering Complexity** | Manual wrapping, explicit collective management | `simple_fsdp()` + `torch.compile()` | Compiler-automated optimization |
| **Debuggability** | Distributed-specific debugging tools required | Eager mode functionally identical; standard PyTorch debugging | Parametrization preserves eager semantics |

### 10.2 Architectural Principles

The SimpleFSDP design demonstrates several principles applicable to production distributed training systems at any scale:

1. **Compiler-integrated communication scheduling** outperforms hand-written overlap code because the compiler has global visibility across all communication and computation nodes, enabling optimal reordering decisions that would be infeasible to maintain manually.

2. **Declarative sharding through DTensor** eliminates a class of distributed correctness bugs (mismatched collective calls, incorrect gradient accumulation, shard-size mismatches) by encoding the distributed semantics in the type system rather than in imperative code.

3. **Profiling-driven auto-tuning** (the $\alpha + \beta n$ cost model calibrated per-cluster) ensures that bucketing decisions are optimal for the specific hardware topology, adapting automatically to NVLink/NVSwitch intra-node versus InfiniBand/RoCE inter-node communication characteristics.

4. **Memory-aware scheduling** (the $M_c + M_{c_i} \leq M_{\max}$ constraint) prevents the optimizer from trading memory safety for communication performance, which is critical for operating near the HBM capacity boundary—the normal regime for production LLM training.

5. **Composability through orthogonal primitives** (DTensor + parametrization + selective checkpointing) ensures that each parallelism dimension (DP, TP, PP, activation checkpointing) can be added independently without modifying the FSDP implementation, reducing the combinatorial complexity of multi-dimensional parallelism configurations.

---

> **Conclusion:** SimpleFSDP represents a principled integration of FSDP distributed training semantics into the PyTorch compiler stack. By expressing all-gather and reduce-scatter operations as differentiable DTensor redistributions, exposing them to TorchInductor's IR, and applying bucketing and reordering optimizations with profiling-driven, memory-constrained greedy scheduling, it achieves near-zero communication exposure while maintaining a minimal user-facing API. The architecture is natively composable with tensor, pipeline, and activation checkpointing parallelism, portable across NVIDIA and AMD accelerator stacks, and debuggable in eager mode—making it a production-ready foundation for large-scale LLM training infrastructure.